# Four-architecture TEM benchmark and physics evaluation

AtomSegNet, UNet++, HRNet and SwinUNet, five folds each, with and without
Noise2Void warm start, evaluated on TEM-ImageNet-v1.3 and then transferred to
unlabelled experimental frames.

Run the cells in order. Section 2 discovers the dataset, section 4 discovers
every checkpoint under `TEM_RESULTS_ROOT`, section 5 evaluates them once and caches
the rows in a master CSV, and everything after that reads the CSV. Sections 8
to 13 are the physics: column localization, sub-pixel precision, strain,
dislocation cores, radial distribution functions. Sections 15 to 18 are the
real-frame transfer.

Two things need your attention before running:

1. `CFG.NOTEBOOK_SOURCES` in section 1 must point at the notebooks or `.py`
   files that define your model classes. Section 3 reads the class source out
   of them; it does not re-implement your architectures.
2. `CFG.SEED` and `CFG.N_FOLDS` must match training. Section 5 checks the
   regenerated split against any `split_md5` or `val_stems` stored in your
   JSON records and refuses to report numbers if they disagree.

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
import os, re, sys, json, math, time, shutil, hashlib, platform
import warnings, itertools, traceback
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path(os.environ.get("TEM_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

class CFG:
    # ---- roots -------------------------------------------------------
    RESULTS_ROOT = Path(os.environ.get("TEM_RESULTS_ROOT", PROJECT_ROOT / "checkpoints"))
    DATA_ROOT    = Path(os.environ.get("TEM_DATA_ROOT", PROJECT_ROOT / "data" / "TEM-ImageNet-v1.3"))
    REAL_ROOT    = Path(os.environ.get("TEM_REAL_DATA", PROJECT_ROOT / "data" / "experimental"))
    OUT_ROOT     = Path(os.environ.get("TEM_BENCHMARK_ROOT", PROJECT_ROOT / "results_local" / "benchmark_v3"))

    # Files that define the model classes. Training notebooks are fine;
    # section 3 pulls only the cells that contain class definitions.
    NOTEBOOK_SOURCES = [
        PROJECT_ROOT / "notebooks" / "01_cnn_training_pipeline.ipynb",
        PROJECT_ROOT / "notebooks" / "02_swinunet_training.ipynb",
    ]

    # ---- experiment grid --------------------------------------------
    ARCHS      = ["AtomSegNet", "UNetPP", "HRNet", "SwinUNet"]
    CONDITIONS = ["no_n2v", "with_n2v"]
    N_FOLDS    = 5
    SEED       = 42

    # ---- dataset sub-folders ----------------------------------------
    IMG_DIR   = "image"           # noisy input
    MASK_DIR  = "circularMask"    # binary segmentation target
    GT_POS    = "gaussianMask"    # ground-truth atom centres come from here
    CLEAN_DIR = "noNoise"         # denoising target
    EXT       = (".png", ".tif", ".tiff", ".jpg", ".jpeg", ".npy")

    # ---- evaluation --------------------------------------------------
    THRESHOLD    = 0.5
    EVAL_BATCH   = 4
    EVAL_MAX_IMG = 600      # per fold; None uses the whole validation split
    NUM_WORKERS  = 0 if os.name == "nt" else 6
    GAUSS_SIGMA  = 1.0      # the Gaussian denoising baseline

    # ---- physics -----------------------------------------------------
    # Historical placeholder only. Calibrate before quoting physical units.
    PIXEL_SIZE_A = 0.1229
    PIXEL_SIZE_VERIFIED = False
    MATCH_RADIUS_PX = 3.0   # detection matching tolerance

    # ---- run bookkeeping ---------------------------------------------
    RUN_TAG  = ""           # free text, ends up in the folder name
    RUN_ID   = None
    RUN_DIR  = None
    DEVICE   = None


import torch
import torch.nn as nn
import torch.nn.functional as F

CFG.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------------
# Per-run output tree
#
# Caches stay shared under OUT_ROOT/tables so a rerun does not recompute
# 35 checkpoints; only the outputs of THIS run are copied into its own
# folder. That way a run is reproducible after the fact without paying
# the cost of isolation on every execution.
# ------------------------------------------------------------------
CFG.RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + (
    f"-{CFG.RUN_TAG}" if CFG.RUN_TAG else "")
CFG.RUN_DIR = CFG.OUT_ROOT / "runs" / CFG.RUN_ID

for sub in ("tables", "figures", "cache", "panels", "real"):
    (CFG.OUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
for sub in ("tables", "figures", "panels", "logs"):
    (CFG.RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

MASTER_CSV = CFG.OUT_ROOT / "tables" / "benchmark_master.csv"
RUN_MANIFEST = CFG.RUN_DIR / "manifest.json"
RUN_LOG = CFG.RUN_DIR / "logs" / "events.jsonl"


def run_path(name, kind="tables"):
    """Where this run's copy of an artifact goes."""
    return CFG.RUN_DIR / kind / name


def log_event(event, **fields):
    """Append one line to this run's event log."""
    rec = dict(t=datetime.now().isoformat(timespec="seconds"), event=event, **fields)
    with open(RUN_LOG, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(rec, default=str) + "\n")
    return rec


def save_table(df, name, index=False):
    """Write a table to the shared cache and to this run's folder.

    The shared copy is what the caching logic reads on the next run; the run
    copy is the record of what this execution actually produced, which the
    shared copy stops being the moment anything is rerun.
    """
    shared = CFG.OUT_ROOT / "tables" / name
    df.to_csv(shared, index=index)
    df.to_csv(run_path(name, "tables"), index=index)
    log_event("table", name=name, rows=len(df), cols=list(df.columns)[:12])
    return shared


def save_fig(fig, name, kind="figures", dpi=150):
    """Save a figure into this run's folder and the shared one."""
    shared = CFG.OUT_ROOT / kind / name
    fig.savefig(shared, dpi=dpi, bbox_inches="tight")
    fig.savefig(run_path(name, kind), dpi=dpi, bbox_inches="tight")
    log_event("figure", name=name, kind=kind)
    return shared


def write_manifest(**extra):
    """Everything needed to say what this run was, rewritten as it learns more."""
    cfg = {k: v for k, v in vars(CFG).items()
           if not k.startswith("_") and not callable(v)}
    man = dict(
        run_id=CFG.RUN_ID,
        started=MANIFEST_STARTED,
        updated=datetime.now().isoformat(timespec="seconds"),
        config={k: str(v) if isinstance(v, Path) else v
                for k, v in cfg.items() if k != "DEVICE"},
        device=str(CFG.DEVICE),
        gpu=(torch.cuda.get_device_name(0) if CFG.DEVICE.type == "cuda" else None),
        versions=dict(python=sys.version.split()[0], torch=torch.__version__,
                      numpy=np.__version__, platform=platform.platform()),
        **extra)
    RUN_MANIFEST.write_text(json.dumps(man, indent=2, default=str), encoding="utf-8")
    return man


def list_runs(n=10):
    """Recent runs with what each produced."""
    rows = []
    root = CFG.OUT_ROOT / "runs"
    for d in sorted(root.glob("*"), reverse=True)[:n]:
        m = d / "manifest.json"
        info = json.loads(m.read_text(encoding="utf-8")) if m.exists() else {}
        rows.append(dict(run_id=d.name,
                         tables=len(list((d / "tables").glob("*.csv"))),
                         figures=len(list((d / "figures").glob("*"))),
                         note=info.get("note", "")))
    import pandas as pd
    return pd.DataFrame(rows)


MANIFEST_STARTED = datetime.now().isoformat(timespec="seconds")


def resolve_root(root, markers=("image",)):
    """Handle the usual one-extra-level-of-nesting after an unzip."""
    root = Path(root)
    if all((root / m).exists() for m in markers):
        return root
    for child in sorted(p for p in root.glob("*") if p.is_dir()):
        if all((child / m).exists() for m in markers):
            print(f"dataset root redirected to {child}")
            return child
    return root


CFG.DATA_ROOT = resolve_root(CFG.DATA_ROOT)

ARCH_ALIASES = {
    "atomsegnet": "AtomSegNet", "atomseg": "AtomSegNet", "asn": "AtomSegNet",
    "unetpp": "UNetPP", "unet++": "UNetPP", "unetplusplus": "UNetPP",
    "nestedunet": "UNetPP", "unetplus": "UNetPP",
    "hrnet": "HRNet", "hrnetv2": "HRNet",
    "swinunet": "SwinUNet", "swin": "SwinUNet", "swinunetr": "SwinUNet",
    "swintransformerunet": "SwinUNet",
}
COND_ALIASES = {
    "no_n2v": "no_n2v", "non2v": "no_n2v", "without n2v": "no_n2v",
    "withoutn2v": "no_n2v", "without_n2v": "no_n2v", "baseline": "no_n2v",
    "with_n2v": "with_n2v", "withn2v": "with_n2v", "with n2v": "with_n2v",
    "n2v": "with_n2v", "n2v_warmstart": "with_n2v",
}


def canon_arch(v):
    if v is None:
        return None
    k = str(v).strip().lower().replace("-", "").replace(" ", "").replace("_", "")
    return ARCH_ALIASES.get(k, str(v).strip())


def canon_cond(v):
    if v is None:
        return None
    k = str(v).strip().lower()
    return COND_ALIASES.get(k, COND_ALIASES.get(k.replace(" ", "_"), k.replace(" ", "_")))


write_manifest(note="started")
log_event("run_start", cwd=str(Path.cwd()))

print(f"run    {CFG.RUN_ID}")
print(f"torch {torch.__version__}   device {CFG.DEVICE}")
if CFG.DEVICE.type == "cuda":
    print(f"gpu    {torch.cuda.get_device_name(0)}")
print(f"data   {CFG.DATA_ROOT}   exists={CFG.DATA_ROOT.exists()}")
print(f"result {CFG.RESULTS_ROOT} exists={CFG.RESULTS_ROOT.exists()}")
print(f"real   {CFG.REAL_ROOT}   exists={CFG.REAL_ROOT.exists()}")
print(f"shared {CFG.OUT_ROOT}")
print(f"run    {CFG.RUN_DIR}")
if not CFG.PIXEL_SIZE_VERIFIED:
    print(f"\nPIXEL_SIZE_A = {CFG.PIXEL_SIZE_A} is unverified. With a 50 px "
          f"lattice spacing it implies a {50 * CFG.PIXEL_SIZE_A:.2f} A lattice "
          f"constant. Every picometre figure in this notebook scales with it.")

## 2. Dataset index and inventory

TEM-ImageNet-v1.3 ships more than the two folders a segmentation run needs.
This cell indexes every variant folder that is present, reports how many
stems each one has, and builds the maps the rest of the notebook uses. The
variants that are not used for training are the raw material for the
ablations in section 19.

In [ ]:
# ============================================================
# 2. Dataset index
# ============================================================
KNOWN_VARIANTS = {
    "image": "noisy simulated HAADF frames, the network input",
    "noNoise": "same frame without shot noise, the denoising target",
    "noBackgroundnoNoise": "no noise and no amorphous background",
    "noNoiseUpinterpolation2x": "2x bicubic upsampled clean frame",
    "noNoiseNoBackgroundUpinterpolation2x": "2x upsampled, background removed",
    "noNoiseNoBackgroundSuperresolution": "super-resolution target",
    "circularMask": "binary disc at each column, the segmentation target",
    "smallcircularMask": "smaller disc, a stricter localization target",
    "gaussianMask": "Gaussian peak at each column, sub-pixel ground truth",
    "radius": "per-column radius map",
    "smallradius": "per-column radius map, small variant",
    "position": "unit-cell vectors, not atom coordinates",
    "positionRadius": "position plus radius encoding",
    "params": "simulation parameters per frame",
    "misc": "assorted extras",
}


def index_folder(folder):
    """stem -> path for one variant folder."""
    out = {}
    if not folder.exists():
        return out
    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() in CFG.EXT:
            out[p.stem] = p
    return out


VARIANTS = {}
for name in sorted(set(KNOWN_VARIANTS) | {p.name for p in CFG.DATA_ROOT.glob("*") if p.is_dir()}):
    idx = index_folder(CFG.DATA_ROOT / name)
    if idx:
        VARIANTS[name] = idx

print("dataset inventory")
print(f"{'folder':40s} {'files':>7s}  note")
for name, idx in VARIANTS.items():
    print(f"{name:40s} {len(idx):7d}  {KNOWN_VARIANTS.get(name, '')}")

noisy_map = VARIANTS.get(CFG.IMG_DIR, {})
mask_map  = VARIANTS.get(CFG.MASK_DIR, {})
clean_map = VARIANTS.get(CFG.CLEAN_DIR, {})
gauss_gt_map = VARIANTS.get(CFG.GT_POS, {})

common = sorted(set(noisy_map) & set(mask_map))
print(f"\npaired image/{CFG.MASK_DIR} stems: {len(common)}")
print(f"of those, with {CFG.CLEAN_DIR}: {len(set(common) & set(clean_map))}")
print(f"of those, with {CFG.GT_POS}:  {len(set(common) & set(gauss_gt_map))}")
if len(common) == 0:
    raise RuntimeError("no paired stems; check CFG.DATA_ROOT, CFG.IMG_DIR, CFG.MASK_DIR")

SPLIT_MD5 = hashlib.md5("|".join(common).encode()).hexdigest()
print(f"split hash (md5 of the ordered stem list): {SPLIT_MD5}")

In [ ]:
# ============================================================
# 2b. Image IO helpers
# ============================================================
from PIL import Image
from scipy.ndimage import gaussian_filter

Image.MAX_IMAGE_PIXELS = None


def load_gray(path, dtype=np.float32):
    """Read any supported file as a 2-D float array, no scaling applied."""
    path = Path(path)
    if path.suffix.lower() == ".npy":
        a = np.load(path)
    else:
        im = Image.open(path)
        if im.mode == "P":
            im = im.convert("RGB")
        a = np.asarray(im)
    a = np.asarray(a, dtype=dtype)
    if a.ndim == 3:
        if a.shape[2] >= 3 and np.allclose(a[..., 0], a[..., 1], atol=1e-3):
            a = a[..., 0]                       # grayscale stored as RGB
        else:
            a = a[..., :3].mean(axis=2)
    return a


def norm01(a, lo_pct=0.0, hi_pct=100.0):
    a = np.asarray(a, dtype=np.float32)
    lo = np.percentile(a, lo_pct) if lo_pct > 0 else float(a.min())
    hi = np.percentile(a, hi_pct) if hi_pct < 100 else float(a.max())
    if hi - lo < 1e-8:
        return np.zeros_like(a)
    return np.clip((a - lo) / (hi - lo), 0.0, 1.0)


def to_binary(a, thr=0.5):
    a = norm01(a)
    return (a > thr).astype(np.float32)


_probe = load_gray(noisy_map[common[0]])
print(f"example frame {common[0]}  shape {_probe.shape}  "
      f"range [{_probe.min():.3g}, {_probe.max():.3g}]")

## 3. Model classes and checkpoint loading

The classes are read out of your training notebooks rather than rewritten
here, because a re-implementation that differs by one BatchNorm silently
loads 6 keys out of 97 and reports nonsense. If a class cannot be found the
cell says which architecture is missing and which file it looked in.

In [ ]:
"""Architectures for the TEM benchmark: AtomSegNet, UNetPP, HRNet, SwinUNet.

Extracted verbatim from the two training notebooks, so the classes here are the
ones the checkpoints under D:\\REsults were trained with. Definitions only:
importing this file builds nothing, trains nothing, and touches no CFG.

    AtomSegNet, UNetPP, HRNet   from TEM IMAGE SEGMENTATION.ipynb, cell 7
    SwinUNet                    from SWUN NET.ipynb, cell 10

Contract for all four: denoised, segmentation_logits = model(noisy), each
[B, 1, H, W]. Segmentation is logits, no sigmoid. The denoising head is the
identity at initialisation, unclamped in train mode, clamped to [0, 1] in eval.

Usage from the benchmark notebook:

    import sys; sys.path.insert(0, r"D:\\UNet")
    from tem_models import ARCH_REGISTRY, build_model

Two names from the Swin notebook were renamed to avoid colliding with the CNN
notebook's versions: count_macs is _swin_count_macs, and the Swin self-test is
_swin_probe. The line that overwrote CFG.ARCHS with ["SwinUNet"] was removed.
"""

import copy
import io
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

__all__ = ["AtomSegNet", "UNetPP", "HRNet", "SwinUNet", "ARCH_REGISTRY",
           "build_model", "architecture_table", "self_test"]


def norm_layer(c, groups=8):
    """GroupNorm rather than BatchNorm. Batch statistics over 8-32 highly
    self-similar crops are noisy, and GroupNorm makes the reported metrics
    independent of batch size, which matters because HRNet runs at a smaller
    batch than the other two. GroupNorm requires num_channels % num_groups == 0,
    and the nested UNet++ concatenations produce counts like 90 and 150, so fall
    back to the largest valid divisor."""
    g = min(groups, c)
    while g > 1 and c % g:
        g -= 1
    return nn.GroupNorm(num_groups=g, num_channels=c)


def init_weights(module):
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.GroupNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)


class ResBlock(nn.Module):
    """Pre-activation residual block with a zero-initialised second convolution,
    so the block is exactly the identity at initialisation and the effective
    depth grows during training rather than being fixed."""

    def __init__(self, c_in, c_out, drop=0.0, zero_init=True):
        super().__init__()
        self.n1 = norm_layer(c_in)
        self.c1 = nn.Conv2d(c_in, c_out, 3, padding=1, padding_mode='reflect', bias=False)
        self.n2 = norm_layer(c_out)
        self.c2 = nn.Conv2d(c_out, c_out, 3, padding=1, padding_mode='reflect', bias=False)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.skip = nn.Identity() if c_in == c_out else nn.Conv2d(c_in, c_out, 1, bias=False)
        if zero_init:
            nn.init.zeros_(self.c2.weight)

    def forward(self, x):
        h = self.c1(self.act(self.n1(x)))
        h = self.drop(h)
        h = self.c2(self.act(self.n2(h)))
        return h + self.skip(x)


def conv_block(c_in, c_out, drop=0.0):
    return ResBlock(c_in, c_out, drop)


class AttentionGate(nn.Module):
    """Additive attention gate (Oktay et al. 2018). Atom columns cover roughly
    5-15% of a frame, so an ungated skip floods the decoder with vacuum. The psi
    bias starts at +2 so gates are ~88% open at initialisation: a gate that
    starts at 0.5 halves every skip and slows the first epochs."""

    def __init__(self, c_skip, c_gate, c_int=None):
        super().__init__()
        c_int = c_int or max(c_skip // 2, 8)
        self.wg = nn.Conv2d(c_gate, c_int, 1, bias=True)
        self.wx = nn.Conv2d(c_skip, c_int, 1, bias=True)
        self.psi = nn.Conv2d(c_int, 1, 1, bias=True)
        nn.init.zeros_(self.psi.weight)
        nn.init.constant_(self.psi.bias, 2.0)

    def forward(self, skip, gate):
        if gate.shape[-2:] != skip.shape[-2:]:
            gate = F.interpolate(gate, size=skip.shape[-2:], mode='bilinear',
                                 align_corners=False)
        a = torch.sigmoid(self.psi(F.silu(self.wg(gate) + self.wx(skip))))
        # detached so the map can be plotted without keeping the graph alive; a
        # non-leaf tensor stashed on a module also breaks the deepcopy that the
        # EMA evaluation copy relies on
        self.last_gate = a.detach()
        return skip * a


class DenoiseHead(nn.Module):
    """Additive residual in image space. delta is zero at initialisation, so the
    network starts as the identity. During training the output is left
    unclamped: the Charbonnier term against a target in [0,1] keeps it in range,
    and clamping here would zero the gradient on exactly the saturated pixels
    that need correcting. At evaluation the output is clamped."""

    def __init__(self, c_in):
        super().__init__()
        self.conv = nn.Conv2d(c_in, 1, 1)
        nn.init.zeros_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)

    def forward(self, x, feat):
        y = x + self.conv(feat)
        return y if self.training else y.clamp(0.0, 1.0)


def up_block(c_in, c_out):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        nn.Conv2d(c_in, c_out, 3, padding=1, padding_mode='reflect', bias=False))


# ---------------- AtomSegNet ----------------
class AtomSegNet(nn.Module):
    """Five-level U-Net with residual blocks and attention-gated skips."""

    def __init__(self, in_ch=1, base=32, drop=0.1, attention=True):
        super().__init__()
        self.attention = attention
        c1, c2, c3, c4, c5 = base, base * 2, base * 4, base * 8, base * 16

        self.stem = nn.Conv2d(in_ch, c1, 3, padding=1, padding_mode='reflect', bias=False)
        self.enc1 = conv_block(c1, c1)
        self.enc2 = conv_block(c1, c2)
        self.enc3 = conv_block(c2, c3)
        self.enc4 = conv_block(c3, c4)
        self.bot = conv_block(c4, c5, drop=drop)
        self.pool = nn.MaxPool2d(2)

        self.up4, self.dec4 = up_block(c5, c4), conv_block(c4 * 2, c4)
        self.up3, self.dec3 = up_block(c4, c3), conv_block(c3 * 2, c3)
        self.up2, self.dec2 = up_block(c3, c2), conv_block(c2 * 2, c2)
        self.up1, self.dec1 = up_block(c2, c1), conv_block(c1 * 2, c1)

        if attention:
            self.ag4 = AttentionGate(c4, c5)
            self.ag3 = AttentionGate(c3, c4)
            self.ag2 = AttentionGate(c2, c3)
            self.ag1 = AttentionGate(c1, c2)

        self.out_norm = norm_layer(c1)
        self.head_denoise = DenoiseHead(c1)
        self.head_seg = nn.Conv2d(c1, 1, 1)
        init_weights(self.stem)
        nn.init.constant_(self.head_seg.bias, -2.0)   # sparse-positive prior

    def forward(self, x):
        e1 = self.enc1(self.stem(x))
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bot(self.pool(e4))

        s4 = self.ag4(e4, b) if self.attention else e4
        d4 = self.dec4(torch.cat([self.up4(b), s4], 1))
        s3 = self.ag3(e3, d4) if self.attention else e3
        d3 = self.dec3(torch.cat([self.up3(d4), s3], 1))
        s2 = self.ag2(e2, d3) if self.attention else e2
        d2 = self.dec2(torch.cat([self.up2(d3), s2], 1))
        s1 = self.ag1(e1, d2) if self.attention else e1
        d1 = self.dec1(torch.cat([self.up1(d2), s1], 1))

        f = F.silu(self.out_norm(d1))
        self.aux_logits = None
        return self.head_denoise(x, f), self.head_seg(f)


# ---------------- UNet++ ----------------
class UNetPP(nn.Module):
    """Nested skip pathways with deep supervision from four separate heads. The
    heads are combined with learned softmax weights instead of a plain mean, so
    the network can down-weight the shallow branches once they stop
    contributing; the learned weights are worth reporting."""

    def __init__(self, in_ch=1, base=31, drop=0.1, deep_supervision=True):
        super().__init__()
        nb = [base, base * 2, base * 4, base * 8, base * 16]
        self.deep_supervision = deep_supervision
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        self.x00 = conv_block(in_ch, nb[0])
        self.x10 = conv_block(nb[0], nb[1])
        self.x20 = conv_block(nb[1], nb[2])
        self.x30 = conv_block(nb[2], nb[3])
        self.x40 = conv_block(nb[3], nb[4], drop=drop)

        self.x01 = conv_block(nb[0] + nb[1], nb[0])
        self.x11 = conv_block(nb[1] + nb[2], nb[1])
        self.x21 = conv_block(nb[2] + nb[3], nb[2])
        self.x31 = conv_block(nb[3] + nb[4], nb[3])

        self.x02 = conv_block(nb[0] * 2 + nb[1], nb[0])
        self.x12 = conv_block(nb[1] * 2 + nb[2], nb[1])
        self.x22 = conv_block(nb[2] * 2 + nb[3], nb[2])

        self.x03 = conv_block(nb[0] * 3 + nb[1], nb[0])
        self.x13 = conv_block(nb[1] * 3 + nb[2], nb[1])

        self.x04 = conv_block(nb[0] * 4 + nb[1], nb[0])

        self.seg_heads = nn.ModuleList([nn.Conv2d(nb[0], 1, 1) for _ in range(4)])
        for h in self.seg_heads:
            nn.init.constant_(h.bias, -2.0)
        self.branch_logits = nn.Parameter(torch.zeros(4))
        self.head_denoise = DenoiseHead(nb[0])

    @property
    def branch_weights(self):
        return torch.softmax(self.branch_logits, 0)

    def forward(self, x):
        x00 = self.x00(x)
        x10 = self.x10(self.pool(x00))
        x20 = self.x20(self.pool(x10))
        x30 = self.x30(self.pool(x20))
        x40 = self.x40(self.pool(x30))

        x01 = self.x01(torch.cat([x00, self.up(x10)], 1))
        x11 = self.x11(torch.cat([x10, self.up(x20)], 1))
        x21 = self.x21(torch.cat([x20, self.up(x30)], 1))
        x31 = self.x31(torch.cat([x30, self.up(x40)], 1))

        x02 = self.x02(torch.cat([x00, x01, self.up(x11)], 1))
        x12 = self.x12(torch.cat([x10, x11, self.up(x21)], 1))
        x22 = self.x22(torch.cat([x20, x21, self.up(x31)], 1))

        x03 = self.x03(torch.cat([x00, x01, x02, self.up(x12)], 1))
        x13 = self.x13(torch.cat([x10, x11, x12, self.up(x22)], 1))

        x04 = self.x04(torch.cat([x00, x01, x02, x03, self.up(x13)], 1))

        logits = [h(t) for h, t in zip(self.seg_heads, [x01, x02, x03, x04])]
        if self.deep_supervision:
            seg = (torch.stack(logits, 0) * self.branch_weights.view(4, 1, 1, 1, 1)).sum(0)
        else:
            seg = logits[-1]

        # exposed for the auxiliary loss; cleared by the training loop
        self.aux_logits = logits[:-1] if (self.training and self.deep_supervision) else None
        return self.head_denoise(x, x04), seg


# ---------------- HRNet ----------------
class FusionUnit(nn.Module):
    """Multi-resolution exchange. Downsampling uses stride-2 convolutions,
    upsampling uses 1x1 + bilinear, following Wang et al. 2020. Interpolating
    downward instead would alias the atomic lattice, which is exactly the signal
    being measured."""

    def __init__(self, chans):
        super().__init__()
        n = len(chans)
        self.n = n
        self.paths = nn.ModuleList()
        for i in range(n):
            row = nn.ModuleList()
            for j in range(n):
                if i == j:
                    row.append(nn.Identity())
                elif j > i:                      # coarse -> fine: 1x1 then up
                    row.append(nn.Sequential(nn.Conv2d(chans[j], chans[i], 1, bias=False),
                                             norm_layer(chans[i])))
                else:                            # fine -> coarse: strided conv
                    ops, c_cur = [], chans[j]
                    for k in range(i - j):
                        c_next = chans[i] if k == i - j - 1 else c_cur
                        ops += [nn.Conv2d(c_cur, c_next, 3, 2, 1, bias=False),
                                norm_layer(c_next)]
                        if k < i - j - 1:
                            ops.append(nn.SiLU(inplace=True))
                        c_cur = c_next
                    row.append(nn.Sequential(*ops))
            self.paths.append(row)

    def forward(self, xs):
        out = []
        for i, xi in enumerate(xs):
            acc = xi
            for j, xj in enumerate(xs):
                if i == j:
                    continue
                y = self.paths[i][j](xj)
                if y.shape[-2:] != xi.shape[-2:]:
                    y = F.interpolate(y, size=xi.shape[-2:], mode='bilinear',
                                      align_corners=False)
                acc = acc + y
            out.append(F.silu(acc))
        return out


class HRNet(nn.Module):
    """Three resolutions held in parallel throughout, with an exchange unit
    after every stage. Full resolution is never lost, which is the property that
    matters for sub-pixel column localisation."""

    def __init__(self, in_ch=1, base=72, stages=3, drop=0.1):
        super().__init__()
        c = [base, base * 2, base * 4]
        self.chans = c
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, c[0], 3, padding=1, padding_mode='reflect', bias=False),
            conv_block(c[0], c[0]))
        self.down1 = nn.Sequential(nn.Conv2d(c[0], c[1], 3, 2, 1, bias=False),
                                   norm_layer(c[1]), nn.SiLU(inplace=True))
        self.down2 = nn.Sequential(nn.Conv2d(c[1], c[2], 3, 2, 1, bias=False),
                                   norm_layer(c[2]), nn.SiLU(inplace=True))

        self.stages = nn.ModuleList()
        self.fusions = nn.ModuleList()
        for _ in range(stages):
            self.stages.append(nn.ModuleList(
                [conv_block(ci, ci, drop=drop if i == 2 else 0.0)
                 for i, ci in enumerate(c)]))
            self.fusions.append(FusionUnit(c))

        self.head_in = conv_block(sum(c), c[0])
        self.out_norm = norm_layer(c[0])
        self.head_denoise = DenoiseHead(c[0])
        self.head_seg = nn.Conv2d(c[0], 1, 1)
        nn.init.constant_(self.head_seg.bias, -2.0)

    def forward(self, x):
        xs = [self.stem(x)]
        xs.append(self.down1(xs[0]))
        xs.append(self.down2(xs[1]))

        for blocks, fuse in zip(self.stages, self.fusions):
            xs = [blk(t) for blk, t in zip(blocks, xs)]
            xs = fuse(xs)

        size = xs[0].shape[-2:]
        cat = torch.cat([xs[0]] + [F.interpolate(t, size=size, mode='bilinear',
                                                 align_corners=False) for t in xs[1:]], 1)
        f = F.silu(self.out_norm(self.head_in(cat)))
        self.aux_logits = None
        return self.head_denoise(x, f), self.head_seg(f)




from contextlib import contextmanager
import copy
import io
import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


def _swin_setting(name, default):
    return getattr(globals().get("CFG", None), name, default)


def _pair(value):
    if isinstance(value, int):
        return value, value
    if len(value) != 2:
        raise ValueError("img_size must be an integer or a (height, width) pair")
    return int(value[0]), int(value[1])


def _window_partition(x, window):
    """[B, H, W, C] -> [B*n_windows, window**2, C]."""
    b, h, w, c = x.shape
    return (x.reshape(b, h // window, window, w // window, window, c)
            .permute(0, 1, 3, 2, 4, 5).reshape(-1, window * window, c))


def _window_reverse(x, window, height, width, batch):
    return (x.reshape(batch, height // window, width // window,
                      window, window, -1)
            .permute(0, 1, 3, 2, 4, 5).reshape(batch, height, width, -1))


def _attention_mask(height, width, window, shift, device=None):
    """Mask cyclic wraparound AND padded keys; keep all softmax rows defined."""
    ph = math.ceil(height / window) * window
    pw = math.ceil(width / window) * window
    sh = shift if height > window else 0
    sw = shift if width > window else 0
    if sh == sw == 0 and (height, width) == (ph, pw):
        return None

    regions = torch.zeros((1, ph, pw, 1), device=device, dtype=torch.long)
    hs = (slice(0, -window), slice(-window, -sh), slice(-sh, None)) if sh else (slice(None),)
    ws = (slice(0, -window), slice(-window, -sw), slice(-sw, None)) if sw else (slice(None),)
    label = 0
    for h_slice in hs:
        for w_slice in ws:
            regions[:, h_slice, w_slice, :] = label
            label += 1
    labels = _window_partition(regions, window).squeeze(-1)
    allowed = labels.unsqueeze(1) == labels.unsqueeze(2)

    if (height, width) != (ph, pw):
        valid = torch.zeros((1, ph, pw, 1), device=device, dtype=torch.bool)
        valid[:, :height, :width] = True
        if sh or sw:
            valid = torch.roll(valid, shifts=(-sh, -sw), dims=(1, 2))
        valid = _window_partition(valid, window).squeeze(-1)
        allowed = allowed & valid.unsqueeze(1)  # Only valid keys for real queries.
        # Padded query rows are discarded after attention. Give each a self
        # connection to avoid an all -inf row (NaNs on some attention backends).
        eye = torch.eye(window * window, device=device, dtype=torch.bool)
        allowed = allowed | ((~valid).unsqueeze(-1) & eye.unsqueeze(0))

    return torch.zeros(allowed.shape, device=device).masked_fill(~allowed, float("-inf"))


class SwinDropPath(nn.Module):
    def __init__(self, probability=0.0):
        super().__init__()
        if not 0 <= probability < 1:
            raise ValueError("drop_path_rate must be in [0, 1)")
        self.probability = float(probability)

    def forward(self, x):
        if not self.training or self.probability == 0:
            return x
        keep = 1.0 - self.probability
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        return x * x.new_empty(shape).bernoulli_(keep) / keep


class SwinWindowAttention(nn.Module):
    def __init__(self, dim, heads, window, attn_drop=0.0, drop=0.0, use_sdpa=True):
        super().__init__()
        if dim % heads:
            raise ValueError(f"Embedding dimension {dim} must be divisible by {heads} heads")
        self.dim, self.heads, self.window = dim, heads, window
        self.head_dim = dim // heads
        self.attn_drop = float(attn_drop)
        self.use_sdpa = bool(use_sdpa)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(drop)
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window - 1) ** 2, heads))

        yy, xx = torch.meshgrid(torch.arange(window), torch.arange(window), indexing="ij")
        coordinates = torch.stack((yy, xx)).flatten(1)
        offsets = coordinates[:, :, None] - coordinates[:, None, :]
        indices = ((offsets[0] + window - 1) * (2 * window - 1)
                   + offsets[1] + window - 1)
        self.register_buffer("relative_position_index", indices, persistent=False)

    def forward(self, x, mask=None):
        bw, tokens, channels = x.shape
        qkv = self.qkv(x).reshape(bw, tokens, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        relative = self.relative_position_bias_table[self.relative_position_index.reshape(-1)]
        bias = relative.reshape(tokens, tokens, self.heads).permute(2, 0, 1)
        bias = bias.unsqueeze(0).to(dtype=q.dtype)
        if mask is not None:
            nw = mask.shape[0]
            bias = bias + mask.to(dtype=q.dtype).unsqueeze(1)
            # Window order is [sample 0 windows..., sample 1 windows...].
            bias = bias.repeat(bw // nw, 1, 1, 1)

        if self.use_sdpa:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=bias,
                dropout_p=self.attn_drop if self.training else 0.0)
        else:
            # Explicit FP32 attention is a debugging fallback, not forced AMP.
            scores = (q.float() * self.head_dim ** -0.5) @ k.float().transpose(-2, -1)
            weights = (scores + bias.float()).softmax(dim=-1).to(v.dtype)
            weights = F.dropout(weights, self.attn_drop, self.training)
            y = weights @ v
        y = y.transpose(1, 2).reshape(bw, tokens, channels)
        return self.proj_drop(self.proj(y))


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, window, shifted, resolution,
                 mlp_ratio=4.0, drop=0.0, attn_drop=0.0, drop_path=0.0,
                 use_sdpa=True):
        super().__init__()
        self.window = window
        self.shift = window // 2 if shifted else 0
        self.resolution = tuple(resolution)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SwinWindowAttention(dim, heads, window, attn_drop, drop, use_sdpa)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(hidden, dim), nn.Dropout(drop))
        self.drop_path = SwinDropPath(drop_path)
        self.register_buffer("nominal_mask", _attention_mask(
            *self.resolution, window, self.shift), persistent=False)

    def forward(self, x):
        b, h, w, _ = x.shape
        residual = x
        x = self.norm1(x)
        pad_h, pad_w = (-h) % self.window, (-w) % self.window
        if pad_h or pad_w:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        hp, wp = h + pad_h, w + pad_w
        sh = self.shift if h > self.window else 0
        sw = self.shift if w > self.window else 0
        if sh or sw:
            x = torch.roll(x, shifts=(-sh, -sw), dims=(1, 2))
        mask = (self.nominal_mask if (h, w) == self.resolution else
                _attention_mask(h, w, self.window, self.shift, x.device))
        x = self.attn(_window_partition(x, self.window), mask)
        x = _window_reverse(x, self.window, hp, wp, b)
        if sh or sw:
            x = torch.roll(x, shifts=(sh, sw), dims=(1, 2))
        x = residual + self.drop_path(x[:, :h, :w, :])
        return x + self.drop_path(self.mlp(self.norm2(x)))


class SwinStage(nn.Module):
    def __init__(self, dim, heads, window, resolution, depth, path_rates,
                 mlp_ratio, drop, attn_drop, use_checkpoint, use_sdpa):
        super().__init__()
        self.use_checkpoint = bool(use_checkpoint)
        self.blocks = nn.ModuleList([
            SwinBlock(dim, heads, window, bool(i % 2), resolution,
                      mlp_ratio, drop, attn_drop, path_rates[i], use_sdpa)
            for i in range(depth)])

    def forward(self, x):
        for block in self.blocks:
            if self.use_checkpoint and self.training and torch.is_grad_enabled():
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return x


class SwinPatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduce = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        if x.shape[1] % 2 or x.shape[2] % 2:
            raise ValueError("Patch merging needs even grids; input padding failed")
        x = torch.cat((x[:, 0::2, 0::2], x[:, 1::2, 0::2],
                       x[:, 0::2, 1::2], x[:, 1::2, 1::2]), dim=-1)
        return self.reduce(self.norm(x))


class SwinPatchExpand(nn.Module):
    """Learned token-to-pixel rearrangement, not bilinear interpolation."""
    def __init__(self, dim, out_dim, scale=2):
        super().__init__()
        self.scale, self.out_dim = scale, out_dim
        self.expand = nn.Linear(dim, scale * scale * out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        b, h, w, _ = x.shape
        s = self.scale
        x = self.expand(x).reshape(b, h, w, s, s, self.out_dim)
        x = x.permute(0, 1, 3, 2, 4, 5).reshape(b, h * s, w * s, self.out_dim)
        return self.norm(x)


class SwinDenoiseHead(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, noisy, features):
        denoised = noisy + self.conv(features)
        return denoised if self.training else denoised.clamp(0.0, 1.0)


class SwinUNet(nn.Module):
    """Four-level Swin encoder + Swin decoder + two full-resolution heads.

    Default widths: 56, 112, 224, 448; 2 blocks per encoder/decoder level.
    At 256px: token grids 64, 32, 16, 8; patch=4 and window=8.
    'base' is an optional alias for embed_dim for existing model factories.
    Arbitrary spatial sizes are padded at right/bottom, never resized, then
    outputs are cropped back. img_size sets the cached nominal attention masks.
    """
    def __init__(self, in_ch=1, base=None, drop=0.0, *, img_size=None,
                 patch_size=None, window_size=None, embed_dim=None,
                 depths=None, num_heads=None, mlp_ratio=4.0, attn_drop=0.0,
                 drop_path_rate=0.1, use_checkpoint=None, use_sdpa=True):
        super().__init__()
        img_size = _pair(img_size if img_size is not None else _swin_setting("IMG_SIZE", 256))
        patch_size = int(patch_size if patch_size is not None else _swin_setting("SWIN_PATCH_SIZE", 4))
        window_size = int(window_size if window_size is not None else _swin_setting("SWIN_WINDOW_SIZE", 8))
        if embed_dim is not None and base is not None and embed_dim != base:
            raise ValueError("base and embed_dim specify conflicting widths")
        embed_dim = int(embed_dim if embed_dim is not None else
                        (base if base is not None else _swin_setting("SWIN_EMBED_DIM", 56)))
        depths = tuple(depths if depths is not None else _swin_setting("SWIN_DEPTHS", (2, 2, 2, 2)))
        num_heads = tuple(num_heads if num_heads is not None else _swin_setting("SWIN_HEADS", (2, 4, 8, 16)))
        use_checkpoint = bool(use_checkpoint if use_checkpoint is not None else
                              _swin_setting("SWIN_USE_CHECKPOINT", False))
        if in_ch != 1:
            raise ValueError("This TEM model requires in_ch=1; do not repeat grayscale to RGB")
        if len(depths) != 4 or len(num_heads) != 4:
            raise ValueError("Provide four depths and four head counts")
        if min(*img_size, patch_size, window_size, embed_dim, *depths, *num_heads) <= 0:
            raise ValueError("Image sizes, widths, depths, and head counts must be positive")
        if not 0 <= drop < 1 or not 0 <= attn_drop < 1 or not 0 <= drop_path_rate < 1:
            raise ValueError("Dropout probabilities must be in [0, 1)")
        if mlp_ratio <= 0 or int(embed_dim * mlp_ratio) < 1:
            raise ValueError("mlp_ratio produces an empty hidden layer")

        self.in_ch = in_ch
        self.img_size, self.patch_size, self.window_size = img_size, patch_size, window_size
        self.embed_dim = embed_dim
        self.depths, self.num_heads = depths, num_heads
        self.input_multiple = patch_size * 8
        self.aux_logits = None
        self.model_config = dict(in_ch=1, img_size=img_size, patch_size=patch_size,
                                 window_size=window_size, embed_dim=embed_dim, depths=depths,
                                 num_heads=num_heads, mlp_ratio=mlp_ratio, drop=drop,
                                 attn_drop=attn_drop, drop_path_rate=drop_path_rate,
                                 use_checkpoint=use_checkpoint, use_sdpa=bool(use_sdpa))
        dimensions = [embed_dim * 2 ** i for i in range(4)]
        nominal = tuple(math.ceil(s / self.input_multiple) * self.input_multiple for s in img_size)
        resolutions = [(nominal[0] // (patch_size * 2 ** i),
                        nominal[1] // (patch_size * 2 ** i)) for i in range(4)]
        for dim, heads in zip(dimensions, num_heads):
            if dim % heads:
                raise ValueError(f"Width {dim} is not divisible by head count {heads}")

        # Stride-p patch projection is equivalent to a linear map per patch.
        self.patch_embed = nn.Conv2d(1, embed_dim, patch_size, stride=patch_size)
        self.patch_norm = nn.LayerNorm(embed_dim)
        self.patch_drop = nn.Dropout(drop)
        rates = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        stage_rates, offset = [], 0
        for depth in depths:
            stage_rates.append(rates[offset:offset + depth])
            offset += depth

        def make_stage(i):
            return SwinStage(dimensions[i], num_heads[i], window_size, resolutions[i],
                             depths[i], stage_rates[i], mlp_ratio, drop, attn_drop,
                             use_checkpoint, use_sdpa)

        self.encoder = nn.ModuleList([make_stage(i) for i in range(4)])
        self.mergers = nn.ModuleList([SwinPatchMerging(dimensions[i]) for i in range(3)])
        self.bottleneck_norm = nn.LayerNorm(dimensions[-1])
        self.expanders = nn.ModuleList([
            SwinPatchExpand(dimensions[i + 1], dimensions[i]) for i in (2, 1, 0)])
        self.skip_projections = nn.ModuleList([
            nn.Linear(2 * dimensions[i], dimensions[i]) for i in (2, 1, 0)])
        self.decoder = nn.ModuleList([make_stage(i) for i in (2, 1, 0)])
        self.final_expand = SwinPatchExpand(embed_dim, embed_dim, scale=patch_size)
        self.head_denoise = SwinDenoiseHead(embed_dim)
        self.head_seg = nn.Conv2d(embed_dim, 1, kernel_size=1)

        self.apply(self._init_weights)
        for module in self.modules():
            if isinstance(module, SwinWindowAttention):
                nn.init.trunc_normal_(module.relative_position_bias_table, std=0.02)
        # Zero AFTER global initialization, or identity initialization is lost.
        nn.init.zeros_(self.head_denoise.conv.weight)
        nn.init.zeros_(self.head_denoise.conv.bias)
        nn.init.constant_(self.head_seg.bias, -2.0)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != 1:
            raise ValueError(f"Expected [B,1,H,W], got {tuple(x.shape)}")
        if not x.is_floating_point():
            raise TypeError("Convert uint8 cache input to float / 255 in the Dataset")
        h, w = x.shape[-2:]
        if min(h, w) <= 0:
            raise ValueError("Empty spatial dimension")
        pad_h, pad_w = (-h) % self.input_multiple, (-w) % self.input_multiple
        # Replication padding works for tiny inputs too. No spatial rescaling.
        padded = F.pad(x, (0, pad_w, 0, pad_h), mode="replicate") if pad_h or pad_w else x
        features = self.patch_embed(padded).permute(0, 2, 3, 1)
        features = self.patch_drop(self.patch_norm(features))
        skips = []
        for i, stage in enumerate(self.encoder):
            features = stage(features)
            if i < 3:
                skips.append(features)
                features = self.mergers[i](features)
        features = self.bottleneck_norm(features)
        for expand, project, stage, skip in zip(
                self.expanders, self.skip_projections, self.decoder, reversed(skips)):
            features = expand(features)
            features = stage(project(torch.cat((features, skip), dim=-1)))
        features = self.final_expand(features).permute(0, 3, 1, 2).contiguous()
        features = features[:, :, :h, :w]
        self.aux_logits = None  # No invented deep-supervision targets.
        return self.head_denoise(x, features), self.head_seg(features)




# ============================================================
# Registry and construction
# ============================================================
ARCH_REGISTRY = {
    "AtomSegNet": AtomSegNet,
    "UNetPP": UNetPP,
    "HRNet": HRNet,
    "SwinUNet": SwinUNet,
}

ARCH_ALIASES = {
    "atomsegnet": "AtomSegNet", "atomseg": "AtomSegNet", "asn": "AtomSegNet",
    "unetpp": "UNetPP", "unet++": "UNetPP", "unetplusplus": "UNetPP",
    "nestedunet": "UNetPP",
    "hrnet": "HRNet", "hrnetv2": "HRNet",
    "swinunet": "SwinUNet", "swin": "SwinUNet", "swinunetr": "SwinUNet",
}


def canon(name):
    key = str(name).strip().lower().replace("-", "").replace(" ", "").replace("_", "")
    return ARCH_ALIASES.get(key, str(name).strip())


def build_model(arch, **kwargs):
    """Construct an untrained model, passing only arguments its __init__ takes.

    A checkpoint's model_config can be handed straight in: keys the class does
    not accept are dropped rather than raising, which is what lets one call site
    serve four different constructors.
    """
    import inspect
    cls = ARCH_REGISTRY.get(canon(arch))
    if cls is None:
        raise KeyError(f"unknown architecture {arch!r}; "
                       f"have {sorted(ARCH_REGISTRY)}")
    accepted = inspect.signature(cls.__init__).parameters
    return cls(**{k: v for k, v in kwargs.items() if k in accepted})


def architecture_table(shape=(1, 1, 256, 256)):
    import pandas as pd
    rows = []
    for name, cls in ARCH_REGISTRY.items():
        m = cls()
        row = dict(arch=name,
                   params_M=sum(p.numel() for p in m.parameters()) / 1e6,
                   convs=sum(1 for x in m.modules() if isinstance(x, nn.Conv2d)),
                   linears=sum(1 for x in m.modules() if isinstance(x, nn.Linear)),
                   tensors=len(m.state_dict()))
        try:
            row["GMACs"] = (_swin_count_macs(m, shape) if name == "SwinUNet"
                            else count_macs(m, shape)) / 1e9
        except Exception:
            row["GMACs"] = float("nan")
        rows.append(row)
        del m
    return pd.DataFrame(rows)


def self_test(verbose=True):
    """Shape, identity-initialisation and finite-gradient check for all four."""
    results = {}
    for name, cls in ARCH_REGISTRY.items():
        torch.manual_seed(0)
        m = cls()
        x = torch.rand(2, 1, 64, 64)
        m.eval()
        with torch.no_grad():
            d, s = m(x)
        assert d.shape == s.shape == x.shape, (name, d.shape, s.shape)
        identity = bool(torch.allclose(d, x, atol=1e-5))
        m.train()
        d, s = m(x)
        loss = d.mean() + s.mean()
        aux = getattr(m, "aux_logits", None)
        if aux:
            loss = loss + sum(a.mean() for a in aux)
        loss.backward()
        dead = [n for n, p in m.named_parameters()
                if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all())]
        assert not dead, f"{name}: no finite gradient for {dead[:4]}"
        results[name] = dict(params_M=sum(p.numel() for p in m.parameters()) / 1e6,
                             tensors=len(m.state_dict()),
                             identity_init=identity)
        if verbose:
            print(f"  {name:11s} {results[name]['params_M']:7.3f} M params  "
                  f"{results[name]['tensors']:4d} tensors  "
                  f"identity init: {identity}")
        del m
    return results


if __name__ == "__main__":
    print("self test")
    self_test()

In [ ]:
# ============================================================
# 3. Model classes, read from the two training notebooks
# ============================================================
import ast, inspect

ARCH_NOTEBOOKS = list(CFG.NOTEBOOK_SOURCES)

WANTED = ("AtomSegNet", "UNetPP", "HRNet", "SwinUNet")


def notebook_code_cells(path):
    nb = json.loads(Path(path).read_text(encoding="utf-8", errors="ignore"))
    return ["".join(c.get("source", [])) for c in nb.get("cells", [])
            if c.get("cell_type") == "code"]


_LITERAL = (ast.Constant, ast.Tuple, ast.List, ast.Dict, ast.Set, ast.Name,
            ast.Attribute, ast.UnaryOp, ast.BinOp, ast.Subscript, ast.Lambda)


def _keep(node):
    """Definitions and plain assignments only.

    Drops every top-level call, loop and `CFG.X = ...`, so the architecture
    cells cannot run their own self-tests or rewrite CFG.ARCHS on import. The
    Swin notebook ends with `CFG.ARCHS = ["SwinUNet"]`, which is what stopped
    the other three architectures being trained with N2V.
    """
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.ClassDef,
                         ast.FunctionDef, ast.AsyncFunctionDef)):
        return True
    if isinstance(node, ast.AnnAssign):
        return not isinstance(node.target, ast.Attribute)
    if isinstance(node, ast.Assign):
        if any(isinstance(t, ast.Attribute) for t in node.targets):
            return False
        return node.value is None or isinstance(node.value, _LITERAL)
    return False


def load_architectures(paths=ARCH_NOTEBOOKS, wanted=WANTED, verbose=True):
    ns = {"torch": torch, "nn": nn, "F": F, "np": np, "math": math,
          "copy": __import__("copy"), "io": __import__("io"),
          "contextmanager": __import__("contextlib").contextmanager,
          "checkpoint": __import__("torch.utils.checkpoint",
                                   fromlist=["checkpoint"]).checkpoint,
          "Path": Path, "__name__": "tem_arch", "__builtins__": __builtins__}
    found = {}
    for p in paths:
        if not Path(p).exists():
            print(f"  MISSING {p}")
            continue
        cells = notebook_code_cells(p)
        used = 0
        for i, src in enumerate(cells):
            if not any(f"class {w}" in src for w in wanted):
                continue
            tree = ast.parse(src)
            body = [n for n in tree.body if _keep(n)]
            mod = ast.Module(body=body, type_ignores=[])
            ast.fix_missing_locations(mod)
            exec(compile(mod, f"<{Path(p).name}#cell{i}>", "exec"), ns)
            names = [n.name for n in body if isinstance(n, ast.ClassDef)]
            used += 1
            if verbose:
                print(f"  {Path(p).name} cell {i}: {len(body)} definitions, "
                      f"classes {names}")
        if used == 0:
            print(f"  {Path(p).name}: no cell defines any of {wanted}")
    for w in wanted:
        cls = ns.get(w)
        if isinstance(cls, type) and issubclass(cls, nn.Module):
            found[w] = cls
    return found, ns


print("reading architecture cells:")
ARCH_REGISTRY, ARCH_NS = load_architectures()

HARVESTED = dict(ARCH_REGISTRY)
MODEL_FACTORY = None

missing = [a for a in CFG.ARCHS if a not in ARCH_REGISTRY]
if missing:
    raise RuntimeError(f"no class found for {missing}. Classes seen: "
                       f"{sorted(k for k, v in ARCH_NS.items() if isinstance(v, type))}")


def build_model(arch, **kwargs):
    """Construct an untrained model, ignoring arguments its __init__ rejects."""
    key = canon_arch(arch)
    cls = ARCH_REGISTRY.get(key)
    if cls is None:
        raise KeyError(f"unknown architecture {arch!r}; have {sorted(ARCH_REGISTRY)}")
    accepted = inspect.signature(cls.__init__).parameters
    return cls(**{k: v for k, v in kwargs.items() if k in accepted})


print(f"\nregistry: { {k: v.__name__ for k, v in ARCH_REGISTRY.items()} }")
for a, cls in ARCH_REGISTRY.items():
    torch.manual_seed(0)
    m = build_model(a)
    x = torch.rand(1, 1, 64, 64)
    m.eval()
    with torch.no_grad():
        d, s = m(x)
    ok = (d.shape == s.shape == x.shape) and torch.allclose(d, x, atol=1e-5)
    print(f"  {a:11s} {sum(p.numel() for p in m.parameters())/1e6:7.3f} M  "
          f"{len(m.state_dict()):4d} tensors  contract: {'ok' if ok else 'FAILED'}")
    del m

In [ ]:
# ============================================================
# 3c. Checkpoint loading
# ============================================================
def _load_torch_file(path, map_location="cpu"):
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


def ckpt_meta(path):
    """The metadata fields fit() writes next to the weights."""
    obj = _load_torch_file(path)
    if not isinstance(obj, dict):
        return {}
    keep = ("arch", "tag", "condition", "fold", "epoch", "best_iou", "iou",
            "model_config", "config", "run_id", "split_md5", "val_stems")
    return {k: obj[k] for k in keep if k in obj}


def _strip_prefixes(sd):
    for pre in ("module.", "_orig_mod."):
        if sd and all(k.startswith(pre) for k in sd):
            sd = {k[len(pre):]: v for k, v in sd.items()}
    return sd


def load_model_weights(path, model, strict=False, verbose=False):
    """Load a fold checkpoint into `model`, whatever wrapper saved it."""
    obj = _load_torch_file(path)
    sd = obj
    if isinstance(obj, dict):
        for key in ("model", "state_dict", "ema", "raw", "weights", "net"):
            v = obj.get(key)
            if isinstance(v, dict) and v and all(
                    isinstance(t, torch.Tensor) for t in list(v.values())[:4]):
                sd = v
                break
    if not isinstance(sd, dict):
        raise RuntimeError(f"{path}: no state dict in checkpoint")
    sd = _strip_prefixes({k: v for k, v in sd.items() if isinstance(v, torch.Tensor)})
    target = getattr(model, "_orig_mod", model)
    target = getattr(target, "module", target)
    res = target.load_state_dict(sd, strict=strict)
    miss = list(getattr(res, "missing_keys", []))
    unex = list(getattr(res, "unexpected_keys", []))
    if miss and len(miss) > 0.5 * len(list(target.state_dict())):
        raise RuntimeError(f"{Path(path).name}: {len(miss)} missing keys, "
                           f"wrong architecture for this file")
    if verbose and (miss or unex):
        print(f"    {Path(path).name}: {len(miss)} missing, {len(unex)} unexpected")
    return miss, unex


def model_from_checkpoint(path, arch, **kw):
    meta = ckpt_meta(path)
    cfgd = meta.get("model_config") or meta.get("config") or {}
    if isinstance(cfgd, dict):
        kw = {**{k: v for k, v in cfgd.items()
                 if isinstance(v, (int, float, str, bool))}, **kw}
    m = build_model(arch, **kw)
    load_model_weights(path, m, strict=False, verbose=True)
    return m.to(CFG.DEVICE).eval()


def n_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


def split_heads(out, prob_hint=None):
    """(denoised, seg_logits) from whatever forward returns."""
    den = seg = None
    if isinstance(out, dict):
        for k in ("denoise", "den", "recon", "clean", "dn"):
            if k in out:
                den = out[k]
                break
        for k in ("seg", "mask", "logits", "segmentation", "out"):
            if k in out:
                seg = out[k]
                break
        if seg is None:
            seg = list(out.values())[-1]
    elif isinstance(out, (tuple, list)):
        if len(out) == 1:
            seg = out[0]
        else:
            den, seg = out[0], out[1]
    else:
        seg = out
    return den, seg


def seg_prob(x):
    """Sigmoid unless the head already emits probabilities."""
    if x is None:
        return None
    if x.shape[1] > 1:
        x = x[:, :1]
    lo, hi = float(x.min()), float(x.max())
    if 0.0 <= lo and hi <= 1.0 and (hi - lo) > 1e-6:
        return x
    return torch.sigmoid(x)


print("loaders ready:", [f for f in ("ckpt_meta", "load_model_weights",
                                     "model_from_checkpoint", "split_heads",
                                     "seg_prob") if f in dir()])
print("build_model comes from section 3:", callable(globals().get("build_model")))

## 4. Checkpoint and record inventory

Everything under `D:\REsults` is scanned once. Architecture comes from the
file name or its parent folder, condition from the path, and both are
overridden by the `arch`/`tag` string inside the checkpoint when it is
present, which is what resolves `D:\REsults\SwinUNet` sitting outside the
with/without tree.

In [ ]:
# ============================================================
# 4. Inventory
# ============================================================
import pandas as pd
from functools import lru_cache

FOLD_RE = re.compile(r"fold[_\-\s]?(\d+)|(?:^|[^\d])f(\d)(?:[^\d]|$)", re.I)
CKPT_EXT = (".pt", ".pth", ".ckpt")
DEFAULT_COND_BY_DIR = {"SwinUNet": "no_n2v"}

NAME_COND_TOKENS = [("with_n2v", "with_n2v"), ("withn2v", "with_n2v"),
                    ("with n2v", "with_n2v"),
                    ("no_n2v", "no_n2v"), ("non2v", "no_n2v"),
                    ("without_n2v", "no_n2v"), ("without n2v", "no_n2v")]

if "ckpt_meta" not in dir():
    raise RuntimeError(
        "ckpt_meta is not defined. Run the model-loading cell (3c) before this "
        "one. Without it every checkpoint would be labelled from its folder "
        "alone and the mislabelling would not be visible.")


@lru_cache(maxsize=None)
def ckpt_meta_cached(path):
    """ckpt_meta with the result kept, so section 5 does not re-read the file."""
    return tuple(sorted(ckpt_meta(path).items(), key=lambda kv: kv[0]))


def fold_from_name(s):
    m = FOLD_RE.search(s)
    if not m:
        return None
    g = m.group(1) or m.group(2)
    return int(g) if g is not None else None


def cond_from_name(name):
    """Condition from a file's own name, which outranks its folder.

    `without n2v\\JSON\\atomsegnet_with_n2v_cv_results_partial.json` is a
    with-N2V log filed in the wrong folder. The name is what the writing
    process recorded; the folder is where someone later dragged it.
    """
    s = str(name).lower()
    for tok, cond in NAME_COND_TOKENS:
        if tok in s:
            return cond
    return None


def cond_from_folder(p):
    for x in (x.lower() for x in Path(p).parts[:-1]):
        c = COND_ALIASES.get(x) or COND_ALIASES.get(x.replace(" ", "_"))
        if c:
            return c
    joined = " / ".join(x.lower() for x in Path(p).parts[:-1])
    if "without n2v" in joined or "no_n2v" in joined:
        return "no_n2v"
    if "with n2v" in joined:
        return "with_n2v"
    return None


def cond_from_path(p):
    return cond_from_name(Path(p).name) or cond_from_folder(p)


def arch_from_path(p):
    for part in reversed(Path(p).parts):
        a = canon_arch(part.replace(".pt", "").replace(".pth", ""))
        if a in CFG.ARCHS:
            return a
    stem = Path(p).stem.lower().replace("_", "").replace("-", "")
    for k, v in ARCH_ALIASES.items():
        if k in stem:
            return v
    return None


def is_pretrain(p):
    p = Path(p)
    return (p.parent.name.strip().lower() == "n2v"
            or p.stem.lower().endswith("_n2v")
            or p.stem.lower().startswith("n2v"))


def scan_checkpoints(root=None, read_tags=True):
    root = Path(root or CFG.RESULTS_ROOT)
    rows, pretrain, tag_errors = [], [], []
    for p in root.rglob("*"):
        if not (p.is_file() and p.suffix.lower() in CKPT_EXT):
            continue
        if p.stat().st_size < 1_000_000:
            continue
        rec = dict(path=str(p), arch=arch_from_path(p),
                   condition=cond_from_path(p),
                   fold=fold_from_name(p.name) or fold_from_name(p.parent.name),
                   size_mb=round(p.stat().st_size / 2**20, 1),
                   mtime=time.strftime("%Y-%m-%d",
                                       time.localtime(p.stat().st_mtime)))
        if is_pretrain(p):
            pretrain.append(rec)
            continue
        tag_arch = tag_cond = None
        if read_tags:
            try:
                meta = dict(ckpt_meta_cached(str(p)))
                raw = str(meta.get("arch") or meta.get("tag") or "")
                for piece in re.split(r"[:|/\s]+", raw):
                    if canon_arch(piece) in CFG.ARCHS:
                        tag_arch = canon_arch(piece)
                    if canon_cond(piece) in CFG.CONDITIONS:
                        tag_cond = canon_cond(piece)
                if meta.get("condition"):
                    tag_cond = canon_cond(meta["condition"]) or tag_cond
                if rec["fold"] is None and meta.get("fold") is not None:
                    rec["fold"] = int(meta["fold"])
            except Exception as e:
                # reported, never swallowed: an unreadable tag means the row
                # below was resolved from its path alone
                tag_errors.append((p.name, f"{type(e).__name__}: {e}"[:90]))
        rec["arch"] = tag_arch or rec["arch"]
        rec["condition"] = (tag_cond or rec["condition"]
                            or DEFAULT_COND_BY_DIR.get(rec["arch"]))
        rec["from_tag"] = bool(tag_arch or tag_cond)
        rows.append(rec)
    return rows, pretrain, tag_errors


def _rows_from_obj(obj):
    if isinstance(obj, list):
        return [r for r in obj if isinstance(r, dict)]
    if isinstance(obj, dict):
        for key in ("records", "rows", "folds", "results"):
            v = obj.get(key)
            if isinstance(v, list):
                return [r for r in v if isinstance(r, dict)]
        return [obj]
    return []


def _tag_record(r, path):
    a = canon_arch(r.get("arch") or r.get("architecture") or arch_from_path(path))
    c = canon_cond(r.get("condition") or r.get("cond"))
    if isinstance(r.get("arch"), str) and ":" in str(r["arch"]):
        lhs, rhs = str(r["arch"]).split(":", 1)
        c = canon_cond(lhs) or c
        a = canon_arch(rhs) or a
    if isinstance(r.get("tag"), str) and ":" in str(r["tag"]):
        lhs, rhs = str(r["tag"]).split(":", 1)
        c = c or canon_cond(lhs)
        a = a or canon_arch(rhs)
    name_c = cond_from_name(Path(path).name)
    folder_c = cond_from_folder(path)
    resolved = c or name_c or folder_c or DEFAULT_COND_BY_DIR.get(a)
    fold = r.get("fold", fold_from_name(Path(path).name))
    try:
        fold = int(fold)
    except (TypeError, ValueError):
        fold = None
    return {**r, "arch": a, "condition": resolved, "fold": fold,
            "source_json": str(path),
            "cond_from_name": name_c, "cond_from_folder": folder_c}


def scan_records(root=None):
    """Every JSON and CSV log, including the SwinUNet per-fold sidecars."""
    root = Path(root or CFG.RESULTS_ROOT)
    recs, misfiled = [], []
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in (".json", ".csv"):
            continue
        if "attempt" in p.name.lower():
            continue                       # started, not finished
        try:
            if p.suffix.lower() == ".json":
                rows = _rows_from_obj(json.loads(
                    p.read_text(encoding="utf-8", errors="ignore")))
            else:
                rows = pd.read_csv(p).to_dict("records")
        except Exception:
            continue
        for r in rows:
            if not isinstance(r, dict):
                continue
            t = _tag_record(r, p)
            if t["arch"] is None or t["fold"] is None:
                continue
            if (t["cond_from_name"] and t["cond_from_folder"]
                    and t["cond_from_name"] != t["cond_from_folder"]):
                misfiled.append((p.name, t["cond_from_name"], t["cond_from_folder"]))
            recs.append(t)
    return recs, misfiled


def dedupe_records(recs, metric="iou"):
    if not len(recs):
        return pd.DataFrame(), []
    df = pd.DataFrame(recs)
    # fold as a plain int, or 1 and 1.0 land in different groups and a
    # duplicate goes unnoticed
    df["fold"] = pd.to_numeric(df.fold, errors="coerce")
    df = df.dropna(subset=["condition", "arch", "fold"]).copy()
    df["fold"] = df.fold.astype(int)
    df["_n_fields"] = df.notna().sum(axis=1)
    conflicts = []
    keys = ["condition", "arch", "fold"]
    for key, g in df.groupby(keys):
        vals = pd.to_numeric(g.get(metric, pd.Series(dtype=float)),
                             errors="coerce").dropna().round(6).unique()
        if len(g) > 1 and len(vals) > 1:
            conflicts.append(dict(condition=key[0], arch=key[1], fold=key[2],
                                  n_records=len(g), values=sorted(vals.tolist()),
                                  files=sorted({Path(s).name for s in g.source_json})))
    out = (df.sort_values("_n_fields", ascending=False)
             .drop_duplicates(subset=keys, keep="first")
             .drop(columns="_n_fields").reset_index(drop=True))
    return out, conflicts


_ck, _pre, TAG_ERRORS = scan_checkpoints()
CKPTS = pd.DataFrame(_ck)
N2V_WEIGHTS = pd.DataFrame(_pre)
_rec, MISFILED = scan_records()
RAW_RECS = pd.DataFrame(_rec)
RECS, REC_CONFLICTS = dedupe_records(RAW_RECS)

print(f"fold checkpoints: {len(CKPTS)}")
if not len(CKPTS):
    raise RuntimeError(f"no checkpoints under {CFG.RESULTS_ROOT}")

resolved = CKPTS.dropna(subset=["arch", "condition"])
grid = (resolved.pivot_table(index="arch", columns="condition", values="fold",
                             aggfunc="count", fill_value=0)
                .reindex(index=CFG.ARCHS, columns=CFG.CONDITIONS, fill_value=0))
print(grid.to_string())
print(f"  {int(CKPTS.from_tag.sum())}/{len(CKPTS)} resolved from the tag inside "
      f"the file, the rest from their path")

for a in CFG.ARCHS:
    for c in CFG.CONDITIONS:
        if grid.loc[a, c] != CFG.N_FOLDS:
            print(f"  incomplete: {a} {c} has {int(grid.loc[a, c])}/"
                  f"{CFG.N_FOLDS} folds")

dup = resolved.dropna(subset=["fold"]).duplicated(
    subset=["arch", "condition", "fold"], keep=False)
if dup.any():
    print(f"\n{int(dup.sum())} checkpoint(s) claiming a cell that is already taken "
          f"(section 5c would evaluate both and keep the last):")
    print(resolved[dup][["path", "arch", "condition", "fold", "mtime"]]
          .sort_values(["arch", "condition", "fold"]).to_string(index=False))

unresolved = CKPTS[CKPTS.arch.isna() | CKPTS.condition.isna() | CKPTS.fold.isna()]
if len(unresolved):
    print(f"\n{len(unresolved)} checkpoint(s) with an unresolved field:")
    print(unresolved[["path", "arch", "condition", "fold"]].to_string(index=False))

if TAG_ERRORS:
    print(f"\n{len(TAG_ERRORS)} checkpoint(s) whose metadata could not be read, "
          f"resolved from their path instead:")
    for name, err in TAG_ERRORS[:8]:
        print(f"  {name}: {err}")

print(f"\nN2V pretraining weights (not evaluated): {len(N2V_WEIGHTS)}")
for _, r in N2V_WEIGHTS.iterrows():
    print(f"  {r.arch or '?':11s} {r.condition or '?':9s} {r.size_mb:6.1f} MB  "
          f"{Path(r.path).name}")

print(f"\nrecords: {len(RAW_RECS)} raw, {len(RECS)} after deduplication")
if len(RECS):
    print(RECS.groupby(["condition", "arch"]).size().to_string())
if MISFILED:
    print("\nlogs whose filename disagrees with their folder (the filename wins):")
    for name, byname, byfolder in sorted(set(MISFILED)):
        print(f"  {name}: name says {byname}, folder says {byfolder}")
if REC_CONFLICTS:
    print(f"\n{len(REC_CONFLICTS)} fold(s) recorded twice with different metrics:")
    for c in REC_CONFLICTS:
        print(f"  {c['condition']:9s} {c['arch']:11s} fold {c['fold']}: "
              f"iou {c['values']}  in {c['files']}")
    print("  section 5 recomputes from the weights, so the benchmark is "
          "unaffected. Do not quote the JSON numbers until you know which "
          "run is which.")

CKPTS.to_csv(CFG.OUT_ROOT / "tables" / "checkpoint_inventory.csv", index=False)

## 5. Fold splits and the evaluation driver

One pass over every checkpoint. Per fold it writes IoU, Dice, pixel F1,
precision, recall, the best-F1 threshold from a sweep, PSNR and SSIM of the
denoising head against `noNoise`, and the PSNR of a Gaussian sigma = 1 filter
on the same images so the denoising column has a baseline that belongs to the
fold rather than to the model.

Rows are cached in `benchmark_master.csv` and keyed by
(condition, arch, fold, threshold). Set `FORCE_REEVAL = True` to recompute.

In [ ]:
# ============================================================
# 5a. Deterministic folds, validated against stored splits
# ============================================================
from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader

_kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
FOLDS = [(tr, va) for tr, va in _kf.split(np.arange(len(common)))]


def fold_indices(fold):
    """fold is 1-based, matching the checkpoint names."""
    return FOLDS[int(fold) - 1]


def verify_splits(records):
    """Compare regenerated validation stems against anything stored in JSON."""
    checked = ok = 0
    for r in records:
        f = r.get("fold")
        if f is None:
            continue
        stems = r.get("val_stems") or r.get("val_files")
        h = r.get("split_md5") or r.get("split_hash")
        try:
            _, va = fold_indices(int(f))
        except Exception:
            continue
        mine = [common[i] for i in va]
        if stems:
            checked += 1
            ok += int(set(Path(s).stem for s in stems) == set(mine))
        elif h:
            checked += 1
            ok += int(str(h) == hashlib.md5("|".join(mine).encode()).hexdigest())
    return checked, ok


chk, good = verify_splits(RECS.to_dict("records") if len(RECS) else [])
if chk == 0:
    print("no stored split information found in the JSON records.")
    print(f"folds are regenerated with KFold(n_splits={CFG.N_FOLDS}, shuffle=True, "
          f"random_state={CFG.SEED}) over {len(common)} sorted stems.")
    print("if that is not what training used, every held-out number below is "
          "contaminated by training data. Fix CFG.SEED before trusting section 6.")
elif good == chk:
    print(f"split check: {good}/{chk} stored splits reproduce exactly.")
else:
    raise RuntimeError(f"split check FAILED: only {good}/{chk} stored splits "
                       f"reproduce. CFG.SEED or CFG.N_FOLDS does not match training.")

for i, (tr, va) in enumerate(FOLDS, 1):
    print(f"  fold {i}: train {len(tr):6d}  val {len(va):6d}")

In [ ]:
# ============================================================
# 5b. Dataset and metric primitives
# ============================================================
from skimage.metrics import structural_similarity as ssim_fn


class TEMEvalDataset(Dataset):
    """Validation-side only: no augmentation, deterministic order."""

    def __init__(self, stems):
        self.stems = list(stems)

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, i):
        s = self.stems[i]
        x = norm01(load_gray(noisy_map[s]))
        y = to_binary(load_gray(mask_map[s]))
        c = norm01(load_gray(clean_map[s])) if s in clean_map else np.zeros_like(x)
        return (torch.from_numpy(x)[None], torch.from_numpy(y)[None],
                torch.from_numpy(c)[None], i)


def pixel_stats(prob, gt, thr):
    p = (prob > thr)
    g = gt > 0.5
    tp = float(np.logical_and(p, g).sum())
    fp = float(np.logical_and(p, ~g).sum())
    fn = float(np.logical_and(~p, g).sum())
    iou = tp / max(tp + fp + fn, 1e-9)
    dice = 2 * tp / max(2 * tp + fp + fn, 1e-9)
    prec = tp / max(tp + fp, 1e-9)
    rec = tp / max(tp + fn, 1e-9)
    return iou, dice, prec, rec


def psnr(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    mse = float(np.mean((a - b) ** 2))
    return 10 * math.log10(1.0 / max(mse, 1e-12))


THRESH_SWEEP = np.round(np.arange(0.10, 0.91, 0.05), 2)


@torch.no_grad()
def evaluate_fold(ckpt_path, arch, fold, max_img=None, thr=None):
    thr = CFG.THRESHOLD if thr is None else thr
    _, va = fold_indices(fold)
    stems = [common[i] for i in va]
    if max_img:
        stems = stems[:max_img]
    ds = TEMEvalDataset(stems)
    dl = DataLoader(ds, batch_size=CFG.EVAL_BATCH, shuffle=False,
                    num_workers=CFG.NUM_WORKERS)
    model = model_from_checkpoint(ckpt_path, arch)
    acc = defaultdict(list)
    sweep_tp = np.zeros_like(THRESH_SWEEP)
    sweep_fp = np.zeros_like(THRESH_SWEEP)
    sweep_fn = np.zeros_like(THRESH_SWEEP)
    for xb, yb, cb, _ in dl:
        xb = xb.to(CFG.DEVICE)
        out = model(xb)
        den, seg = split_heads(out)
        prob = seg_prob(seg).float().cpu().numpy()[:, 0]
        gt = yb.numpy()[:, 0]
        dn = den.float().cpu().numpy()[:, 0] if den is not None else None
        cl = cb.numpy()[:, 0]
        nz = xb.float().cpu().numpy()[:, 0]
        for k in range(len(gt)):
            i_, d_, p_, r_ = pixel_stats(prob[k], gt[k], thr)
            acc["iou"].append(i_); acc["dice"].append(d_)
            acc["precision"].append(p_); acc["recall"].append(r_)
            g = gt[k] > 0.5
            for j, t in enumerate(THRESH_SWEEP):
                pm = prob[k] > t
                sweep_tp[j] += np.logical_and(pm, g).sum()
                sweep_fp[j] += np.logical_and(pm, ~g).sum()
                sweep_fn[j] += np.logical_and(~pm, g).sum()
            if dn is not None and cl[k].max() > 0:
                d01 = norm01(dn[k])
                acc["psnr"].append(psnr(d01, cl[k]))
                acc["ssim"].append(float(ssim_fn(d01, cl[k], data_range=1.0)))
            if cl[k].max() > 0:
                gb = norm01(gaussian_filter(nz[k], CFG.GAUSS_SIGMA))
                acc["psnr_gauss"].append(psnr(gb, cl[k]))
                acc["psnr_raw"].append(psnr(nz[k], cl[k]))
    f1s = 2 * sweep_tp / np.maximum(2 * sweep_tp + sweep_fp + sweep_fn, 1e-9)
    best = int(np.argmax(f1s))
    row = {"arch": arch, "fold": int(fold), "threshold": thr,
           "n_images": len(ds), "params_M": round(n_params(model), 3),
           "best_f1": float(f1s[best]), "best_f1_threshold": float(THRESH_SWEEP[best]),
           "ckpt": str(ckpt_path)}
    for k, v in acc.items():
        row[k] = float(np.mean(v)) if v else float("nan")
        row[k + "_sd"] = float(np.std(v)) if v else float("nan")
    row["psnr_gain_vs_gauss"] = row.get("psnr", float("nan")) - row.get("psnr_gauss", float("nan"))
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return row


print(f"threshold sweep: {THRESH_SWEEP.tolist()}")

In [ ]:
# ============================================================
# 5c. Run the benchmark over every checkpoint
# ============================================================
FORCE_REEVAL = False

if MASTER_CSV.exists() and not FORCE_REEVAL:
    MASTER = pd.read_csv(MASTER_CSV)
else:
    MASTER = pd.DataFrame()

todo = CKPTS.dropna(subset=["arch", "condition", "fold"]).copy()
todo = todo[todo.arch.isin(CFG.ARCHS) & todo.condition.isin(CFG.CONDITIONS)]
todo["fold"] = todo.fold.astype(int)
todo = todo.sort_values(["condition", "arch", "fold"])

have = set()
if len(MASTER):
    have = set(zip(MASTER.condition, MASTER.arch, MASTER.fold.astype(int),
                   MASTER.threshold.round(3)))

rows = MASTER.to_dict("records") if len(MASTER) else []
for _, r in todo.iterrows():
    key = (r.condition, r.arch, int(r.fold), round(CFG.THRESHOLD, 3))
    if key in have:
        continue
    t0 = time.time()
    try:
        row = evaluate_fold(r.path, r.arch, int(r.fold), CFG.EVAL_MAX_IMG)
        row["condition"] = r.condition
        rows.append(row)
        print(f"{r.condition:9s} {r.arch:11s} fold {int(r.fold)}  "
              f"IoU {row['iou']:.4f}  Dice {row['dice']:.4f}  "
              f"PSNR {row.get('psnr', float('nan')):.2f} dB  "
              f"({time.time() - t0:.0f} s)")
    except Exception as e:
        print(f"{r.condition:9s} {r.arch:11s} fold {int(r.fold)}  FAILED: "
              f"{type(e).__name__}: {e}")
        traceback.print_exc(limit=2)

MASTER = pd.DataFrame(rows)
if len(MASTER):
    MASTER = MASTER.drop_duplicates(subset=["condition", "arch", "fold", "threshold"],
                                    keep="last")
    MASTER.to_csv(MASTER_CSV, index=False)
    print(f"\n{len(MASTER)} rows -> {MASTER_CSV}")
    print(MASTER.groupby(["condition", "arch"]).size().to_string())
else:
    print("no rows produced; check the inventory in section 4.")

## 6. Benchmark summary

Per-architecture means over folds, the with/without N2V contrast on shared
folds, and paired t-tests with Holm correction across the architecture pairs.
A t-test on fewer than five shared folds is printed as descriptive only: with
two degrees of freedom a "significant" p is an artefact of the design.

In [ ]:
# ============================================================
# 6a. Summary tables
# ============================================================
from scipy import stats

MASTER = pd.read_csv(MASTER_CSV)
MASTER["fold"] = MASTER.fold.astype(int)

METRICS = ["iou", "dice", "precision", "recall", "best_f1",
           "psnr", "ssim", "psnr_gain_vs_gauss"]
PRIMARY = "iou"
MIN_FOLDS_FOR_INFERENCE = 5


def summarise(df):
    g = df.groupby(["condition", "arch"])
    out = g.agg(folds=("fold", "nunique"),
                params_M=("params_M", "first"),
                **{m: (m, "mean") for m in METRICS if m in df},
                **{m + "_sd": (m, "std") for m in ["iou", "dice", "psnr", "ssim"] if m in df})
    return out.reset_index()


SUMMARY = summarise(MASTER)
pd.set_option("display.width", 200, "display.max_columns", 40)

for cond in CFG.CONDITIONS:
    sub = SUMMARY[SUMMARY.condition == cond]
    if not len(sub):
        continue
    print(f"\n--- {cond} ---")
    show = sub.sort_values(PRIMARY, ascending=False)
    for _, r in show.iterrows():
        inc = "" if r.folds == CFG.N_FOLDS else f"  [{int(r.folds)} folds only]"
        print(f"{r.arch:11s} params {r.params_M:6.2f} M  "
              f"IoU {r.iou:.4f} +- {r.iou_sd if pd.notna(r.iou_sd) else 0:.4f}  "
              f"Dice {r.dice:.4f}  F1* {r.best_f1:.4f}  "
              f"PSNR {r.psnr:6.2f} dB  SSIM {r.ssim:.3f}  "
              f"dPSNR vs Gauss {r.psnr_gain_vs_gauss:+.2f} dB{inc}")

SUMMARY.to_csv(CFG.OUT_ROOT / "tables" / "benchmark_summary.csv", index=False)

# denoising sanity: a head below the Gaussian baseline did not converge
bad = SUMMARY[SUMMARY.psnr_gain_vs_gauss < 0]
if len(bad):
    print("\ndenoising heads below a Gaussian sigma=1 filter:")
    for _, r in bad.iterrows():
        print(f"  {r.condition:9s} {r.arch:11s} {r.psnr_gain_vs_gauss:+.2f} dB "
              f"-- treat its PSNR column as a training failure, not a result")

In [ ]:
# ============================================================
# 6b. N2V contrast and Holm-corrected architecture comparisons
# ============================================================
def paired(df, metric, key_a, key_b, by=("condition", "arch")):
    """Pair rows on fold. key_* are tuples matching `by`."""
    a = df[(df[by[0]] == key_a[0]) & (df[by[1]] == key_a[1])].set_index("fold")[metric]
    b = df[(df[by[0]] == key_b[0]) & (df[by[1]] == key_b[1])].set_index("fold")[metric]
    sh = sorted(set(a.index) & set(b.index))
    return a.loc[sh].values, b.loc[sh].values, sh


def holm(pvals):
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for rank, idx in enumerate(order):
        val = (len(pvals) - rank) * pvals[idx]
        running = max(running, val)
        adj[idx] = min(running, 1.0)
    return adj


print("N2V warm start, paired on fold")
n2v_rows = []
for arch in CFG.ARCHS:
    w, n, sh = paired(MASTER, PRIMARY, ("with_n2v", arch), ("no_n2v", arch))
    if len(sh) == 0:
        print(f"  {arch:11s} no shared folds")
        continue
    d = w - n
    t, p = stats.ttest_rel(w, n) if len(sh) > 1 else (np.nan, np.nan)
    tag = "" if len(sh) >= MIN_FOLDS_FOR_INFERENCE else "  (descriptive, n<5)"
    print(f"  {arch:11s} dIoU {d.mean():+.4f} +- {d.std(ddof=1) if len(d)>1 else 0:.4f}  "
          f"wins {int((d>0).sum())}/{len(d)}  p {p:.3f}{tag}")
    n2v_rows.append(dict(arch=arch, n_folds=len(sh), d_iou=d.mean(), p=p))

print(f"\narchitecture pairs within each condition ({PRIMARY}, Holm corrected)")
for cond in CFG.CONDITIONS:
    pairs, ps = [], []
    for a, b in itertools.combinations(CFG.ARCHS, 2):
        x, y, sh = paired(MASTER, PRIMARY, (cond, a), (cond, b))
        if len(sh) < 2:
            continue
        t, p = stats.ttest_rel(x, y)
        pairs.append((a, b, (x - y).mean(), int((x > y).sum()), len(sh), p))
        ps.append(p)
    if not pairs:
        continue
    adj = holm(np.array(ps))
    print(f"  --- {cond} ---")
    for (a, b, d, w, n, p), pa in sorted(zip(pairs, adj), key=lambda z: z[0][2], reverse=True):
        flag = "*" if pa < 0.05 and n >= MIN_FOLDS_FOR_INFERENCE else " "
        print(f"   {a:11s} vs {b:11s} d {d:+.4f}  wins {w}/{n}  "
              f"p {p:.3f}  Holm {pa:.3f} {flag}")

json.dump(n2v_rows, open(CFG.OUT_ROOT / "tables" / "n2v_contrast.json", "w"), indent=1)

In [ ]:
# ============================================================
# 6c. Figures
# ============================================================
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "savefig.bbox": "tight"})

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, metric, label in zip(axes, ["iou", "best_f1", "psnr"],
                             ["IoU at 0.5", "best-F1 over threshold", "denoise PSNR (dB)"]):
    width = 0.36
    xs = np.arange(len(CFG.ARCHS))
    for k, cond in enumerate(CFG.CONDITIONS):
        mu, sd = [], []
        for a in CFG.ARCHS:
            v = MASTER[(MASTER.condition == cond) & (MASTER.arch == a)][metric].dropna()
            mu.append(v.mean() if len(v) else np.nan)
            sd.append(v.std(ddof=1) if len(v) > 1 else 0.0)
        ax.bar(xs + (k - 0.5) * width, mu, width, yerr=sd, capsize=3,
               label=cond, alpha=0.85)
    ax.set_xticks(xs); ax.set_xticklabels(CFG.ARCHS, rotation=20)
    ax.set_title(label)
axes[0].legend(fontsize=8)
plt.savefig(CFG.OUT_ROOT / "figures" / "benchmark_bars.png")
plt.show()

# per-fold spread, the honest view of stability
fig, ax = plt.subplots(figsize=(7, 3.4))
for k, cond in enumerate(CFG.CONDITIONS):
    for j, a in enumerate(CFG.ARCHS):
        v = MASTER[(MASTER.condition == cond) & (MASTER.arch == a)][PRIMARY].dropna()
        if not len(v):
            continue
        x = j + (k - 0.5) * 0.25
        ax.scatter(np.full(len(v), x), v, s=22, alpha=0.8,
                   color=f"C{k}", label=cond if j == 0 else None)
        ax.hlines(v.mean(), x - 0.08, x + 0.08, color=f"C{k}")
ax.set_xticks(range(len(CFG.ARCHS))); ax.set_xticklabels(CFG.ARCHS)
ax.set_ylabel(PRIMARY); ax.set_title("per-fold IoU, each point is one fold")
ax.legend(fontsize=8)
plt.savefig(CFG.OUT_ROOT / "figures" / "per_fold_iou.png")
plt.show()

In [ ]:
from skimage.feature import peak_local_max
from scipy.ndimage import gaussian_filter

In [ ]:
# ============================================================
# 7. Per-frame comparison panels, all models
# ============================================================
BEST_CONDITIONS = ["with_n2v", "no_n2v"]
BEST_ARCHS      = list(CFG.ARCHS)
BEST_THRESHOLD  = 0.5
PANEL_FOLD      = 2        # one fold for every model, so the rows compare
PANEL_N_FRAMES  = 3        # one figure per frame
PANEL_SEED      = None
PANEL_MIN_COLS  = 15
PANEL_POOL      = 600
PANEL_MIN_GAP   = 20

ERR_COLORS = dict(correct=(0.15, 0.75, 0.35),   # green
                  false_pos=(0.90, 0.15, 0.15), # red
                  missed=(0.98, 0.85, 0.10))    # yellow


def panel_checkpoints(fold=PANEL_FOLD):
    """(condition, arch, checkpoint, fold IoU) for every model at one fold."""
    out = []
    for cond in BEST_CONDITIONS:
        for arch in BEST_ARCHS:
            sub = MASTER[(MASTER.condition == cond) & (MASTER.arch == arch) &
                         (MASTER.fold == fold)]
            if not len(sub):
                print(f"  {cond:9s} {arch:11s} no fold-{fold} checkpoint, skipped")
                continue
            r = sub.iloc[0]
            out.append((cond, arch, r.ckpt, float(r[PRIMARY]),
                        float(r.psnr) if "psnr" in r and pd.notna(r.psnr) else np.nan))
    return out


def panel_frames(n=PANEL_N_FRAMES, fold=PANEL_FOLD, seed=PANEL_SEED):
    """Random dense held-out frames, spread across the fold in stem order."""
    _, va = fold_indices(fold)
    pool = [common[i] for i in va[:PANEL_POOL]]
    order = {s: i for i, s in enumerate(pool)}
    cache = counts if "counts" in dir() else {}
    cts = {}
    for s in pool:
        if s not in gauss_gt_map:
            continue
        c = cache.get(s)
        if c is None:
            c = gt_column_count(s)
            if c is not None:
                cache[s] = c
        if c is not None:
            cts[s] = c
    seed_used = seed if seed is not None else int(time.time() * 1000) % 2**31
    rng = np.random.default_rng(seed_used)
    thr = PANEL_MIN_COLS
    rich = [s for s, c in cts.items() if c >= thr]
    while len(rich) < n * 6 and thr > 8:
        thr -= 3
        rich = [s for s, c in cts.items() if c >= thr]
    if not rich:
        return pool[:n], cts, seed_used
    for gap in (PANEL_MIN_GAP, PANEL_MIN_GAP // 2, 1):
        picked = []
        for s in rng.permutation(rich):
            if all(abs(order[s] - order[t]) >= gap for t in picked):
                picked.append(s)
            if len(picked) == n:
                break
        if len(picked) == n:
            break
    return picked, cts, seed_used


def error_overlay(base, pred, truth, alpha=0.62):
    """Green correct, red painted on vacuum, yellow atom left unpainted."""
    rgb = np.dstack([norm01(base)] * 3)
    for key, sel in (("correct", np.logical_and(pred, truth)),
                     ("false_pos", np.logical_and(pred, ~truth)),
                     ("missed", np.logical_and(~pred, truth))):
        c = ERR_COLORS[key]
        for ch in range(3):
            rgb[..., ch] = np.where(sel, (1 - alpha) * rgb[..., ch] + alpha * c[ch],
                                    rgb[..., ch])
    return np.clip(rgb, 0, 1)


@torch.no_grad()
def model_outputs(ckpt, arch, stems):
    model = model_from_checkpoint(ckpt, arch)
    out = {}
    for s in stems:
        img = norm01(load_gray(noisy_map[s]))
        den, seg = split_heads(model(torch.from_numpy(img)[None, None].to(CFG.DEVICE)))
        out[s] = dict(prob=seg_prob(seg)[0, 0].float().cpu().numpy(),
                      den=(norm01(den[0, 0].float().cpu().numpy())
                           if den is not None else None))
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out


if not len(MASTER):
    print("no rows in MASTER yet; run section 5c first.")
elif not callable(globals().get("locate_columns")):
    raise RuntimeError("run 7a (the localization functions) before this cell")
else:
    print(f"models at fold {PANEL_FOLD}")
    models = panel_checkpoints()
    frames, gt_counts, seed_used = panel_frames()
    print(f"{len(models)} models, frames " +
          ", ".join(f"{s} ({gt_counts.get(s, '?')} cols)" for s in frames) +
          f"   seed {seed_used}")

    have_det = all(callable(globals().get(f)) for f in
                   ("detection_metrics", "gt_peaks_from_gaussian"))

    preds = {}
    for cond, arch, ck, fold_iou, fold_psnr in models:
        try:
            preds[(cond, arch)] = model_outputs(ck, arch, frames)
        except Exception as e:
            print(f"  {cond} {arch}: {type(e).__name__}: {e}")

    rows_out = []
    for stem in frames:
        img = norm01(load_gray(noisy_map[stem]))
        clean = norm01(load_gray(clean_map[stem])) if stem in clean_map else None
        truth = to_binary(load_gray(mask_map[stem])) > 0.5

        live = [(c, a, fi) for c, a, _, fi, _ in models if (c, a) in preds]
        nrow = len(live) + 1
        fig, axes = plt.subplots(nrow, 3, figsize=(10.5, 3.4 * nrow),
                                 gridspec_kw=dict(wspace=0.04, hspace=0.22))
        axes = np.atleast_2d(axes)

        axes[0, 0].imshow(img, cmap="gray")
        axes[0, 0].set_title("noisy input", fontsize=9)
        if clean is not None:
            axes[0, 1].imshow(clean, cmap="gray")
            axes[0, 1].set_title("clean target", fontsize=9)
        else:
            axes[0, 1].axis("off")
        axes[0, 2].imshow(truth, cmap="gray")
        axes[0, 2].set_title(f"atom mask, coverage {truth.mean():.3f}", fontsize=9)
        axes[0, 0].set_ylabel(f"{stem}\n{gt_counts.get(stem, '?')} columns",
                              fontsize=9)

        for i, (cond, arch, fold_iou) in enumerate(live, start=1):
            p = preds[(cond, arch)][stem]
            prob, den = p["prob"], p["den"]
            pred = prob > BEST_THRESHOLD
            iou, dice, prec, rec = pixel_stats(prob, truth.astype(float),
                                               BEST_THRESHOLD)
            ps = psnr(den, clean) if (den is not None and clean is not None) else np.nan

            if den is not None:
                axes[i, 0].imshow(den, cmap="gray")
                axes[i, 0].set_title(f"denoised   {ps:.2f} dB" if np.isfinite(ps)
                                     else "denoised", fontsize=8)
            else:
                axes[i, 0].axis("off")
                axes[i, 0].set_title("no denoising head", fontsize=8)

            im = axes[i, 1].imshow(prob, cmap="magma", vmin=0, vmax=1)
            axes[i, 1].set_title(f"probability, coverage {pred.mean():.3f}",
                                 fontsize=8)

            axes[i, 2].imshow(error_overlay(img, pred, truth))
            fp = float(np.logical_and(pred, ~truth).mean()) * 100
            fn = float(np.logical_and(~pred, truth).mean()) * 100
            t = f"IoU {iou:.3f}   FP {fp:.2f}%  FN {fn:.2f}%"
            if have_det and stem in gauss_gt_map:
                pk, md = locate_columns(prob, den if den is not None else img)
                m = detection_metrics(pk[["x", "y"]].values,
                                      gt_peaks_from_gaussian(stem, md))
                t += f"\nF1 {m['f1']:.3f}   {m['n_pred']}/{m['n_gt']} columns"
                rows_out.append(dict(stem=stem, condition=cond, arch=arch,
                                     iou=iou, f1=m["f1"], rmse_px=m["rmse_px"],
                                     psnr=ps, fp_pct=fp, fn_pct=fn))
            else:
                rows_out.append(dict(stem=stem, condition=cond, arch=arch,
                                     iou=iou, psnr=ps, fp_pct=fp, fn_pct=fn))
            axes[i, 2].set_title(t, fontsize=8)
            axes[i, 0].set_ylabel(f"{arch}\n{cond}\nfold {PANEL_FOLD} "
                                  f"{PRIMARY} {fold_iou:.3f}", fontsize=8)

        for ax in axes.ravel():
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(f"{stem}   fold {PANEL_FOLD}      "
                     f"green correct, red painted on vacuum, "
                     f"yellow atom left unpainted", fontsize=10, y=0.995)
        plt.tight_layout(rect=[0, 0, 1, 0.985])
        name = f"panel_{stem}.png"
        if callable(globals().get("save_fig")):
            save_fig(fig, name, kind="panels")
        else:
            plt.savefig(CFG.OUT_ROOT / "panels" / name, dpi=150,
                        bbox_inches="tight")
        plt.show()

    if rows_out:
        PANEL = pd.DataFrame(rows_out)
        agg = PANEL.groupby(["condition", "arch"]).agg(
            iou=("iou", "mean"), fp_pct=("fp_pct", "mean"),
            fn_pct=("fn_pct", "mean"),
            psnr=("psnr", "mean"),
            **({"f1": ("f1", "mean"), "rmse_px": ("rmse_px", "mean")}
               if "f1" in PANEL else {}))
        print(f"\nmeans over the {len(frames)} frames above")
        print(agg.sort_values("iou", ascending=False).round(4).to_string())
        print("\n  FP is vacuum painted as atom, FN is atom left unpainted, "
              "both as a percent of the frame. A model with both high but a "
              "normal coverage has its discs displaced, which is what pushes "
              "detection F1 down while IoU stays high.")
        if callable(globals().get("save_table")):
            save_table(PANEL, "panel_per_frame.csv")

## 7. Qualitative panels on held-out images

Best fold per architecture, both conditions, on the same held-out frames so
the columns are comparable. Rows are the models, columns are the frames.

In [ ]:
# ============================================================
# 7. Inline segmentation panels, best fold per architecture
# ============================================================
BEST_CONDITIONS = ["with_n2v", "no_n2v"]
BEST_ARCHS      = list(CFG.ARCHS)
BEST_THRESHOLD  = 0.5
N_PANEL_IMAGES  = 4
PANEL_SEED     = None
PANEL_MIN_COLS = 15        # the fold's median is 11; 40 leaves nothing to pick
PANEL_POOL     = 600
PANEL_MIN_GAP  = 20
PANEL_FOLD     = 2         # one fold for every model, so rows are comparable


def panel_frames(n=N_PANEL_IMAGES, fold=None, seed="use_global"):
    """Random dense held-out frames, spread across the fold."""
    seed = PANEL_SEED if seed == "use_global" else seed
    fold = fold or PANEL_FOLD or int(MASTER.loc[MASTER[PRIMARY].idxmax()].fold)
    _, va = fold_indices(fold)
    pool = [common[i] for i in va[:PANEL_POOL]]
    order = {s: i for i, s in enumerate(pool)}

    cache = counts if "counts" in dir() else {}
    cts = {}
    for s in pool:
        if s not in gauss_gt_map:
            continue
        c = cache.get(s)
        if c is None:
            c = gt_column_count(s)
            if c is not None:
                cache[s] = c
        if c is not None:
            cts[s] = c

    seed_used = seed if seed is not None else int(time.time() * 1000) % 2**31
    rng = np.random.default_rng(seed_used)

    # a threshold that leaves a dozen candidates is not a filter, it is a
    # ranking, and ranking a series that barely varies returns the same frames
    thr = PANEL_MIN_COLS
    rich = [s for s, c in cts.items() if c >= thr]
    while len(rich) < n * 6 and thr > 8:
        thr -= 3
        rich = [s for s, c in cts.items() if c >= thr]
    if not rich:
        return pool[:n], cts, fold, seed_used

    picked = []
    for gap in (PANEL_MIN_GAP, PANEL_MIN_GAP // 2, 1):
        picked = []
        for s in rng.permutation(rich):
            if all(abs(order[s] - order[t]) >= gap for t in picked):
                picked.append(s)
            if len(picked) == n:
                break
        if len(picked) == n:
            break
    return picked, cts, fold, seed_used

def best_fold(cond, arch, metric=PRIMARY):
    sub = MASTER[(MASTER.condition == cond) & (MASTER.arch == arch)]
    if not len(sub):
        return None, None
    r = sub.loc[sub[metric].idxmax()]
    return int(r.fold), r.ckpt


def fold_report(cond, arch, metric=PRIMARY):
    """Which fold won, by how much, and how stable the architecture is."""
    sub = MASTER[(MASTER.condition == cond) & (MASTER.arch == arch)]
    if not len(sub):
        return None
    s = sub.sort_values(metric, ascending=False)
    best, worst = s.iloc[0], s.iloc[-1]
    return dict(fold=int(best.fold), ckpt=best.ckpt,
                best=float(best[metric]), worst=float(worst[metric]),
                worst_fold=int(worst.fold),
                mean=float(sub[metric].mean()),
                sd=float(sub[metric].std(ddof=1)) if len(sub) > 1 else 0.0,
                spread=float(best[metric] - worst[metric]),
                folds=int(sub.fold.nunique()),
                psnr=float(best.psnr) if "psnr" in best else np.nan)


def panel_frames(n=N_PANEL_IMAGES, fold=None, seed=PANEL_SEED):
    """Random dense held-out frames, spread across the fold.

    Two things make a naive pick return the same images every time. A pool of
    60 consecutive stems is one simulation series at different noise seeds, so
    there is nothing to choose between. And picking the densest frames ranks
    that series by a quantity that barely varies within it. The pool is
    therefore wide and the picks are forced apart in stem order.
    """
    fold = fold or int(MASTER.loc[MASTER[PRIMARY].idxmax()].fold)
    _, va = fold_indices(fold)
    pool = [common[i] for i in va[:PANEL_POOL]]
    order = {s: i for i, s in enumerate(pool)}

    # reuse 7b's cache and counter so every section counts the same way
    cache = counts if "counts" in dir() else {}
    cts = {}
    for s in pool:
        if s not in gauss_gt_map:
            continue
        c = cache.get(s)
        if c is None:
            c = gt_column_count(s)
            if c is not None:
                cache[s] = c
        if c is not None:
            cts[s] = c

    rng = np.random.default_rng(seed)
    rich = [s for s, c in cts.items() if c >= PANEL_MIN_COLS]
    if len(rich) < n:
        rich = sorted(cts, key=cts.get, reverse=True)[:max(n * 4, 12)]
    if not rich:
        return pool[:n], cts, fold

    picked = []
    for s in rng.permutation(rich):
        if all(abs(order[s] - order[t]) >= PANEL_MIN_GAP for t in picked):
            picked.append(s)
        if len(picked) == n:
            break
    for s in rng.permutation(rich):          # top up if the gap was too strict
        if len(picked) == n:
            break
        if s not in picked:
            picked.append(s)
    return picked, cts, fold

@torch.no_grad()
def infer_stems(ckpt, arch, stems):
    """One forward pass per stem, returning (probabilities, denoised)."""
    model = model_from_checkpoint(ckpt, arch)
    probs, dens = [], []
    for s in stems:
        x = torch.from_numpy(norm01(load_gray(noisy_map[s])))[None, None].to(CFG.DEVICE)
        den, seg = split_heads(model(x))
        probs.append(seg_prob(seg)[0, 0].float().cpu().numpy())
        dens.append(den[0, 0].float().cpu().numpy() if den is not None else None)
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return probs, dens
    
if not len(MASTER):
    print("no rows in MASTER yet; run section 5c first.")
elif not callable(globals().get("locate_columns")):
    raise RuntimeError("run 7a (the localization functions) before this cell")
else:
    print(f"best fold per architecture, chosen on {PRIMARY}\n")
    print(f"{'condition':10s} {'arch':11s} {'best':>6s} {'fold':>5s} "
          f"{'worst':>6s} {'fold':>5s} {'spread':>7s} {'mean+-sd':>16s} {'n':>3s}")
    panel_rows = []
    for cond in BEST_CONDITIONS:
        for arch in BEST_ARCHS:
            rep = fold_report(cond, arch)
            if rep is None:
                print(f"{cond:10s} {arch:11s}   no folds")
                continue
            flag = "  UNSTABLE" if rep["spread"] > 0.05 else ""
            print(f"{cond:10s} {arch:11s} {rep['best']:6.4f} {rep['fold']:5d} "
                  f"{rep['worst']:6.4f} {rep['worst_fold']:5d} "
                  f"{rep['spread']:7.4f} "
                  f"{rep['mean']:.4f} +- {rep['sd']:.4f} {rep['folds']:3d}{flag}")
            panel_rows.append((cond, arch, rep["fold"], rep["ckpt"], rep))
    print("\nspread is best minus worst fold. Above 0.05 means the panel below "
          "shows this architecture at its luckiest, not its typical.")

    pick, gt_counts, panel_fold = panel_frames()
    print(f"\nframes from fold {panel_fold} "
          f"(pool {PANEL_POOL}, seed {PANEL_SEED}): " +
          ", ".join(f"{s} ({gt_counts.get(s, '?')} cols)" for s in pick))
    if len({gt_counts.get(s) for s in pick}) == 1:
        print("  all picks have the same column count, so they are probably one "
              "structure at different noise seeds. Raise PANEL_POOL.")
    folds_in_panel = {r[2] for r in panel_rows}
    if len(folds_in_panel) > 1:
        print(f"  models come from folds {sorted(folds_in_panel)}, so these "
              f"frames are held out for some and were trained on by others. "
              f"The per-image numbers below are not comparable across rows; "
              f"use the fold table above for that.")

    have_detection = all(callable(globals().get(f)) for f in
                         ("detection_metrics", "gt_peaks_from_gaussian"))

    gts = {s: to_binary(load_gray(mask_map[s])) for s in pick}
    scores = {}
    for cond, arch, f, ck, rep in panel_rows:
        try:
            probs, dens = infer_stems(ck, arch, pick)
        except Exception as e:
            print(f"{cond} {arch} fold {f}: {e}")
            continue
        row = []
        for s, p in zip(pick, probs):
            i_, d_, pr_, rc_ = pixel_stats(p, gts[s], BEST_THRESHOLD)
            entry = dict(iou=i_, dice=d_, coverage=float((p > BEST_THRESHOLD).mean()))
            if have_detection and s in gauss_gt_map:
                pk, md = locate_columns(p, p)
                gt_xy = gt_peaks_from_gaussian(s, md)
                m = detection_metrics(pk[["x", "y"]].values, gt_xy)
                entry.update(f1=m["f1"], n_pred=m["n_pred"], n_gt=m["n_gt"])
            row.append(entry)
        scores[(cond, arch)] = (probs, row)

    ncol = len(pick) + 1
    nrow = len(scores) + 1
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.7 * nrow))
    axes = np.atleast_2d(axes)

    for j, s in enumerate(pick):
        axes[0, j].imshow(norm01(load_gray(noisy_map[s])), cmap="gray")
        axes[0, j].set_title(f"{s}\n{gt_counts.get(s, '?')} columns", fontsize=7)
    axes[0, -1].imshow(gts[pick[-1]], cmap="magma")
    axes[0, -1].set_title(f"ground truth\ncoverage {gts[pick[-1]].mean():.3f}",
                          fontsize=7)
    axes[0, 0].set_ylabel("input", fontsize=8)

    for i, ((cond, arch), (probs, row)) in enumerate(scores.items(), start=1):
        rep = next(r[4] for r in panel_rows if r[0] == cond and r[1] == arch)
        for j, (p, sc) in enumerate(zip(probs, row)):
            axes[i, j].imshow(p > BEST_THRESHOLD, cmap="magma")
            t = f"IoU {sc['iou']:.3f}"
            if "f1" in sc:
                t += f"  F1 {sc['f1']:.3f}\n{sc['n_pred']}/{sc['n_gt']} cols"
            else:
                t += f"\ncov {sc['coverage']:.3f}"
            axes[i, j].set_title(t, fontsize=6.5)
        ov = probs[-1]
        axes[i, -1].imshow(norm01(load_gray(noisy_map[pick[-1]])), cmap="gray")
        axes[i, -1].imshow(np.ma.masked_where(ov <= BEST_THRESHOLD, ov),
                           cmap="autumn", alpha=0.55)
        axes[i, -1].set_title("overlay", fontsize=6.5)
        mean_iou = np.mean([s["iou"] for s in row])
        axes[i, 0].set_ylabel(f"{arch}\n{cond}\nfold {rep['fold']} of "
                              f"{rep['folds']}\nfold {PRIMARY} {rep['best']:.3f}\n"
                              f"panel IoU {mean_iou:.3f}", fontsize=6.5)

    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    if callable(globals().get("save_fig")):
        save_fig(fig, "qualitative_best_folds.png", kind="panels")
    else:
        plt.savefig(CFG.OUT_ROOT / "panels" / "qualitative_best_folds.png", dpi=150)
    plt.show()

    print(f"\npanel means over {len(pick)} frames, not a fold:")
    for (cond, arch), (_, row) in sorted(
            scores.items(), key=lambda kv: -np.mean([s["iou"] for s in kv[1][1]])):
        mi = np.mean([s["iou"] for s in row])
        mf = np.mean([s["f1"] for s in row]) if "f1" in row[0] else np.nan
        mc = np.mean([s["coverage"] for s in row])
        gtc = np.mean([gts[s].mean() for s in pick])
        print(f"  {cond:9s} {arch:11s} IoU {mi:.3f}  "
              f"{'F1 ' + format(mf, '.3f') if np.isfinite(mf) else '':9s}  "
              f"coverage {mc:.4f} vs true {gtc:.4f}  "
              f"blob ratio {mc / max(gtc, 1e-9):.2f}")
    print("  blob ratio above ~1.3 means the mask is fatter than the truth: "
          "good for IoU, bad for separating adjacent columns.")

In [ ]:
"""Mathematically corrected atomic-column localization utilities.

Drop-in replacement for section 7a.  Existing public function names and the
original return keys are preserved.  Additional diagnostic fields are added;
callers that only use the old fields continue to work.

Important semantics
-------------------
* ``sigma`` from a Gaussian fit is the area-equivalent width
  ``sqrt(sigma_x * sigma_y)``.
* ``sigma_px``/``sigma_A`` from ``first_shell_spacing`` are robust shell
  spreads, not standard errors of the reported median.
* ``se_px``/``se_A`` are approximate site-level standard errors.  For a final
  scientific uncertainty, bootstrap independent images/regions and include
  pixel-size calibration uncertainty.
* A nearest-column distance is not automatically a crystallographic lattice
  parameter.  That conversion requires the projected structure and basis.
"""

import math

import numpy as np
import pandas as pd
try:
    import torch
except ImportError:  # localization utilities remain usable without inference
    torch = None
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.optimize import least_squares
from scipy.signal import fftconvolve, find_peaks
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max
from skimage.segmentation import watershed


class LOC_CFG:
    # Detection
    SMOOTH_SIGMA = 1.0
    PROB_THRESH = 0.30       # absolute threshold on the smoothed probability
    MIN_DISTANCE = None      # nearest-column spacing in px; None estimates it
    PEAK_DIST_FRAC = 0.50    # proposal-suppression radius / spacing
    DEDUPE_DIST_FRAC = 0.50  # post-refinement NMS radius / spacing
    BORDER_PX = 6

    # Gaussian refinement.  The patch must contain visible Gaussian tails;
    # merely having more pixels than parameters is not enough.
    FIT_HALF = 4             # 9 x 9 patch
    R2_FLOOR = 0.65
    SNR_FLOOR = 3.0
    MAX_SHIFT_PX = 1.5
    MAX_CENTER_STD_PX = 0.75
    SIGMA_RANGE = (0.50, 3.00)
    MIN_SIGMA_COVERAGE = 1.50  # half-width must be >= this * fitted sigma

    # Linear, signed autocorrelation spacing estimator
    MAX_SPACING_FRAC = 0.25
    ACF_SMOOTH_SIGMA = 1.0
    ACF_MIN_PROMINENCE = 0.015  # fraction of zero-lag covariance

    # Mask/blob handling
    MODE = "auto"            # "auto", "blob", or "peak"
    BLOB_MIN_RADIUS = 3.5
    MASK_LEVEL = 0.5
    WINDOW_FRAC = 1.4
    WINDOW_MAX_SPACING_FRAC = 0.45

    # First-shell selection around the robust nearest-neighbour seed
    SHELL_MIN_HALF_WIDTH_FRAC = 0.08
    SHELL_MAX_HALF_WIDTH_FRAC = 0.30
    SHELL_MAD_MULTIPLIER = 3.0
    MAX_SHELL_REL_SPREAD = 0.25


def _robust_sigma(values):
    """Gaussian-consistent MAD scale; this is spread, not standard error."""
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan
    med = np.median(values)
    return float(1.4826 * np.median(np.abs(values - med)))


def _gauss2d(p, X, Y):
    """Rotated elliptical Gaussian plus a constant background."""
    A, x0, y0, sx, sy, theta, background = p
    ct, st = math.cos(theta), math.sin(theta)
    xr = (X - x0) * ct + (Y - y0) * st
    yr = -(X - x0) * st + (Y - y0) * ct
    return A * np.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2)) + background


def _canonical_ellipse(sx, sy, theta):
    """Return major/minor sigmas and a unique major-axis angle."""
    sx, sy, theta = float(sx), float(sy), float(theta)
    if sy > sx:
        sx, sy = sy, sx
        theta += math.pi / 2
    theta = (theta + math.pi / 2) % math.pi - math.pi / 2
    if abs(sx - sy) <= 1e-6 * max(sx, sy, 1.0):
        theta = 0.0  # orientation is undefined for a circular Gaussian
    return sx, sy, theta


def _linear_autocorrelation(image):
    """Signed, overlap-normalized linear autocovariance (not circular ACF)."""
    p = np.asarray(image, dtype=np.float64)
    finite = np.isfinite(p)
    if not finite.any():
        return None
    fill = float(np.median(p[finite]))
    p = np.where(finite, p, fill)
    p -= p.mean()
    if p.std() < 1e-12:
        return None

    # fftconvolve with the reversed image is a zero-padded linear correlation.
    # Dividing by overlap removes the deterministic falloff with lag.
    ac = fftconvolve(p, p[::-1, ::-1], mode="full")
    support = np.ones_like(p, dtype=np.float64)
    overlap = fftconvolve(support, support[::-1, ::-1], mode="full")
    ac = ac / np.maximum(overlap, 1.0)
    return np.asarray(ac.real, dtype=np.float64)


def _radial_mean(image, cy, cx, rmax):
    """One-pixel radial-bin mean using rounded Euclidean radius."""
    y0 = max(0, int(cy - rmax))
    y1 = min(image.shape[0], int(cy + rmax + 1))
    x0 = max(0, int(cx - rmax))
    x1 = min(image.shape[1], int(cx + rmax + 1))
    yy, xx = np.mgrid[y0:y1, x0:x1]
    rr = np.rint(np.hypot(yy - cy, xx - cx)).astype(np.int32)
    use = rr <= rmax
    count = np.bincount(rr[use].ravel(), minlength=rmax + 1)
    total = np.bincount(
        rr[use].ravel(), weights=image[y0:y1, x0:x1][use].ravel(),
        minlength=rmax + 1,
    )
    return total[:rmax + 1] / np.maximum(count[:rmax + 1], 1)


def estimate_min_distance(prob, fallback=6):
    """Estimate the shortest translational repeat from signed linear ACF.

    The first resolved positive maximum after the central-lobe minimum is used.
    The result is a detector-spacing heuristic, not a crystallographic lattice
    parameter.  If no statistically resolved peak exists, ``fallback`` is
    returned instead of manufacturing a spacing from noise.
    """
    arr = np.asarray(prob, dtype=np.float64)
    if arr.ndim != 2 or min(arr.shape) < 12:
        return int(fallback)
    ac = _linear_autocorrelation(arr)
    if ac is None:
        return int(fallback)

    cy, cx = arr.shape[0] - 1, arr.shape[1] - 1
    rmax = min(int(LOC_CFG.MAX_SPACING_FRAC * min(arr.shape)),
               cy, cx, ac.shape[0] - 1 - cy, ac.shape[1] - 1 - cx)
    if rmax < 6:
        return int(fallback)

    prof = _radial_mean(ac, cy, cx, rmax)
    if not np.isfinite(prof[0]) or prof[0] <= 0:
        return int(fallback)
    prof = gaussian_filter1d(prof / prof[0], LOC_CFG.ACF_SMOOTH_SIGMA,
                             mode="nearest")

    # Central-lobe boundary: preferably the first resolved local minimum;
    # otherwise the first crossing below 20% of the zero-lag covariance.
    central_prom = max(0.005, 0.01 * float(np.ptp(prof)))
    minima, _ = find_peaks(-prof[1:], prominence=central_prom)
    if len(minima):
        start = int(minima[0] + 1)
    else:
        crossings = np.flatnonzero(prof[1:] <= 0.20)
        if not len(crossings):
            return int(fallback)
        start = int(crossings[0] + 1)

    if start + 3 >= len(prof):
        return int(fallback)

    # Estimate numerical/background fluctuation from first differences in the
    # outer third.  Differencing prevents a slowly varying ACF baseline from
    # being mistaken for noise.
    tail = prof[max(start + 2, int(0.67 * len(prof))):]
    noise = _robust_sigma(np.diff(tail)) / math.sqrt(2) if len(tail) >= 5 else 0.0
    if not np.isfinite(noise):
        noise = 0.0
    prominence = max(LOC_CFG.ACF_MIN_PROMINENCE, 3.0 * noise)
    peaks, props = find_peaks(prof[start + 1:], prominence=prominence,
                              distance=2)
    if not len(peaks):
        return int(fallback)

    baseline = float(np.median(tail)) if len(tail) else 0.0
    for rel in peaks:
        radius = int(start + 1 + rel)
        if radius >= 3 and prof[radius] > baseline + 2.0 * noise:
            return radius
    return int(fallback)


def _moment_initialization(patch, X, Y):
    """Robust background, centroid and covariance for Gaussian initialization."""
    if min(patch.shape) < 3:
        raise ValueError("patch is too small")
    edge = np.concatenate((patch[0], patch[-1], patch[1:-1, 0],
                           patch[1:-1, -1]))
    background = float(np.median(edge))
    noise = _robust_sigma(edge)
    if not np.isfinite(noise) or noise < 1e-9:
        noise = max(float(np.std(edge)), 1e-6)

    weights = np.clip(patch - background, 0.0, None)
    total = float(weights.sum())
    if total <= 1e-12:
        raise ValueError("no positive signal above background")
    x0 = float(np.sum(weights * X) / total)
    y0 = float(np.sum(weights * Y) / total)
    dx = X - x0
    dy = Y - y0
    covariance = np.array([
        [np.sum(weights * dx * dx), np.sum(weights * dx * dy)],
        [np.sum(weights * dx * dy), np.sum(weights * dy * dy)],
    ], dtype=np.float64) / total
    eigval, eigvec = np.linalg.eigh(covariance)
    eigval = np.maximum(eigval, 1e-6)
    major = int(np.argmax(eigval))
    minor = 1 - major
    sx = float(math.sqrt(eigval[major]))
    sy = float(math.sqrt(eigval[minor]))
    theta = float(math.atan2(eigvec[1, major], eigvec[0, major]))
    amplitude = float(np.max(patch) - background)
    return amplitude, x0, y0, sx, sy, theta, background, noise


def refine_peak(img, x, y, half=None):
    """Refine one candidate with a bounded elliptical Gaussian.

    ``ok`` describes localization quality.  Width estimates additionally carry
    ``width_ok`` because a centre can remain stable when a width touches a
    bound.  Approximate fit standard deviations assume independent residuals;
    they must not be treated as full experimental uncertainty after denoising.
    """
    half = LOC_CFG.FIT_HALF if half is None else int(half)
    image = np.asarray(img, dtype=np.float64)
    h, w = image.shape
    xi, yi = int(round(float(x))), int(round(float(y)))
    empty = dict(
        x=float(x), y=float(y), amp=np.nan, sigma=np.nan,
        sigma_x=np.nan, sigma_y=np.nan, theta=np.nan, bg=np.nan,
        r2=np.nan, fit_std_x=np.nan, fit_std_y=np.nan,
        snr=np.nan, width_ok=False, ok=False,
    )
    if (xi - half < 0 or yi - half < 0 or
            xi + half >= w or yi + half >= h):
        return dict(empty, reason="edge")

    patch = image[yi - half:yi + half + 1,
                  xi - half:xi + half + 1].astype(np.float64)
    if not np.isfinite(patch).all():
        return dict(empty, reason="nonfinite_patch")
    Y, X = np.mgrid[-half:half + 1, -half:half + 1].astype(np.float64)

    try:
        A0, dx0, dy0, sx0, sy0, theta0, b0, noise = \
            _moment_initialization(patch, X, Y)
    except ValueError:
        return dict(empty, amp=float(np.nanmax(patch)), reason="flat")
    if A0 <= 1e-8:
        return dict(empty, amp=A0, bg=b0, reason="flat")

    shift = min(float(LOC_CFG.MAX_SHIFT_PX), max(0.5, half - 0.5))
    sigma_lo = float(LOC_CFG.SIGMA_RANGE[0])
    sigma_hi = min(float(LOC_CFG.SIGMA_RANGE[1]),
                   half / float(LOC_CFG.MIN_SIGMA_COVERAGE))
    if sigma_hi <= sigma_lo:
        sigma_hi = sigma_lo * 1.05
    sx0 = float(np.clip(sx0, sigma_lo * 1.05, sigma_hi * 0.95))
    sy0 = float(np.clip(sy0, sigma_lo * 1.05, sigma_hi * 0.95))
    dx0 = float(np.clip(dx0, -0.8 * shift, 0.8 * shift))
    dy0 = float(np.clip(dy0, -0.8 * shift, 0.8 * shift))
    theta0 = (theta0 + math.pi / 2) % math.pi - math.pi / 2

    dynamic = max(float(np.ptp(patch)), A0, 1e-6)
    p0 = [A0, dx0, dy0, sx0, sy0, theta0, b0]
    lower = [0.0, -shift, -shift, sigma_lo, sigma_lo,
             -math.pi / 2, float(np.min(patch) - 2.0 * dynamic)]
    upper = [5.0 * dynamic, shift, shift, sigma_hi, sigma_hi,
             math.pi / 2, float(np.max(patch) + 2.0 * dynamic)]

    try:
        result = least_squares(
            lambda p: (_gauss2d(p, X, Y) - patch).ravel(),
            p0, bounds=(lower, upper), method="trf", loss="soft_l1",
            f_scale=max(noise, 0.02 * A0, 1e-6), x_scale="jac",
            max_nfev=500,
        )
        A, dx, dy, sx, sy, theta, background = result.x
        fit = _gauss2d(result.x, X, Y)
        ss_res = float(np.sum((patch - fit) ** 2))
        ss_tot = float(np.sum((patch - patch.mean()) ** 2))
        r2 = 1.0 - ss_res / max(ss_tot, 1e-12)

        dof = patch.size - len(result.x)
        fit_std_x = fit_std_y = np.nan
        if dof > 0 and result.jac.shape[0] >= result.jac.shape[1]:
            covariance = (ss_res / dof) * np.linalg.pinv(result.jac.T @ result.jac)
            diag = np.diag(covariance)
            if np.all(np.isfinite(diag)):
                fit_std_x = float(math.sqrt(max(diag[1], 0.0)))
                fit_std_y = float(math.sqrt(max(diag[2], 0.0)))
    except Exception:
        return dict(empty, amp=A0, bg=b0, snr=A0 / noise,
                    reason="fit_error")

    sx, sy, theta = _canonical_ellipse(sx, sy, theta)
    snr = float(A / max(noise, 1e-12))
    centre_std_ok = (
        np.isfinite(fit_std_x) and np.isfinite(fit_std_y) and
        max(fit_std_x, fit_std_y) <= LOC_CFG.MAX_CENTER_STD_PX
    )
    width_margin = 0.02 * max(sigma_hi - sigma_lo, 1e-6)
    width_ok = (sy > sigma_lo + width_margin and
                sx < sigma_hi - width_margin)

    failures = []
    if not result.success:
        failures.append("optimizer")
    if r2 < LOC_CFG.R2_FLOOR:
        failures.append("low_r2")
    if snr < LOC_CFG.SNR_FLOOR:
        failures.append("low_snr")
    if math.hypot(dx, dy) > LOC_CFG.MAX_SHIFT_PX:
        failures.append("large_shift")
    if not centre_std_ok:
        failures.append("centre_uncertain")
    ok = not failures

    if not ok:
        # A background-subtracted centre of mass is a transparent fallback.
        # Keep it flagged; never mix it silently with successful fits.
        weights = np.clip(patch - b0, 0.0, None)
        total = float(weights.sum())
        if total > 0:
            dx_fallback = float(np.sum(weights * X) / total)
            dy_fallback = float(np.sum(weights * Y) / total)
            dx = float(np.clip(dx_fallback, -1.0, 1.0))
            dy = float(np.clip(dy_fallback, -1.0, 1.0))
            if abs(dx_fallback) > 1.0 or abs(dy_fallback) > 1.0:
                failures.append("centroid_clamped")
        else:
            dx = dy = 0.0
            failures.append("no_signal")

    return dict(
        x=xi + float(dx), y=yi + float(dy), amp=float(A),
        sigma=float(math.sqrt(sx * sy)), sigma_x=float(sx),
        sigma_y=float(sy), theta=float(theta), bg=float(background),
        r2=float(r2), fit_std_x=float(fit_std_x),
        fit_std_y=float(fit_std_y), snr=snr,
        width_ok=bool(width_ok), ok=bool(ok),
        reason=";".join(failures),
    )


def dedupe(pts, amps, min_dist):
    """Deterministic amplitude-ordered Euclidean non-maximum suppression."""
    points = np.asarray(pts, dtype=np.float64)
    scores = np.asarray(amps, dtype=np.float64)
    if len(points) < 2:
        return np.arange(len(points), dtype=int)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("pts must have shape (N, 2)")

    scores = np.where(np.isfinite(scores), scores, -np.inf)
    radius = max(0.0, float(min_dist) * LOC_CFG.DEDUPE_DIST_FRAC)
    tree = cKDTree(points)
    order = np.argsort(-scores, kind="mergesort")
    suppressed = np.zeros(len(points), dtype=bool)
    keep = []
    for index in order:
        if suppressed[index]:
            continue
        keep.append(int(index))
        neighbours = tree.query_ball_point(points[index], r=radius)
        suppressed[np.asarray(neighbours, dtype=int)] = True
    return np.asarray(sorted(keep), dtype=int)


def mask_scale(prob, level=None):
    """Typical mask inradius from resolved maxima of the mask EDT.

    Local maxima are used rather than one maximum per connected component so
    that touching discs do not collapse into one falsely enormous object.
    """
    level = LOC_CFG.MASK_LEVEL if level is None else float(level)
    binary = np.asarray(prob) > level
    if not binary.any():
        return 0.0, 0
    distance = ndi.distance_transform_edt(binary)
    maxima = peak_local_max(
        distance, min_distance=2, threshold_abs=1.0, labels=binary,
        exclude_border=False, p_norm=2,
    )
    if not len(maxima):
        return 0.0, 0
    radii = distance[maxima[:, 0], maxima[:, 1]].astype(np.float64)
    # Small contour irregularities can create shallow maxima.  Retain maxima
    # compatible with the dominant object scale without assuming equal radii.
    reference = float(np.percentile(radii, 75))
    radii = radii[radii >= max(1.0, 0.5 * reference)]
    if not len(radii):
        return 0.0, 0
    return float(np.median(radii)), int(len(radii))


def _local_background_plane(patch, X, Y, outside):
    """Least-squares planar background estimated outside the centroid window."""
    if int(np.sum(outside)) < 3:
        background = float(np.median(patch))
        return np.full_like(patch, background, dtype=np.float64), background
    design = np.column_stack((np.ones(int(np.sum(outside))),
                              X[outside], Y[outside]))
    values = patch[outside]
    coef, *_ = np.linalg.lstsq(design, values, rcond=None)
    plane = coef[0] + coef[1] * X + coef[2] * Y
    return plane, float(coef[0])


def locate_blobs(prob, image=None, level=None, border=None, window_frac=None):
    """Locate centres of broad/disc-like predicted mask regions.

    Watershed supplies one proposal per distance-transform maximum.  When an
    intensity image is supplied, Gaussian refinement gives the final subpixel
    centre.  Otherwise a soft, background-corrected centroid is used.  Mask
    radius and Gaussian sigma are returned as different quantities.
    """
    level = LOC_CFG.MASK_LEVEL if level is None else float(level)
    border = LOC_CFG.BORDER_PX if border is None else int(border)
    window_frac = LOC_CFG.WINDOW_FRAC if window_frac is None else float(window_frac)
    probability = np.asarray(prob, dtype=np.float64)
    source = probability if image is None else np.asarray(image, dtype=np.float64)
    cols = [
        "x", "y", "score", "amp", "sigma", "sigma_x", "sigma_y",
        "theta", "radius_px", "area", "r2", "fit_std_x", "fit_std_y",
        "snr", "width_ok", "ok", "reason",
    ]
    binary = probability > level
    if not binary.any():
        return pd.DataFrame(columns=cols)

    distance = ndi.distance_transform_edt(binary)
    radius_typical, _ = mask_scale(probability, level)
    radius_typical = max(radius_typical, 1.0)
    marker_separation = int(max(2, round(0.8 * radius_typical)))
    marker_coords = peak_local_max(
        distance, min_distance=marker_separation,
        threshold_abs=max(0.5, 0.35 * radius_typical), labels=binary,
        exclude_border=False, p_norm=2,
    )
    markers = np.zeros(probability.shape, dtype=np.int32)
    for marker_id, (yy, xx) in enumerate(marker_coords, 1):
        markers[yy, xx] = marker_id
    labels = (watershed(-distance, markers, mask=binary)
              if markers.any() else ndi.label(binary)[0])

    regions = []
    for label_id, slc in enumerate(ndi.find_objects(labels), 1):
        if slc is None:
            continue
        region = labels[slc] == label_id
        area = int(region.sum())
        if area < 6:
            continue
        yy, xx = np.mgrid[slc[0].start:slc[0].stop,
                          slc[1].start:slc[1].stop]
        cy0 = float(np.sum(region * yy) / area)
        cx0 = float(np.sum(region * xx) / area)
        local_radius = float(np.max(distance[slc][region]))
        regions.append((label_id, slc, region, area, cx0, cy0, local_radius))

    if not regions:
        return pd.DataFrame(columns=cols)
    rough_xy = np.asarray([[r[4], r[5]] for r in regions], dtype=np.float64)
    if len(rough_xy) > 1:
        nn_dist, _ = cKDTree(rough_xy).query(rough_xy, k=2)
        nearest = nn_dist[:, 1]
    else:
        nearest = np.full(1, np.inf)

    h, w = probability.shape
    records = []
    for region_index, (_, slc, region, area, cx0, cy0, local_radius) in enumerate(regions):
        score = float(np.max(probability[slc][region]))
        radius_eq = float(math.sqrt(area / math.pi))

        # Prefer a physically interpretable fit to the intensity image.  The
        # segmentation disc only proposes which column to fit.
        if image is not None:
            rec = refine_peak(source, cx0, cy0)
            rec.update(score=score, radius_px=radius_eq, area=area)
            records.append(rec)
            continue

        radius = max(2.0, window_frac * local_radius)
        if np.isfinite(nearest[region_index]):
            radius = min(radius,
                         LOC_CFG.WINDOW_MAX_SPACING_FRAC * nearest[region_index])
        radius = max(2.0, radius)
        R = int(math.ceil(radius))
        yi, xi = int(round(cy0)), int(round(cx0))
        if not (R <= yi < h - R and R <= xi < w - R):
            records.append(dict(
                x=cx0, y=cy0, score=score, amp=np.nan, sigma=np.nan,
                sigma_x=np.nan, sigma_y=np.nan, theta=np.nan,
                radius_px=radius_eq, area=area, r2=np.nan,
                fit_std_x=np.nan, fit_std_y=np.nan, snr=np.nan,
                width_ok=False, ok=False, reason="edge",
            ))
            continue

        patch = source[yi - R:yi + R + 1, xi - R:xi + R + 1]
        oy, ox = np.mgrid[-R:R + 1, -R:R + 1]
        disc = (ox * ox + oy * oy) <= radius * radius
        plane, background = _local_background_plane(patch, ox, oy, ~disc)
        weights = np.clip(patch - plane, 0.0, None) * disc
        total = float(weights.sum())
        if total <= 1e-12:
            cx, cy, ok, reason = cx0, cy0, False, "no_signal"
            amplitude = np.nan
        else:
            cx = xi + float(np.sum(weights * ox) / total)
            cy = yi + float(np.sum(weights * oy) / total)
            drift = math.hypot(cx - cx0, cy - cy0)
            ok = drift <= 0.5 * radius
            reason = "" if ok else "centroid_drift"
            if not ok:
                cx, cy = cx0, cy0
            amplitude = float(np.max(patch - plane))
        records.append(dict(
            x=cx, y=cy, score=score, amp=amplitude, sigma=np.nan,
            sigma_x=np.nan, sigma_y=np.nan, theta=np.nan,
            radius_px=radius_eq, area=area, r2=np.nan,
            fit_std_x=np.nan, fit_std_y=np.nan, snr=np.nan,
            width_ok=False, ok=bool(ok), reason=reason, bg=background,
        ))

    frame = pd.DataFrame(records)
    frame = frame[np.isfinite(frame.x) & np.isfinite(frame.y)]
    return frame[(frame.x > border) & (frame.y > border) &
                 (frame.x < w - border) & (frame.y < h - border)].reset_index(drop=True)


def locate_columns(prob, image=None, min_distance=None, thresh=None, border=None,
                   diagnose=False, mode=None):
    """Find and refine atomic-column centres with consistent Euclidean geometry."""
    mode = LOC_CFG.MODE if mode is None else str(mode).lower()
    if mode not in {"auto", "blob", "peak"}:
        raise ValueError("mode must be 'auto', 'blob', or 'peak'")
    border = LOC_CFG.BORDER_PX if border is None else int(border)
    threshold = LOC_CFG.PROB_THRESH if thresh is None else float(thresh)
    probability = np.asarray(prob, dtype=np.float64)
    source = probability if image is None else np.asarray(image, dtype=np.float64)
    if probability.ndim != 2 or source.shape != probability.shape:
        raise ValueError("prob and image must be same-shape 2-D arrays")

    radius_median, n_regions = mask_scale(probability)
    selected_mode = mode
    if selected_mode == "auto":
        selected_mode = ("blob" if radius_median >= LOC_CFG.BLOB_MIN_RADIUS
                         else "peak")

    if selected_mode == "blob":
        frame = locate_blobs(probability, image=image, border=border)
        finite = frame[np.isfinite(frame.x) & np.isfinite(frame.y)]
        clean = finite[finite.ok] if len(finite) and "ok" in finite else finite
        spacing_source = clean if len(clean) >= 2 else finite
        if len(spacing_source) > 1:
            xy = spacing_source[["x", "y"]].to_numpy(dtype=np.float64)
            nn, _ = cKDTree(xy).query(xy, k=2)
            spacing = int(max(3, round(float(np.median(nn[:, 1])))))
        elif min_distance is not None:
            spacing = int(max(3, round(float(min_distance))))
        else:
            spacing = int(max(3, round(4.0 * max(radius_median, 1.0))))
        if diagnose:
            clean_count = int(frame.ok.sum()) if len(frame) else 0
            print(
                f"    blob mode: mask inradius {radius_median:.2f} px, "
                f"{n_regions} resolved mask maxima, {len(frame)} centres "
                f"({clean_count} clean), spacing {spacing} px, coverage "
                f"{float((probability > LOC_CFG.MASK_LEVEL).mean()):.4f}"
            )
        return frame, spacing

    smoothed = gaussian_filter(probability, LOC_CFG.SMOOTH_SIGMA)
    if min_distance is not None:
        spacing = int(max(3, round(float(min_distance))))
    elif LOC_CFG.MIN_DISTANCE is not None:
        spacing = int(max(3, round(float(LOC_CFG.MIN_DISTANCE))))
    else:
        spacing = estimate_min_distance(smoothed)
    peak_distance = int(max(2, round(LOC_CFG.PEAK_DIST_FRAC * spacing)))

    # Threshold the original calibrated probability.  Smoothing is used only
    # to stabilize the location/ranking of peaks, so a one-pixel probability
    # peak is not rejected merely because convolution reduced its height.
    eligible = probability >= threshold
    if not eligible.any():
        return pd.DataFrame(columns=[
            "x", "y", "score", "amp", "sigma", "sigma_x", "sigma_y",
            "theta", "bg", "r2", "fit_std_x", "fit_std_y", "snr",
            "width_ok", "ok", "reason",
        ]), spacing
    coords = peak_local_max(
        smoothed, min_distance=peak_distance,
        threshold_abs=float(np.min(smoothed)) - np.finfo(float).eps,
        labels=eligible, exclude_border=border, p_norm=2,
    )
    if diagnose:
        print(
            f"    peak mode: mask inradius {radius_median:.2f} px, coverage "
            f"{float((probability > LOC_CFG.MASK_LEVEL).mean()):.4f}, "
            f"spacing {spacing} px, NMS radius {peak_distance} px, "
            f"raw peaks {len(coords)}"
        )
    columns = [
        "x", "y", "score", "amp", "sigma", "sigma_x", "sigma_y",
        "theta", "bg", "r2", "fit_std_x", "fit_std_y", "snr",
        "width_ok", "ok", "reason",
    ]
    if not len(coords):
        return pd.DataFrame(columns=columns), spacing

    records = []
    scores = []
    for yy, xx in coords:
        rec = refine_peak(source, xx, yy)
        score = float(smoothed[yy, xx])
        rec["score"] = score
        records.append(rec)
        scores.append(score)
    frame = pd.DataFrame(records)
    keep = dedupe(frame[["x", "y"]].to_numpy(), np.asarray(scores), spacing)
    frame = frame.iloc[keep].reset_index(drop=True)
    h, w = probability.shape
    frame = frame[np.isfinite(frame.x) & np.isfinite(frame.y)]
    frame = frame[(frame.x > border) & (frame.y > border) &
                  (frame.x < w - border) & (frame.y < h - border)].reset_index(drop=True)
    if diagnose:
        print(f"    after deterministic NMS/border: {len(frame)}")
    return frame, spacing


def _unique_knn_pairs(xy, k):
    """Unique undirected k-nearest-neighbour edges and their lengths."""
    n = len(xy)
    k_eff = int(min(max(1, k), n - 1))
    _, indices = cKDTree(xy).query(xy, k=k_eff + 1)
    if indices.ndim == 1:
        indices = indices[:, None]
    pairs = set()
    for i in range(n):
        for j in np.atleast_1d(indices[i])[1:]:
            j = int(j)
            if j == i or j < 0 or j >= n:
                continue
            pairs.add((min(i, j), max(i, j)))
    pair_array = np.asarray(sorted(pairs), dtype=int)
    if not len(pair_array):
        return np.empty((0, 2), dtype=int), np.empty(0, dtype=np.float64)
    delta = xy[pair_array[:, 1]] - xy[pair_array[:, 0]]
    distance = np.linalg.norm(delta, axis=1)
    valid = np.isfinite(distance) & (distance > 0)
    return pair_array[valid], distance[valid]


def first_shell_spacing(xy, k=8, pixel_size=None, pixel_size_sigma=0.0):
    """Robust shortest-neighbour shell distance with honest uncertainty fields.

    The shell is centred on the median one-nearest-neighbour distance and is
    populated from unique, undirected kNN edges.  This avoids histogram-bin
    dependence, double-counted pairs and accidental inclusion of every shell
    when a histogram valley is absent.

    ``sigma_*`` is the robust physical spread of bonds in the selected shell.
    ``se_*`` is only an approximate site-level standard error of the median.
    Independent-frame/block bootstrap uncertainty is preferred for reporting.
    """
    pixel_size = CFG.PIXEL_SIZE_A if pixel_size is None else float(pixel_size)
    pixel_size_sigma = float(pixel_size_sigma)
    points_input = np.asarray(xy, dtype=np.float64)
    out = dict(
        n=int(len(points_input)) if points_input.ndim else 0,
        n_valid=0, d_px=np.nan, mad_px=np.nan, sigma_px=np.nan,
        se_px=np.nan, ci95_px=(np.nan, np.nan), d_A=np.nan,
        sigma_A=np.nan, se_A=np.nan, ci95_A=(np.nan, np.nan),
        n_pairs=0, n_sites=0, ok=False, reason="",
        uncertainty_note=(
            "sigma is shell spread; se is a site-count approximation; "
            "bootstrap independent frames/regions for final uncertainty"
        ),
    )
    if points_input.ndim != 2 or points_input.shape[1] != 2:
        out["reason"] = "xy must have shape (N, 2)"
        return out
    points = points_input[np.isfinite(points_input).all(axis=1)]
    out["n_valid"] = int(len(points))
    if len(points) < 3:
        out["reason"] = f"only {len(points)} finite columns; need at least 3"
        return out
    if not np.isfinite(pixel_size) or pixel_size <= 0:
        out["reason"] = "pixel_size must be positive and finite"
        return out
    if pixel_size_sigma < 0 or not np.isfinite(pixel_size_sigma):
        out["reason"] = "pixel_size_sigma must be finite and non-negative"
        return out

    tree = cKDTree(points)
    nearest, _ = tree.query(points, k=2)
    nearest = nearest[:, 1]
    nearest = nearest[np.isfinite(nearest) & (nearest > 0)]
    if len(nearest) < 3:
        out["reason"] = "too few non-zero nearest-neighbour distances"
        return out
    seed = float(np.median(nearest))
    seed_mad = _robust_sigma(nearest)
    if not np.isfinite(seed) or seed <= 0:
        out["reason"] = "invalid nearest-neighbour spacing seed"
        return out

    pairs, distances = _unique_knn_pairs(points, k=k)
    half_width = max(
        LOC_CFG.SHELL_MIN_HALF_WIDTH_FRAC * seed,
        LOC_CFG.SHELL_MAD_MULTIPLIER * max(seed_mad, 0.0),
    )
    half_width = min(half_width, LOC_CFG.SHELL_MAX_HALF_WIDTH_FRAC * seed)
    select = np.abs(distances - seed) <= half_width
    shell_pairs = pairs[select]
    shell = distances[select]
    minimum_pairs = max(3, int(math.ceil(0.25 * len(points))))
    if len(shell) < minimum_pairs:
        out["reason"] = (
            f"only {len(shell)} unique edges support the shortest shell; "
            f"need at least {minimum_pairs}"
        )
        return out

    # One robust recentering step removes sensitivity to the preliminary seed.
    median = float(np.median(shell))
    mad = float(np.median(np.abs(shell - median)))
    sigma = float(1.4826 * mad)
    if sigma / median > LOC_CFG.MAX_SHELL_REL_SPREAD:
        out["reason"] = (
            f"shortest shell is unresolved: robust relative spread "
            f"{sigma / median:.3f}"
        )
        return out

    sites = np.unique(shell_pairs.ravel())
    n_sites = int(len(sites))
    # For independent Gaussian observations, SE(median) ~= 1.2533*sigma/sqrt(N).
    # Pair distances are correlated, so use the more conservative number of
    # contributing sites rather than the larger number of edges.
    se_px = (1.253314 * sigma / math.sqrt(n_sites)
             if n_sites > 0 else np.nan)
    se_A = math.sqrt((pixel_size * se_px) ** 2 +
                     (median * pixel_size_sigma) ** 2)
    d_A = median * pixel_size
    ci95_px = (median - 1.96 * se_px, median + 1.96 * se_px)
    ci95_A = (d_A - 1.96 * se_A, d_A + 1.96 * se_A)
    out.update(
        d_px=median, mad_px=mad, sigma_px=sigma, se_px=float(se_px),
        ci95_px=tuple(map(float, ci95_px)), d_A=float(d_A),
        sigma_A=float(sigma * pixel_size), se_A=float(se_A),
        ci95_A=tuple(map(float, ci95_A)), n_pairs=int(len(shell)),
        n_sites=n_sites, ok=True,
    )
    return out


def affine_strain_from_matches(reference_xy, measured_xy, weights=None):
    """Fit ``measured = F @ reference + t`` and return proper 2-D strain.

    Inputs must be corresponding reference/measured column coordinates.  Use
    this globally or on a local neighbourhood.  Small strain is
    ``0.5*(F + F.T) - I``; Green-Lagrange strain is
    ``0.5*(F.T @ F - I)``.  A scalar neighbour distance cannot replace this.
    """
    reference_all = np.asarray(reference_xy, dtype=np.float64)
    measured_all = np.asarray(measured_xy, dtype=np.float64)
    if (reference_all.ndim != 2 or reference_all.shape[1] != 2 or
            measured_all.shape != reference_all.shape):
        raise ValueError("reference_xy and measured_xy must both have shape (N, 2)")
    finite = (np.isfinite(reference_all).all(axis=1) &
              np.isfinite(measured_all).all(axis=1))
    reference = reference_all[finite]
    measured = measured_all[finite]
    if len(reference) < 3:
        raise ValueError("at least three finite matched, non-collinear points are required")

    if weights is None:
        weight = np.ones(len(reference), dtype=np.float64)
    else:
        weight_all = np.asarray(weights, dtype=np.float64)
        if weight_all.ndim != 1 or len(weight_all) != len(reference_all):
            raise ValueError("weights must contain one value per input match")
        weight = weight_all[finite]
        if np.any(~np.isfinite(weight)) or np.any(weight <= 0):
            raise ValueError("weights must be finite and strictly positive")
    weight_sum = float(np.sum(weight))
    reference_mean = np.sum(reference * weight[:, None], axis=0) / weight_sum
    measured_mean = np.sum(measured * weight[:, None], axis=0) / weight_sum
    reference_centered = reference - reference_mean
    measured_centered = measured - measured_mean
    root_weight = np.sqrt(weight)
    weighted_reference = reference_centered * root_weight[:, None]
    weighted_measured = measured_centered * root_weight[:, None]
    coefficient, _, rank, singular = np.linalg.lstsq(
        weighted_reference, weighted_measured, rcond=None
    )
    if rank < 2:
        raise ValueError("reference points are collinear or geometrically degenerate")

    F = coefficient.T
    translation = measured_mean - F @ reference_mean
    predicted = reference @ coefficient + translation
    residual = measured - predicted
    rmse = float(np.sqrt(np.mean(np.sum(residual * residual, axis=1))))
    identity = np.eye(2)
    small_strain = 0.5 * (F + F.T) - identity
    green_lagrange = 0.5 * (F.T @ F - identity)
    infinitesimal_rotation = 0.5 * (F - F.T)
    condition = float(singular[0] / singular[-1]) if singular[-1] > 0 else np.inf
    return dict(
        F=F, translation=translation, small_strain=small_strain,
        green_lagrange_strain=green_lagrange,
        infinitesimal_rotation=infinitesimal_rotation,
        exx=float(small_strain[0, 0]), eyy=float(small_strain[1, 1]),
        exy=float(small_strain[0, 1]),
        rotation_xy=float(infinitesimal_rotation[1, 0]),
        rmse_px=rmse, n=int(len(reference)), design_condition=condition,
        ok=bool(np.isfinite(condition) and condition < 1e10),
    )


def localize_from_model(ckpt, arch, stem_or_array, use_denoised=True,
                        diagnose=False):
    """Run model inference and localize one frame.

    If denoised intensities are used for fitting, quantify denoiser-induced
    localization bias on synthetic ground truth and also report the raw-image
    result.  Denoising changes the estimator, not merely its visual appearance.
    """
    if torch is None:
        raise ImportError("PyTorch is required only for localize_from_model")
    if isinstance(stem_or_array, str):
        image = norm01(load_gray(noisy_map[stem_or_array]))
    else:
        image = norm01(np.asarray(stem_or_array, dtype=np.float32))
    model = model_from_checkpoint(ckpt, arch)
    with torch.no_grad():
        denoised_tensor, segmentation = split_heads(
            model(torch.from_numpy(image)[None, None].to(CFG.DEVICE))
        )
        probability = seg_prob(segmentation)[0, 0].float().cpu().numpy()
        denoised = (norm01(denoised_tensor[0, 0].float().cpu().numpy())
                    if denoised_tensor is not None else None)
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    source = denoised if (use_denoised and denoised is not None) else image
    peaks, spacing = locate_columns(probability, source, diagnose=diagnose)
    return dict(
        image=image, denoised=denoised, prob=probability, peaks=peaks,
        min_distance=spacing,
        localization_source=("denoised" if source is denoised else "raw"),
    )


print("7a ready:", [
    name for name in (
        "estimate_min_distance", "refine_peak", "locate_blobs",
        "locate_columns", "first_shell_spacing",
        "affine_strain_from_matches", "localize_from_model",
    ) if callable(globals().get(name))
])


In [ ]:
# ============================================================
# 7b. Choose demonstration frames by column count
# ============================================================
DEMO_SURVEY_N   = 1500     # frames to survey; the whole fold if you can wait
DEMO_MIN_COLS   = 60
DEMO_N_FRAMES   = 5
DEMO_SEED       = 0
DEMO_MIN_GAP    = 25       # minimum separation in stem order between picks
DEMO_CACHE      = CFG.OUT_ROOT / "tables" / "gt_column_counts.csv"


def gt_column_count(stem, return_points=False):
    """Columns in the gaussianMask, independent of any model.

    Goes through locate_columns so the survey counts the same way the scoring
    does.
    """
    if stem not in gauss_gt_map:
        return (None, None) if return_points else None
    g = norm01(load_gray(gauss_gt_map[stem]))
    df, md = locate_columns(g, g)
    return (len(df), df) if return_points else len(df)


_demo_fold = int(MASTER.loc[MASTER[PRIMARY].idxmax()].fold) if len(MASTER) else 1
_, _va = fold_indices(_demo_fold)
survey = [common[i] for i in _va[:DEMO_SURVEY_N]]

# counting 1500 masks takes a few minutes, so cache it
cache = {}
if DEMO_CACHE.exists():
    _c = pd.read_csv(DEMO_CACHE)
    cache = dict(zip(_c.stem.astype(str), _c.n_columns))
counts, new = {}, 0
for s in survey:
    if s in cache:
        counts[s] = int(cache[s])
        continue
    n = gt_column_count(s)
    if n is not None:
        counts[s] = n
        new += 1
if new:
    pd.DataFrame({"stem": list(counts), "n_columns": list(counts.values())}) \
      .to_csv(DEMO_CACHE, index=False)
if not counts:
    raise RuntimeError(f"no {CFG.GT_POS} masks for the validation stems")

vals = np.array(list(counts.values()))
edges = [10, 30, 60, 120, 300]
labels = ["<10", "10-29", "30-59", "60-119", "120-299", "300+"]
hist = Counter(np.digitize(vals, edges))
print(f"surveyed {len(counts)} held-out frames from fold {_demo_fold} "
      f"({new} newly counted)")
print(f"  columns per frame: min {vals.min()}, median {int(np.median(vals))}, "
      f"90th pct {int(np.percentile(vals, 90))}, max {vals.max()}")
print("  " + "  ".join(f"{labels[k]}: {hist.get(k, 0)}" for k in range(len(labels))))
frac = float((vals >= DEMO_MIN_COLS).mean())
print(f"  at least {DEMO_MIN_COLS} columns: {int((vals >= DEMO_MIN_COLS).sum())}"
      f"/{len(vals)} = {frac:.1%} of the fold")
print(f"\nsections 10 to 13 can only run on that {frac:.1%}. Everything they "
      f"report describes the dense end of the dataset, and the benchmark in "
      f"sections 5, 6 and 14 covers all of it.")

eligible = sorted([s for s, n in counts.items() if n >= DEMO_MIN_COLS],
                  key=counts.get, reverse=True)
if not eligible:
    DEMO_MIN_COLS = int(np.percentile(vals, 99))
    eligible = sorted([s for s, n in counts.items() if n >= DEMO_MIN_COLS],
                      key=counts.get, reverse=True)
    print(f"  nothing reached the target; lowered it to {DEMO_MIN_COLS}")

# diversity: consecutive stems are the same structure at different noise seeds,
# so picking the top five by column count returns five copies of one simulation
order = {s: i for i, s in enumerate(survey)}
rng = np.random.default_rng(DEMO_SEED)
shuffled = list(rng.permutation(eligible))
DEMO_STEMS = []
for s in shuffled:
    if all(abs(order[s] - order[t]) >= DEMO_MIN_GAP for t in DEMO_STEMS):
        DEMO_STEMS.append(s)
    if len(DEMO_STEMS) == DEMO_N_FRAMES:
        break
if len(DEMO_STEMS) < DEMO_N_FRAMES:      # pool too small for the gap rule
    for s in shuffled:
        if s not in DEMO_STEMS:
            DEMO_STEMS.append(s)
        if len(DEMO_STEMS) == DEMO_N_FRAMES:
            break
DEMO_STEMS = sorted(DEMO_STEMS, key=counts.get, reverse=True)
DEMO_STEM = DEMO_STEMS[0]

print(f"\nchosen: " + ", ".join(f"{s} ({counts[s]} cols, #{order[s]})"
                                for s in DEMO_STEMS))
spread = len({counts[s] for s in DEMO_STEMS})
if spread == 1:
    print(f"  WARNING: all {DEMO_N_FRAMES} have the same column count, so they "
          f"are probably one structure at different noise seeds. Raise "
          f"DEMO_MIN_GAP or lower DEMO_MIN_COLS to get real variety.")
print(f"DEMO_STEM = {DEMO_STEM}")

# the survey is only as good as the counter, so look at two
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
for a, s in zip(ax, DEMO_STEMS[:2]):
    n, pts = gt_column_count(s, return_points=True)
    a.imshow(norm01(load_gray(gauss_gt_map[s])), cmap="magma")
    a.scatter(pts.x, pts.y, s=28, facecolors="none", edgecolors="cyan", lw=0.8)
    a.set_title(f"{s}: {n} columns", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 8. Atomic-column localization and interatomic spacing

The original four steps, with the failure modes closed:

1. Peaks come from the probability map after a light Gaussian, with
   `min_distance` derived from the lattice rather than hard-coded, and a
   border exclusion so half-columns at the edge do not enter the statistics.
2. Sub-pixel refinement is a bounded 2-D elliptical Gaussian with a local
   background term, fitted on the denoised image when the denoising head is
   usable and on the raw frame otherwise. Every fit returns an R-squared, a
   width, and the shift from the integer peak; fits that move more than 1.5 px
   or fall below an R-squared floor are flagged and fall back to a
   centre-of-mass estimate instead of being silently kept.
3. Duplicates within half the minimum distance are merged, keeping the
   brighter column.
4. Nearest-neighbour spacing is the robust first-shell mode, not the raw
   k = 1 mean: the k = 1 mean is pulled down by split peaks and up by missed
   columns. The first shell is cut at the first minimum of the neighbour
   distance histogram and summarised by a median with a MAD spread.

In [ ]:
# ============================================================
# 8. Localization demonstration and mask-offset measurement
# ============================================================
LOC_DEMO_FOLD   = PANEL_FOLD if "PANEL_FOLD" in dir() else 2
LOC_DEMO_FRAMES = 3


def mask_offset(prob, truth, thr=0.5, min_area=8):
    """Centroid shift between each predicted disc and its true disc.

    The error rings in the section 7 panels are red on one edge and yellow on
    the opposite edge, which is a displaced mask rather than a noisy one. This
    measures that displacement per column. A large mean with a small spread is
    a systematic offset and could be calibrated out; a small mean with a large
    spread is per-column error and cannot.
    """
    pred = prob > thr
    lt, n = ndi.label(truth > 0.5)
    if n == 0:
        return None
    offs, areas = [], []
    for i, sl in enumerate(ndi.find_objects(lt), 1):
        t = lt[sl] == i
        p = pred[sl]
        if p.sum() < min_area or t.sum() < min_area:
            continue
        yy, xx = np.mgrid[sl[0].start:sl[0].stop, sl[1].start:sl[1].stop]
        offs.append([(p * xx).sum() / p.sum() - (t * xx).sum() / t.sum(),
                     (p * yy).sum() / p.sum() - (t * yy).sum() / t.sum()])
        areas.append((p.sum(), t.sum()))
    if not offs:
        return None
    o = np.array(offs)
    a = np.array(areas, float)
    return dict(n=len(o), dx=float(o[:, 0].mean()), dy=float(o[:, 1].mean()),
                sd_x=float(o[:, 0].std()), sd_y=float(o[:, 1].std()),
                mag=float(np.hypot(o[:, 0].mean(), o[:, 1].mean())),
                scatter=float(np.hypot(o[:, 0].std(), o[:, 1].std())),
                area_ratio=float((a[:, 0] / np.maximum(a[:, 1], 1)).mean()),
                offsets=o)


def localization_report(cond, arch, ckpt, stems, fold):
    """Detected versus true positions for one model, over several frames."""
    rows, per_frame = [], {}
    for stem in stems:
        if stem not in gauss_gt_map:
            continue
        R = localize_from_model(ckpt, arch, stem)
        g = norm01(load_gray(gauss_gt_map[stem]))
        gdf, gmd = locate_columns(g, g)
        truth = to_binary(load_gray(mask_map[stem]))
        off = mask_offset(R["prob"], truth)
        sp = first_shell_spacing(R["peaks"][["x", "y"]].values)
        gsp = first_shell_spacing(gdf[["x", "y"]].values)
        rec = dict(condition=cond, arch=arch, fold=fold, stem=stem,
                   n_pred=len(R["peaks"]), n_true=len(gdf),
                   d_px=sp.get("d_px", np.nan), d_true=gsp.get("d_px", np.nan))
        if sp["ok"] and gsp["ok"]:
            rec["spacing_err_pct"] = 100 * (sp["d_px"] - gsp["d_px"]) / gsp["d_px"]
        if off:
            rec.update(off_dx=off["dx"], off_dy=off["dy"], off_mag=off["mag"],
                       off_scatter=off["scatter"], area_ratio=off["area_ratio"])
        if callable(globals().get("detection_metrics")):
            m = detection_metrics(R["peaks"][["x", "y"]].values,
                                  gdf[["x", "y"]].values)
            rec.update(f1=m["f1"], rmse_px=m["rmse_px"],
                       bias_x=m["bias_x"], bias_y=m["bias_y"])
        rows.append(rec)
        per_frame[stem] = (R, gdf, off)
    return pd.DataFrame(rows), per_frame


R = None
if not len(MASTER):
    print("MASTER is empty; run section 5c first.")
else:
    stems = (DEMO_STEMS[:LOC_DEMO_FRAMES] if "DEMO_STEMS" in dir()
             else [common[fold_indices(LOC_DEMO_FOLD)[1][0]]])
    models = [(c, a, MASTER[(MASTER.condition == c) & (MASTER.arch == a) &
                            (MASTER.fold == LOC_DEMO_FOLD)])
              for c in (BEST_CONDITIONS if "BEST_CONDITIONS" in dir()
                        else CFG.CONDITIONS)
              for a in CFG.ARCHS]
    models = [(c, a, s.iloc[0].ckpt) for c, a, s in models if len(s)]
    print(f"{len(models)} models at fold {LOC_DEMO_FOLD}, frames "
          + ", ".join(stems))

    LOC = []
    store = {}
    for cond, arch, ck in models:
        try:
            df, pf = localization_report(cond, arch, ck, stems, LOC_DEMO_FOLD)
        except Exception as e:
            print(f"  {cond} {arch}: {type(e).__name__}: {e}")
            continue
        LOC.append(df)
        store[(cond, arch)] = pf
    LOC = pd.concat(LOC, ignore_index=True) if LOC else pd.DataFrame()

    if len(LOC):
        agg = LOC.groupby(["condition", "arch"]).agg(
            n_pred=("n_pred", "mean"), n_true=("n_true", "mean"),
            spacing_err=("spacing_err_pct", "mean"),
            off_mag=("off_mag", "mean"), off_scatter=("off_scatter", "mean"),
            area_ratio=("area_ratio", "mean"),
            **({"f1": ("f1", "mean"), "rmse_px": ("rmse_px", "mean")}
               if "f1" in LOC else {}))
        print(f"\nover {len(stems)} frames")
        print(agg.round(4).to_string())
        print("\n  off_mag is the mean displacement of a predicted disc from "
              "its true disc; off_scatter is how much that varies between "
              "columns. A large off_mag with a small off_scatter is a "
              "systematic shift and could be calibrated out. area_ratio above "
              "1 means the predicted discs are larger than the true ones.")
        worst = agg.off_mag.idxmax()
        print(f"  largest mean offset: {worst[0]} {worst[1]} at "
              f"{agg.off_mag.max():.3f} px")
        if callable(globals().get("save_table")):
            save_table(LOC, "localization_report.csv")

        # ---- the offset field, which the panels show as one-sided rings ----
        best_key = agg.off_mag.idxmin()
        fig, axes = plt.subplots(2, len(store), figsize=(3.2 * len(store), 6.6))
        axes = np.atleast_2d(axes)
        for j, ((cond, arch), pf) in enumerate(store.items()):
            stem0 = stems[0]
            if stem0 not in pf:
                continue
            Rz, gdfz, offz = pf[stem0]
            axes[0, j].imshow(Rz["image"], cmap="gray")
            axes[0, j].scatter(gdfz.x, gdfz.y, s=50, facecolors="none",
                               edgecolors="cyan", lw=0.8)
            if len(Rz["peaks"]):
                axes[0, j].scatter(Rz["peaks"].x, Rz["peaks"].y, s=10,
                                   color="red")
            axes[0, j].set_title(f"{arch}\n{cond}", fontsize=8)
            if offz is not None:
                o = offz["offsets"]
                axes[1, j].axhline(0, color="0.7", lw=0.6)
                axes[1, j].axvline(0, color="0.7", lw=0.6)
                axes[1, j].scatter(o[:, 0], o[:, 1], s=14, alpha=0.7)
                axes[1, j].scatter([offz["dx"]], [offz["dy"]], s=90, marker="+",
                                   color="red", lw=1.8)
                lim = max(0.6, float(np.abs(o).max()) * 1.1)
                axes[1, j].set_xlim(-lim, lim); axes[1, j].set_ylim(-lim, lim)
                axes[1, j].set_aspect("equal")
                axes[1, j].set_title(f"offset {offz['mag']:.3f} px\n"
                                     f"scatter {offz['scatter']:.3f} px",
                                     fontsize=7.5)
                axes[1, j].tick_params(labelsize=6)
        for a in axes[0]:
            a.set_xticks([]); a.set_yticks([])
        axes[0, 0].set_ylabel(f"{stems[0]}\ncyan truth, red detected", fontsize=8)
        axes[1, 0].set_ylabel("per-column offset (px)", fontsize=8)
        plt.tight_layout()
        if callable(globals().get("save_fig")):
            save_fig(fig, "localization_offsets.png")
        else:
            plt.savefig(CFG.OUT_ROOT / "figures" / "localization_offsets.png",
                        dpi=150)
        plt.show()

        # keep one R for sections 10 to 13, from the model with the least offset
        pf = store.get(best_key)
        if pf and stems[0] in pf:
            R = pf[stems[0]][0]
            print(f"\nR set from {best_key[1]} {best_key[0]} on {stems[0]} "
                  f"({len(R['peaks'])} columns) for sections 10 to 13")

## 9. Sub-pixel localization precision, synthetic shift test

The improvement over shifting an image by a whole pixel and checking the peak
moves: the frame is shifted by a known sub-pixel amount in Fourier space,
noise is regenerated at several electron doses, and the fitted positions are
matched back to the shifted ground truth. That separates the two quantities
that a single "accuracy" number hides, which are the bias (systematic pull
toward the pixel grid, visible as an S-curve against the true sub-pixel
offset) and the jitter (the random scatter, which is what sets the strain
noise floor).

Both are reported against the shot-noise limit, so a model that beats the
Cramer-Rao bound for its own dose is reporting an artefact of over-smoothing
rather than a measurement.

In [ ]:
# ============================================================
# 9. Sub-pixel precision
# ============================================================
from scipy.ndimage import fourier_shift

PREC_FRAMES      = 3
PREC_DOSES       = (50, 200, 1000)
PREC_REPS        = 5          # independent noise realizations per (dose, shift)
PREC_SHIFTS      = np.arange(0.0, 1.0, 0.125)
MIN_REF_COLUMNS  = 20
MIN_MATCHES      = 10
PREC_SOURCES     = ("prob", "denoised", "raw")   # what the centroid is taken on


def fshift(img, dx, dy):
    return np.real(np.fft.ifft2(fourier_shift(np.fft.fft2(img), (dy, dx))))


def add_dose_noise(clean01, dose_e, read_noise=0.0, rng=None):
    """Exact Poisson at `dose_e` electrons per unit-intensity pixel.

    Exact, not a Gaussian approximation: at dose 50 the background pixels carry
    well under one electron, where sqrt(lam)*normal produces negative counts and
    the wrong distribution, which is precisely the regime this measurement is
    about.
    """
    rng = rng or np.random.default_rng(0)
    lam = np.clip(clean01, 0, 1) * dose_e
    noisy = rng.poisson(lam).astype(np.float32)
    if read_noise:
        noisy = noisy + rng.normal(0, read_noise, noisy.shape)
    return norm01(noisy)


def crb_sigma(dose_e, sigma_px, bg_per_px=0.0):
    """Gaussian-spot Cramer-Rao position bound, Thompson et al. 2002.

    sigma^2/N is the photon term, a^2/12N the pixelation term with a = 1 px, and
    the last term the background. N is the total count in the spot, about
    2*pi*sigma^2*dose, not the per-pixel dose.
    """
    s = float(sigma_px)
    N = max(2 * math.pi * s ** 2 * dose_e, 1.0)
    var = (s ** 2 + 1.0 / 12.0) / N
    if bg_per_px > 0:
        var += 8 * math.pi * s ** 4 * bg_per_px / (N ** 2)
    return float(math.sqrt(var))


def column_sigma_from_image(image, xy, fit_half=6):
    """Gaussian width of the intensity peaks in the image the model sees.

    Measured on the noisy frame, not on the noiseless one, and with a patch
    wide enough to contain the peak: a 9x9 window on a 16 px feature fits a
    plateau and returns a width that has nothing to do with the signal.
    """
    h, w = image.shape
    out = []
    for x, y in xy[:80]:
        xi, yi = int(round(x)), int(round(y))
        if not (fit_half <= xi < w - fit_half and fit_half <= yi < h - fit_half):
            continue
        r = refine_peak(image, xi, yi, half=fit_half)
        if r.get("ok") and np.isfinite(r.get("sigma", np.nan)):
            out.append(r["sigma"])
    return float(np.median(out)) if len(out) >= 5 else np.nan


def _match_error(pred_xy, want_xy, radius):
    if len(pred_xy) == 0:
        return None, 0.0
    d, j = cKDTree(pred_xy).query(want_xy, distance_upper_bound=radius)
    m = np.isfinite(d)
    if m.sum() < MIN_MATCHES:
        return None, float(m.mean())
    return pred_xy[j[m]] - want_xy[m], float(m.mean())


def _error_row(err, **extra):
    ex, ey = err[:, 0], err[:, 1]
    return dict(n=len(err),
                bias_x=float(np.median(ex)), bias_y=float(np.median(ey)),
                jitter_x=float(1.4826 * np.median(np.abs(ex - np.median(ex)))),
                jitter_y=float(1.4826 * np.median(np.abs(ey - np.median(ey)))),
                rmse=float(np.sqrt((err ** 2).sum(1).mean())), **extra)


def detector_floor(stems, shifts=PREC_SHIFTS):
    """The detector alone, on noiseless shifted ground-truth masks.

    No model, no noise, nothing to estimate. Whatever bias and jitter appear
    here belong to locate_columns, and no network can do better.
    """
    rows = []
    for stem in stems:
        if stem not in gauss_gt_map:
            continue
        g = norm01(load_gray(gauss_gt_map[stem]))
        gdf, gmd = locate_columns(g, g)
        if len(gdf) < MIN_REF_COLUMNS:
            continue
        gt = gdf[["x", "y"]].values
        for s in shifts:
            gs = fshift(g, s, 0.0)
            df, _ = locate_columns(gs, gs, min_distance=gmd)
            err, frac = _match_error(df[["x", "y"]].values,
                                     gt + np.array([s, 0.0]), gmd * 0.4)
            if err is None:
                continue
            rows.append(_error_row(err, stem=stem, shift=float(s),
                                   matched_frac=frac))
    return pd.DataFrame(rows)


@torch.no_grad()
def shift_precision(ckpt, arch, stems, shifts=PREC_SHIFTS, doses=PREC_DOSES,
                    reps=PREC_REPS, seed=0, sources=PREC_SOURCES, verbose=True):
    """Bias and jitter against a known sub-pixel shift.

    Each (dose, shift, rep) gets an independent Poisson realization, so the
    eight shift points are eight independent samples and `reps` controls how
    much of the noise averages out. Freezing one realization across shifts
    removes the shift-noise confound but leaves a single sample dressed up as
    eight.

    Three position sources are scored from the same forward pass: the
    segmentation probability alone, the model's denoised image weighted inside
    the mask, and the raw noisy frame weighted inside the mask. The difference
    between the last two is what the denoising head is worth for localization,
    which a single-source measurement cannot see.
    """
    stems = [stems] if isinstance(stems, str) else list(stems)
    model = model_from_checkpoint(ckpt, arch)
    rows, skips, detail = [], Counter(), []

    for stem in stems:
        clean = norm01(load_gray(clean_map[stem])) if stem in clean_map \
            else norm01(load_gray(noisy_map[stem]))
        if stem not in gauss_gt_map:
            skips["no ground-truth reference"] += 1
            continue
        g = norm01(load_gray(gauss_gt_map[stem]))
        gdf, md_ref = locate_columns(g, g)
        if len(gdf) < MIN_REF_COLUMNS:
            skips["reference too sparse"] += 1
            continue
        gt_xy = gdf[["x", "y"]].values

        for dose in doses:
            # the peak width the model sees at this dose, for the CRB
            probe = add_dose_noise(clean, dose, rng=np.random.default_rng(seed))
            sig = column_sigma_from_image(probe, gt_xy)
            if not np.isfinite(sig):
                sig = column_sigma_from_image(clean, gt_xy)
            if not np.isfinite(sig):
                skips["could not measure the peak width"] += 1
                continue
            bg = float(np.median(probe)) * dose

            for rep in range(reps):
                for s in shifts:
                    img = add_dose_noise(
                        fshift(clean, s, 0.0), dose,
                        rng=np.random.default_rng(seed + 7919 * rep +
                                                  131 * int(round(s * 8)) +
                                                  17 * dose))
                    den, seg = split_heads(
                        model(torch.from_numpy(img.astype(np.float32))[None, None]
                              .to(CFG.DEVICE)))
                    prob = seg_prob(seg)[0, 0].float().cpu().numpy()
                    dn = norm01(den[0, 0].float().cpu().numpy()) if den is not None else None
                    want = gt_xy + np.array([s, 0.0])

                    for src_name in sources:
                        if src_name == "prob":
                            weight = prob
                        elif src_name == "denoised":
                            if dn is None:
                                continue
                            weight = dn
                        else:
                            weight = img
                        pk, _ = locate_columns(prob, weight, min_distance=md_ref)
                        if len(pk) < MIN_MATCHES:
                            skips[f"{src_name}: too few columns"] += 1
                            continue
                        err, frac = _match_error(pk[["x", "y"]].values, want,
                                                 md_ref * 0.4)
                        if err is None:
                            skips[f"{src_name}: too few matches"] += 1
                            detail.append(f"{stem} dose {dose} shift {s:.3f} "
                                          f"{src_name}: matched {frac:.0%}")
                            continue
                        rows.append(_error_row(
                            err, stem=stem, dose=dose, rep=rep, shift=float(s),
                            source=src_name, matched_frac=frac, sigma_px=sig,
                            crb=crb_sigma(dose, sig, bg_per_px=bg)))
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if verbose and detail:
        for line in detail[:3]:
            print(f"      {line}")
    return pd.DataFrame(rows), skips


# ---- run ----
PRECISION = {}
ALL_SKIPS = Counter()
FLOOR = pd.DataFrame()
fj = np.nan

if not len(MASTER):
    print("MASTER is empty; run section 5c first.")
else:
    prec_stems = (DEMO_STEMS[:PREC_FRAMES] if "DEMO_STEMS" in dir() else [DEMO_STEM])
    print("frames: " + ", ".join(prec_stems))

    print("\ndetector floor (no model, no noise, shifted gaussianMask)")
    FLOOR = detector_floor(prec_stems)
    if len(FLOOR):
        fb = float(np.abs(FLOOR.bias_x).mean())
        fj = float(FLOOR.jitter_x.mean())
        rng_b = float(FLOOR.bias_x.max() - FLOOR.bias_x.min())
        print(f"  |bias| {fb:.4f} px, jitter {fj:.4f} px "
              f"({fj * CFG.PIXEL_SIZE_A * 100:.1f} pm), bias swing {rng_b:.4f} px")
        if fj > 0.05 or rng_b > 0.1:
            print("  the detector alone has this much error, so any model number "
                  "near it is measuring locate_columns rather than the network.")
        else:
            print("  the detector is clean; what follows belongs to the models.")
    else:
        print("  could not measure; no frame had enough reference columns")

    n_calls = len(PREC_DOSES) * PREC_REPS * len(PREC_SHIFTS) * len(prec_stems)
    print(f"\n{n_calls} forward passes per model, {len(PREC_SOURCES)} position "
          f"sources scored from each")

    for cond in BEST_CONDITIONS:
        for arch in BEST_ARCHS:
            f, ck = best_fold(cond, arch)
            if f is None:
                continue
            t0 = time.time()
            try:
                df, skips = shift_precision(ck, arch, prec_stems)
            except Exception as e:
                print(f"{cond:9s} {arch:11s} FAILED: {type(e).__name__}: {e}")
                continue
            ALL_SKIPS.update(skips)
            if not len(df):
                print(f"{cond:9s} {arch:11s} nothing usable. {dict(skips)}")
                continue
            PRECISION[(cond, arch)] = df
            print(f"\n{cond} {arch} fold {f}  ({time.time() - t0:.0f} s)")
            for src in [s for s in PREC_SOURCES if s in set(df.source)]:
                d = df[df.source == src]
                g = d.groupby("dose").agg(jit=("jitter_x", "mean"),
                                          bias=("bias_x", lambda v: np.abs(v).mean()),
                                          crb=("crb", "mean"),
                                          matched=("matched_frac", "mean"))
                sl = (np.polyfit(np.log(g.index.values), np.log(g.jit.values), 1)[0]
                      if len(g) > 1 else np.nan)
                print(f"  [{src}]  jitter ~ dose^{sl:+.2f}")
                for dose, r in g.iterrows():
                    fl = r.jit / fj if np.isfinite(fj) and fj > 0 else np.nan
                    print(f"   {dose:5d} e/px  jitter {r.jit:.4f} px "
                          f"({r.jit * CFG.PIXEL_SIZE_A * 100:5.1f} pm)  "
                          f"|bias| {r.bias:.4f}  CRB {r.crb:.4f} "
                          f"({r.jit / r.crb:5.1f}x)  floor ({fl:4.1f}x)  "
                          f"matched {r.matched:.0%}")

    if PRECISION:
        big = pd.concat([d.assign(condition=k[0], arch=k[1])
                         for k, d in PRECISION.items()])
        print("\nwhat the denoising head is worth for localization")
        piv = big.pivot_table(index=["condition", "arch"], columns="source",
                              values="jitter_x", aggfunc="mean")
        if {"denoised", "raw"} <= set(piv.columns):
            piv["gain_%"] = 100 * (piv["raw"] - piv["denoised"]) / piv["raw"]
        print(piv.round(4).to_string())
        print("  positive gain means the denoised image gives better positions "
              "than the raw frame through the same mask; near zero means the "
              "denoiser is cosmetic for this purpose.")
        big.to_csv(CFG.OUT_ROOT / "tables" / "subpixel_precision.csv", index=False)
    else:
        print(f"\ntotal skips: {dict(ALL_SKIPS)}")

if PRECISION:
    src_main = "denoised" if any("denoised" in set(d.source) for d in PRECISION.values()) else "prob"
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for (cond, arch), d0 in PRECISION.items():
        df = d0[d0.source == src_main]
        if not len(df):
            continue
        lab = f"{arch} {cond}"
        b = df.groupby("shift").bias_x.mean()
        axes[0].plot(b.index, b.values, marker="o", ms=3, label=lab)
        h = df.groupby("dose").jitter_x.mean()
        axes[1].plot(h.index, h.values, marker="s", ms=4, label=lab)
        axes[2].plot(h.index, (h / df.groupby("dose").crb.mean()).values,
                     marker="^", ms=4, label=lab)
    if len(FLOOR):
        fbc = FLOOR.groupby("shift").bias_x.mean()
        axes[0].plot(fbc.index, fbc.values, "k--", lw=2, label="detector floor")
        axes[1].axhline(FLOOR.jitter_x.mean(), color="k", ls=":", lw=2,
                        label="detector floor")
    c = list(PRECISION.values())[0]
    c = c[c.source == src_main].groupby("dose").crb.mean()
    axes[1].plot(c.index, c.values, "k--", label="Cramer-Rao")
    axes[2].axhline(1.0, color="k", ls="--")
    axes[0].set_xlabel("true sub-pixel shift (px)"); axes[0].set_ylabel("bias (px)")
    axes[0].set_title(f"bias vs shift ({src_main})"); axes[0].legend(fontsize=6)
    axes[1].set_xscale("log"); axes[1].set_yscale("log")
    axes[1].set_xlabel("dose (e/px)"); axes[1].set_ylabel("jitter (px)")
    axes[1].set_title("precision against the shot-noise limit")
    axes[1].legend(fontsize=6)
    axes[2].set_xscale("log"); axes[2].set_xlabel("dose (e/px)")
    axes[2].set_ylabel("jitter / CRB"); axes[2].set_title("1.0 is the limit")
    plt.tight_layout()
    plt.savefig(CFG.OUT_ROOT / "figures" / "subpixel_precision.png", dpi=150)
    plt.show()
    if len(FLOOR):
        FLOOR.to_csv(CFG.OUT_ROOT / "tables" / "detector_floor.csv", index=False)

## 10. Displacement and strain mapping from detected columns

Three steps. A basis is found from the first-shell neighbour vectors by
angular clustering, every column is given integer indices (m, n) in that
basis, and the displacement is the measured position minus the ideal lattice
position. Strain is then a local least-squares fit of the displacement
gradient over each column's neighbourhood, which gives exx, eyy, exy and the
rigid rotation.

The indexing step is what makes this a measurement rather than a picture:
without integer indices the displacement field is only defined up to an
arbitrary reference, and a strain map built on nearest-neighbour vectors
alone inherits every missed column as a fake strain spike.

In [ ]:
# ============================================================
# 10. Lattice indexing, displacement, strain
# ============================================================
from scipy.spatial import Delaunay

MIN_COLUMNS_FOR_LATTICE = 30
STRAIN_K = 8


def first_shell_vectors(xy, k=10):
    tree = cKDTree(xy)
    d, idx = tree.query(xy, k=min(k + 1, len(xy)))
    v = (xy[idx[:, 1:]] - xy[:, None, :]).reshape(-1, 2)
    r = np.hypot(v[:, 0], v[:, 1])
    hist, edges = np.histogram(r, bins=90, range=(0, float(np.percentile(r, 95))))
    sm = gaussian_filter(hist.astype(float), 1.5)
    pks, _ = find_peaks(sm, prominence=0.1 * sm.max())
    pk = int(pks[0]) if len(pks) else int(np.argmax(sm))
    valleys, _ = find_peaks(-sm[pk:], prominence=0.05 * sm.max())
    cut = pk + int(valleys[0]) if len(valleys) else len(sm) - 1
    return v[r <= edges[min(cut + 1, len(edges) - 1)]], float(edges[min(cut + 1, len(edges) - 1)])


def reduce_basis(a1, a2, n_iter=8):
    """Lagrange-Gauss reduction: the shortest, most orthogonal equivalent pair.

    Any integer unimodular combination of two basis vectors describes the same
    lattice, and a skewed choice makes exy pick up a shear that is a property
    of the bookkeeping rather than the specimen.
    """
    a1, a2 = np.asarray(a1, float), np.asarray(a2, float)
    for _ in range(n_iter):
        if a1 @ a1 > a2 @ a2:
            a1, a2 = a2, a1
        mu = round((a1 @ a2) / max(a1 @ a1, 1e-12))
        if mu == 0:
            break
        a2 = a2 - mu * a1
    if a1[0] * a2[1] - a1[1] * a2[0] < 0:
        a1, a2 = a2, a1
    return np.array([a1, a2])


def lattice_basis(xy, k=10, why=None, min_sep_deg=20.0):
    """Two primitive vectors from the angular distribution of first-shell bonds.

    Every candidate pair is scored on how well it actually indexes the points,
    not on how many bonds support the two directions. Support alone picks the
    top two directions always, which on a hexagonal lattice are 60 degrees
    apart and give a skewed cell.
    """
    why = why if why is not None else []
    if len(xy) < MIN_COLUMNS_FOR_LATTICE:
        why.append(f"{len(xy)} columns, need at least {MIN_COLUMNS_FOR_LATTICE}")
        return None
    v, rcut = first_shell_vectors(xy, k)
    if len(v) < 20:
        why.append(f"only {len(v)} first-shell bonds within {rcut:.1f} px")
        return None
    ang = np.mod(np.degrees(np.arctan2(v[:, 1], v[:, 0])), 180.0)
    hist, edges = np.histogram(ang, bins=180, range=(0, 180))
    sm = gaussian_filter(hist.astype(float), 3, mode="wrap")

    cands = []
    for i in np.argsort(sm)[::-1]:
        a = edges[i] + 0.5
        if all(min(abs(a - b), 180 - abs(a - b)) > min_sep_deg for b in cands):
            cands.append(a)
        if len(cands) == 5:
            break
    if len(cands) < 2:
        why.append("only one bond direction; no two-dimensional periodic order")
        return None

    def vec_for(a):
        sel = np.abs(np.mod(ang - a + 90, 180) - 90) < 12
        vv = v[sel]
        if len(vv) < 5:
            return None
        u = np.array([math.cos(math.radians(a)), math.sin(math.radians(a))])
        return np.median(vv * np.sign(vv @ u)[:, None], axis=0)

    origin = xy[np.argmin(np.hypot(*(xy - xy.mean(0)).T))]
    best = None
    for i, j in itertools.combinations(range(len(cands)), 2):
        v1, v2 = vec_for(cands[i]), vec_for(cands[j])
        if v1 is None or v2 is None:
            continue
        B = reduce_basis(v1, v2)
        det = abs(B[0, 0] * B[1, 1] - B[0, 1] * B[1, 0])
        if det < 1e-6 or np.linalg.cond(B) > 6:
            continue
        # score by how well this cell indexes the actual points: the median
        # distance from each column to its nearest lattice site
        mn = np.linalg.solve(B.T, (xy - origin).T).T
        resid = np.hypot(*(xy - (origin + np.round(mn) @ B)).T)
        score = float(np.median(resid))
        if best is None or score < best[0]:
            best = (score, B)
    if best is None:
        why.append("no pair of bond directions spans a well-conditioned cell")
        return None
    return best[1]


def index_lattice(xy, basis=None, origin=None, n_iter=6, fixed_basis=None,
                  tol_px=None, verbose=False):
    """Assign integer (m, n) and refine the basis by least squares.

    The acceptance window is recomputed from the residuals at every iteration
    rather than only allowed to shrink: a monotone window strangles itself when
    the first iteration starts from an unrefined basis, and never recovers.

    Colliding sites keep the column nearer the site instead of discarding both,
    which is what emptied the strain map when two detections landed in one cell.
    """
    xy = np.asarray(xy, float)
    why = []
    if fixed_basis is not None:
        basis = np.asarray(fixed_basis, float)
    elif basis is None:
        basis = lattice_basis(xy, why=why)
    else:
        basis = np.asarray(basis, float)
    if basis is None:
        raise RuntimeError("no lattice basis: " + "; ".join(why))

    origin = xy[np.argmin(np.hypot(*(xy - xy.mean(0)).T))] if origin is None else origin
    scale = float(np.linalg.norm(basis, axis=1).mean())
    keep = np.ones(len(xy), bool)
    tol = tol_px if tol_px is not None else 0.30 * scale

    for it in range(n_iter):
        ij = np.round(np.linalg.solve(basis.T, (xy - origin).T).T)
        resid = np.hypot(*(xy - (origin + ij @ basis)).T)
        if tol_px is None and it >= 1:
            med = float(np.median(resid))
            mad = float(np.median(np.abs(resid - med)))
            tol = float(np.clip(med + 4 * 1.4826 * mad, 0.04 * scale, 0.35 * scale))
        keep = resid < tol
        if verbose:
            print(f"    iter {it}: tol {tol:.2f} px, {int(keep.sum())}/{len(xy)} "
                  f"inside, median residual {np.median(resid):.3f} px")
        if keep.sum() < 12:
            break
        A = np.column_stack([np.ones(int(keep.sum())), ij[keep]])
        sol, *_ = np.linalg.lstsq(A, xy[keep], rcond=None)
        if fixed_basis is None:
            origin, basis = sol[0], reduce_basis(*sol[1:])
        else:
            origin = sol[0]

    ij = np.round(np.linalg.solve(basis.T, (xy - origin).T).T).astype(int)
    ideal = origin + ij @ basis
    disp = xy - ideal
    resid = np.hypot(*disp.T)

    # one column per site; the nearer one wins rather than both being dropped
    dup = np.zeros(len(xy), bool)
    seen = {}
    for i, key in enumerate(map(tuple, ij)):
        j = seen.get(key)
        if j is None:
            seen[key] = i
        elif resid[i] < resid[j]:
            dup[j] = True
            seen[key] = i
        else:
            dup[i] = True

    good = keep & ~dup
    return dict(basis=basis, origin=origin, ij=ij, ideal=ideal, disp=disp,
                keep=good, duplicate=dup, tol_px=float(tol),
                n_duplicate=int(dup.sum()), n_far=int((~keep).sum()),
                resid_px=float(np.median(resid[good])) if good.any() else np.nan)


def local_strain(xy, disp, k=STRAIN_K, keep=None, robust=True):
    """Displacement gradient over each column's neighbourhood.

    `k` falls back to whatever the frame can support; a fixed k of 8 returns an
    empty frame, and therefore an empty plot, whenever fewer than nine columns
    are indexed.
    """
    keep = np.ones(len(xy), bool) if keep is None else keep
    P, U = xy[keep], disp[keep]
    if len(P) < 5:
        return pd.DataFrame(columns=["exx", "eyy", "exy", "rot", "x", "y"])
    k = min(k, len(P) - 1)
    _, idx = cKDTree(P).query(P, k=k + 1)
    out = np.full((len(P), 4), np.nan)
    for i in range(len(P)):
        j = idx[i, 1:]
        dP, dU = P[j] - P[i], U[j] - U[i]
        if robust and callable(globals().get("robust_gradient")):
            G = robust_gradient(dP, dU)
        else:
            G = np.linalg.lstsq(np.column_stack([dP, np.ones(len(j))]),
                                dU, rcond=None)[0][:2].T
        out[i] = [G[0, 0], G[1, 1], 0.5 * (G[0, 1] + G[1, 0]),
                  0.5 * (G[0, 1] - G[1, 0])]
    df = pd.DataFrame(out, columns=["exx", "eyy", "exy", "rot"])
    df["x"], df["y"] = P[:, 0], P[:, 1]
    df.attrs = dict(k_used=k)
    return df


def strain_map(peaks, k=STRAIN_K, reference_basis=None, robust=True, verbose=False):
    xy = peaks[["x", "y"]].values.astype(float)
    L = index_lattice(xy, fixed_basis=reference_basis, verbose=verbose)
    S = local_strain(xy, L["disp"], k=k, keep=L["keep"], robust=robust)
    a1, a2 = L["basis"]
    n1, n2 = np.linalg.norm(a1), np.linalg.norm(a2)
    # the smallest strain this frame could report: localization jitter over one
    # lattice vector. Structure below this is noise, whatever the map shows.
    floor = float(L["resid_px"] / max(n1, 1e-9)) if np.isfinite(L["resid_px"]) else np.nan
    S.attrs = dict(a1_px=float(n1), a2_px=float(n2),
                   angle_deg=float(math.degrees(math.acos(
                       np.clip(a1 @ a2 / (n1 * n2), -1, 1)))),
                   indexed=int(L["keep"].sum()), total=len(xy),
                   resid_px=L["resid_px"], tol_px=L["tol_px"],
                   n_duplicate=L["n_duplicate"], n_far=L["n_far"],
                   strain_floor=floor, k_used=S.attrs.get("k_used", k))
    return S, L


def compare_basis_to_truth(stem, basis_fit):
    """Index the gaussianMask the same way and compare the two cells."""
    if stem not in gauss_gt_map:
        return None
    g = norm01(load_gray(gauss_gt_map[stem]))
    gdf, _ = locate_columns(g, g)
    if len(gdf) < MIN_COLUMNS_FOR_LATTICE:
        return None
    try:
        Lg = index_lattice(gdf[["x", "y"]].values)
    except RuntimeError:
        return None
    out = dict(a1_true=float(np.linalg.norm(Lg["basis"][0])),
               a2_true=float(np.linalg.norm(Lg["basis"][1])),
               resid_true=Lg["resid_px"], n_true=int(Lg["keep"].sum()))
    if callable(globals().get("basis_strain")):
        bs = basis_strain(basis_fit, Lg["basis"])
        out.update(d_exx=bs["exx"], d_eyy=bs["eyy"], d_exy=bs["exy"],
                   d_rot_deg=math.degrees(bs["rot"]))
    return out


# ---- demonstration ----
S = L = None
_stem = DEMO_STEM if "DEMO_STEM" in dir() else None
if not len(MASTER):
    print("MASTER is empty. Section 5c has not produced rows.")
elif R is None or not len(R["peaks"]):
    print("no peaks from section 8; nothing to index.")
elif len(R["peaks"]) < MIN_COLUMNS_FOR_LATTICE:
    print(f"{len(R['peaks'])} columns, below the {MIN_COLUMNS_FOR_LATTICE} a "
          f"lattice fit needs.")
else:
    print(f"indexing {len(R['peaks'])} columns from section 8")
    try:
        S, L = strain_map(R["peaks"], verbose=True)
    except RuntimeError as e:
        print(f"  {e}")

    if S is not None and not len(S):
        at = L
        print(f"  indexing kept {int(L['keep'].sum())} of {len(R['peaks'])} "
              f"columns ({L['n_far']} outside the {L['tol_px']:.2f} px window, "
              f"{L['n_duplicate']} sharing a site), too few for a gradient fit")
        S = None

    if S is not None:
        at = S.attrs
        print(f"\ncell: |a1| {at['a1_px']:.3f} px, |a2| {at['a2_px']:.3f} px, "
              f"angle {at['angle_deg']:.2f} deg")
        print(f"indexed {at['indexed']}/{at['total']} "
              f"({at['n_far']} outside the {at['tol_px']:.2f} px window, "
              f"{at['n_duplicate']} sharing a site), gradient over "
              f"{at['k_used']} neighbours")
        print(f"median residual {at['resid_px']:.3f} px")
        print(f"strain noise floor {at['strain_floor']*100:.2f} % "
              f"(residual over one lattice vector). Structure below this is "
              f"localization noise, not strain.")
        if at["indexed"] < 0.7 * at["total"]:
            print("  under 70 percent indexed: a wrong cell, more than one "
                  "grain, or detections that are not on the lattice")

        truth = compare_basis_to_truth(_stem, L["basis"]) if _stem else None
        if truth:
            print(f"\nground truth on the same frame: |a1| {truth['a1_true']:.3f} px, "
                  f"|a2| {truth['a2_true']:.3f} px, {truth['n_true']} columns, "
                  f"residual {truth['resid_true']:.3f} px")
            if "d_exx" in truth:
                print(f"  cell error vs truth: exx {truth['d_exx']*100:+.3f} %, "
                      f"eyy {truth['d_eyy']*100:+.3f} %, "
                      f"exy {truth['d_exy']*100:+.3f} %, "
                      f"rotation {truth['d_rot_deg']:+.3f} deg")

        print()
        for c in ["exx", "eyy", "exy"]:
            v = S[c].dropna()
            ratio = v.std() / at["strain_floor"] if at["strain_floor"] > 0 else np.nan
            print(f"  {c}: mean {v.mean()*100:+.3f} %, sd {v.std()*100:.3f} %, "
                  f"sd/floor {ratio:.2f}")

        fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
        vals = S[["exx", "eyy", "exy"]].values
        q = float(np.nanpercentile(np.abs(vals), 98)) if np.isfinite(vals).any() else 0.01
        q = max(q, 1e-4)
        for a, c in zip(ax[:3], ["exx", "eyy", "exy"]):
            sc = a.scatter(S.x, S.y, c=S[c], cmap="coolwarm", vmin=-q, vmax=q,
                           s=40, edgecolors="k", linewidths=0.2)
            a.set_title(f"{c}   scale +-{q*100:.2f} %\nfloor "
                        f"{at['strain_floor']*100:.2f} %", fontsize=8)
            a.invert_yaxis()
            plt.colorbar(sc, ax=a, fraction=0.046)
        d = L["disp"][L["keep"]]
        ideal_k = L["ideal"][L["keep"]]
        mag = np.hypot(*d.T)
        # autoscaled, because a fixed scale draws sub-pixel displacements as
        # arrows shorter than a pixel and the panel looks blank
        ax[3].quiver(ideal_k[:, 0], ideal_k[:, 1], d[:, 0], d[:, 1], mag,
                     cmap="viridis", angles="xy", scale_units="xy",
                     scale=max(mag.max(), 1e-6) / (0.4 * at["a1_px"]), width=0.006)
        ax[3].set_title(f"displacement x{0.4 * at['a1_px'] / max(mag.max(), 1e-6):.0f}"
                        f"\nmedian {np.median(mag):.3f} px, max {mag.max():.3f} px",
                        fontsize=8)
        ax[3].invert_yaxis()
        for a in ax:
            a.set_xticks([]); a.set_yticks([])
        plt.tight_layout()
        if callable(globals().get("save_fig")):
            save_fig(fig, "strain_map.png")
        else:
            plt.savefig(CFG.OUT_ROOT / "figures" / "strain_map.png", dpi=150)
        plt.show()

## 11. Reference-space robust strain refinement and validation

Run this after section 10. Three changes make the strain numbers defensible.

The fit moves into reference space. Fitting the gradient against measured
positions couples the strain estimate to the localization error twice, once
in the displacement and once in the coordinate. Reference coordinates are
exact by construction.

The fit becomes robust. A single mis-indexed column inside a neighbourhood
moves an ordinary least-squares gradient by a percent-level amount; iterated
re-weighted least squares with a Tukey biweight suppresses it.

The result gets a noise floor and a validation. The floor comes from a
synthetic perfect lattice carrying the measured localization jitter, which is
the smallest strain the method could report on a strain-free sample. The
validation is a split-half test (two disjoint halves of the columns should
produce the same map) and a bootstrap confidence interval on the mean strain.

In [ ]:
# ============================================================
# 11. Robust strain in reference space, plus validation
# ============================================================
def tukey_w(r, c=4.685):
    s = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-12
    u = r / (c * s)
    w = (1 - u ** 2) ** 2
    w[np.abs(u) >= 1] = 0.0
    return w


def robust_gradient(dref, dU, n_iter=5):
    """IRLS fit of dU = G dref + t, Tukey weights on the residual norm."""
    A = np.column_stack([dref, np.ones(len(dref))])
    w = np.ones(len(dref))
    sol = np.linalg.lstsq(A, dU, rcond=None)[0]
    for _ in range(n_iter):
        W = np.sqrt(w)[:, None]
        sol, *_ = np.linalg.lstsq(A * W, dU * W, rcond=None)
        r = np.hypot(*(dU - A @ sol).T)
        w = tukey_w(r)
        if w.sum() < 4:
            break
    return sol[:2].T


def remove_rigid_body(ref, U):
    """Strip translation and rotation, keeping every strain component.

    Fitting a full affine and subtracting only its antisymmetric part leaves
    the symmetric part in the displacement field, which is the uniform strain:
    the operation that is supposed to remove a rigid motion silently removes
    part of the signal instead. Only the two rigid parameters are fitted here.
    """
    ref = np.asarray(ref, float)
    U = np.asarray(U, float)
    c = ref.mean(0)
    d = ref - c
    # u = t + theta * (-y, x); least squares in (tx, ty, theta)
    A = np.zeros((2 * len(ref), 3))
    A[0::2, 0] = 1.0
    A[1::2, 1] = 1.0
    A[0::2, 2] = -d[:, 1]
    A[1::2, 2] = d[:, 0]
    sol, *_ = np.linalg.lstsq(A, U.ravel(), rcond=None)
    tx, ty, th = sol
    rigid = np.column_stack([tx - th * d[:, 1], ty + th * d[:, 0]])
    return U - rigid, float(th)


def strain_reference_space(L, k=8, robust=True, remove_rigid=True, min_pts=6):
    """Displacement gradient fitted against reference coordinates.

    Reference space, because fitting against measured positions couples the
    estimate to the localization error twice, once in the displacement and once
    in the coordinate.
    """
    keep = L["keep"]
    ref = L["ideal"][keep]
    U = L["disp"][keep].copy()
    if len(ref) < min_pts:
        out = pd.DataFrame(columns=["exx", "eyy", "exy", "rot", "x", "y"])
        out.attrs = dict(n=len(ref), reason=f"only {len(ref)} indexed columns, "
                                            f"need at least {min_pts}")
        return out
    rot_removed = 0.0
    if remove_rigid:
        U, rot_removed = remove_rigid_body(ref, U)
    k = min(k, len(ref) - 1)
    _, idx = cKDTree(ref).query(ref, k=k + 1)
    out = np.full((len(ref), 4), np.nan)
    for i in range(len(ref)):
        j = idx[i, 1:]
        dref, dU = ref[j] - ref[i], U[j] - U[i]
        G = (robust_gradient(dref, dU) if robust else
             np.linalg.lstsq(np.column_stack([dref, np.ones(len(j))]),
                             dU, rcond=None)[0][:2].T)
        out[i] = [G[0, 0], G[1, 1], 0.5 * (G[0, 1] + G[1, 0]),
                  0.5 * (G[0, 1] - G[1, 0])]
    df = pd.DataFrame(out, columns=["exx", "eyy", "exy", "rot"])
    df["x"], df["y"] = ref[:, 0], ref[:, 1]
    df.attrs = dict(n=len(ref), k_used=k, rot_removed_rad=rot_removed, reason="")
    return df


def align_basis(basis_fit, basis_ref, rng_int=2):
    """Best integer unimodular U with U @ basis_ref closest to basis_fit.

    A lattice basis is only defined up to such a U, so a raw comparison of two
    bases can report a 150 percent strain for the same lattice written down
    differently.
    """
    Bf = np.asarray(basis_fit, float)
    Br = np.asarray(basis_ref, float)
    best, bestU = None, np.eye(2, dtype=int)
    rr = range(-rng_int, rng_int + 1)
    for a, b, c, d in itertools.product(rr, rr, rr, rr):
        U = np.array([[a, b], [c, d]])
        if abs(round(np.linalg.det(U))) != 1:
            continue
        err = np.linalg.norm(U @ Br - Bf)
        if best is None or err < best:
            best, bestU = err, U
    return bestU, float(best)


def basis_strain(basis_fit, basis_ref):
    """Homogeneous strain between two bases, which the local map cannot see."""
    U, resid = align_basis(basis_fit, basis_ref)
    Bf = np.asarray(basis_fit, float).T
    Br = (U @ np.asarray(basis_ref, float)).T
    Fm = Bf @ np.linalg.inv(Br)
    E = 0.5 * (Fm + Fm.T) - np.eye(2)
    return dict(exx=float(E[0, 0]), eyy=float(E[1, 1]), exy=float(E[0, 1]),
                rot=float(0.5 * (Fm[0, 1] - Fm[1, 0])), U=U,
                align_resid_px=resid, F=Fm)


def strain_noise_floor(L, jitter_px=None, k=8, n_rep=8, seed=0):
    """Strain a perfect lattice shows when it carries only localization noise.

    The jitter is the per-axis robust spread of the residuals, matching the
    per-axis Gaussian the simulation then draws. Pooling both components into
    one np.std measures scatter about a common mean and builds the floor from
    a different noise model than the one observed.
    """
    keep = L["keep"]
    ref = L["ideal"][keep]
    d = L["disp"][keep]
    if len(ref) < 6:
        return dict(jitter_px=np.nan, exx=np.nan, eyy=np.nan, exy=np.nan)
    if jitter_px is None:
        per_axis = [1.4826 * np.median(np.abs(d[:, a] - np.median(d[:, a])))
                    for a in (0, 1)]
        jitter_px = float(np.mean(per_axis))
    rng = np.random.default_rng(seed)
    sds = []
    for _ in range(n_rep):
        fake = dict(ideal=ref, disp=rng.normal(0, jitter_px, ref.shape),
                    keep=np.ones(len(ref), bool))
        s = strain_reference_space(fake, k=k, remove_rigid=True)
        if len(s):
            sds.append([s.exx.std(), s.eyy.std(), s.exy.std()])
    if not sds:
        return dict(jitter_px=jitter_px, exx=np.nan, eyy=np.nan, exy=np.nan)
    sds = np.array(sds)
    return dict(jitter_px=jitter_px, exx=float(sds[:, 0].mean()),
                eyy=float(sds[:, 1].mean()), exy=float(sds[:, 2].mean()))


def split_half_validation(L, k=8, n_rep=5, seed=0):
    """Do two disjoint halves of the columns report the same strain field?

    Both halves are evaluated at the SAME sites: each half fits the gradient
    using only its own columns as neighbours, then both are read out at every
    site. Matching half A to half B by nearest neighbour instead pairs each
    site with a different site, so the correlation measures how smooth the
    field is, not whether the two halves agree.
    """
    idx_keep = np.where(L["keep"])[0]
    if len(idx_keep) < 4 * k:
        return {c: np.nan for c in ("exx", "eyy", "exy")}
    ref_all = L["ideal"][L["keep"]]
    rng = np.random.default_rng(seed)
    acc = {c: [] for c in ("exx", "eyy", "exy")}

    for rep in range(n_rep):
        perm = rng.permutation(len(idx_keep))
        halves = []
        for part in (perm[::2], perm[1::2]):
            sel = np.zeros(len(L["keep"]), bool)
            sel[idx_keep[part]] = True
            sub = dict(L, keep=sel)
            ref = L["ideal"][sel]
            U = L["disp"][sel]
            if len(ref) < k + 2:
                halves = []
                break
            U, _ = remove_rigid_body(ref, U)
            kk = min(k, len(ref) - 1)
            _, nb = cKDTree(ref).query(ref_all, k=kk)      # read out everywhere
            vals = np.full((len(ref_all), 3), np.nan)
            for i in range(len(ref_all)):
                j = np.atleast_1d(nb[i])
                dref, dU = ref[j] - ref_all[i], U[j]
                if len(j) < 4:
                    continue
                G = robust_gradient(dref, dU - dU.mean(0))
                vals[i] = [G[0, 0], G[1, 1], 0.5 * (G[0, 1] + G[1, 0])]
            halves.append(vals)
        if len(halves) != 2:
            continue
        a, b = halves
        for ci, c in enumerate(("exx", "eyy", "exy")):
            ok = np.isfinite(a[:, ci]) & np.isfinite(b[:, ci])
            if ok.sum() > 8 and a[ok, ci].std() > 0 and b[ok, ci].std() > 0:
                acc[c].append(float(np.corrcoef(a[ok, ci], b[ok, ci])[0, 1]))
    return {c: (float(np.mean(v)) if v else np.nan) for c, v in acc.items()}


def bootstrap_mean(df, col, n=500, seed=0):
    v = df[col].dropna().values
    if len(v) < 10:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
    return tuple(np.percentile(m, [2.5, 97.5]))


# ---- demonstration ----
SR = None
REFERENCE_BASIS = None      # e.g. np.array([[12.0, 0.0], [6.0, 10.392]])

if not len(MASTER):
    print("MASTER is empty; run section 5c first.")
elif L is None:
    print("section 10 produced no lattice; nothing to refine.")
elif int(L["keep"].sum()) < 20:
    print(f"only {int(L['keep'].sum())} indexed columns; a strain field needs "
          f"more. This is the section 8 or 10 result, not a problem here.")
else:
    SR = strain_reference_space(L, k=8, robust=True)
    if not len(SR):
        print(f"  {SR.attrs.get('reason', 'no rows')}")
        SR = None

if SR is not None:
    floor = strain_noise_floor(L)
    corr = split_half_validation(L)
    print(f"robust strain in reference space, {SR.attrs['n']} columns, "
          f"{SR.attrs['k_used']} neighbours per fit")
    print(f"rigid rotation removed: "
          f"{math.degrees(SR.attrs['rot_removed_rad']):+.4f} deg\n")
    for c in ["exx", "eyy", "exy"]:
        lo, hi = bootstrap_mean(SR, c)
        sd = float(SR[c].std())
        snr = sd / floor[c] if floor.get(c, 0) > 0 else np.nan
        print(f"  {c}: mean {SR[c].mean()*100:+.4f} % "
              f"(95% CI {lo*100:+.4f} to {hi*100:+.4f}), "
              f"sd {sd*100:.4f} %, floor {floor[c]*100:.4f} %, "
              f"sd/floor {snr:.2f}, split-half r {corr[c]:+.3f}")
    print(f"\n  localization jitter for the floor: {floor['jitter_px']:.3f} px")
    print("  sd/floor near 1 with split-half r near 0 means this frame has no "
          "strain the method can see, which is the expected answer for an "
          "undistorted simulated lattice.")

    if REFERENCE_BASIS is not None:
        bs = basis_strain(L["basis"], REFERENCE_BASIS)
        print(f"\n  uniform strain against the reference basis: "
              f"exx {bs['exx']*100:+.3f} %, eyy {bs['eyy']*100:+.3f} %, "
              f"exy {bs['exy']*100:+.3f} %, "
              f"rotation {math.degrees(bs['rot']):+.3f} deg")
    else:
        print("\n  the basis was refitted to this frame, so a uniform strain is "
              "absorbed into the reference and the map shows only how strain "
              "varies across the frame. Set REFERENCE_BASIS for the absolute "
              "value.")

    fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.8))
    vals = SR[["exx", "eyy", "exy"]].values
    q = float(np.nanpercentile(np.abs(vals), 98)) if np.isfinite(vals).any() else 0.01
    q = max(q, 1e-4)
    for a, c in zip(ax, ["exx", "eyy", "exy"]):
        sc = a.scatter(SR.x, SR.y, c=SR[c], cmap="coolwarm", vmin=-q, vmax=q,
                       s=45, edgecolors="k", linewidths=0.2)
        a.set_title(f"{c}   scale +-{q*100:.3f} %\nfloor "
                    f"{floor[c]*100:.3f} %, r {corr[c]:+.2f}", fontsize=8)
        a.invert_yaxis(); a.set_xticks([]); a.set_yticks([])
        plt.colorbar(sc, ax=a, fraction=0.046)
    plt.tight_layout()
    if callable(globals().get("save_fig")):
        save_fig(fig, "strain_robust.png")
    else:
        plt.savefig(CFG.OUT_ROOT / "figures" / "strain_robust.png", dpi=150)
    plt.show()
    if callable(globals().get("save_table")):
        save_table(SR, "strain_reference_space.csv")
    else:
        SR.to_csv(CFG.OUT_ROOT / "tables" / "strain_reference_space.csv", index=False)

## 12. Topological defects and dislocation core mapping

Two independent detectors, because each one alone produces false positives at
the frame edge and at missed columns.

The Burgers circuit uses the integer indices from section 10. Around every
Delaunay triangle the index differences are summed; a closed lattice gives
zero, and a non-zero sum is a Burgers vector in lattice units. This is exact
where the indexing is correct and silent where it is not.

The coordination detector uses the Delaunay neighbour count with the convex
hull removed. In a hexagonal lattice a 5-fold column next to a 7-fold column
is the classic dislocation core signature; isolated 5s and 7s are usually
missed or split columns instead.

A core is reported only where both detectors fire within one lattice spacing.

In [ ]:
# ============================================================
# 12. Defects and dislocation cores
# ============================================================
def delaunay_neighbours(xy):
    tri = Delaunay(xy)
    nb = defaultdict(set)
    for s in tri.simplices:
        for a, b in itertools.permutations(s, 2):
            nb[a].add(b)
    return tri, {k: sorted(v) for k, v in nb.items()}


def hull_mask(xy, pad=1):
    tri = Delaunay(xy)
    hull = set(np.unique(tri.convex_hull))
    _, nb = delaunay_neighbours(xy)
    border = set(hull)
    for _ in range(pad):
        border |= {j for i in list(border) for j in nb.get(i, [])}
    m = np.zeros(len(xy), bool)
    m[list(border)] = True
    return m


def burgers_field(L):
    """Sum the index difference around every Delaunay triangle."""
    keep = L["keep"]
    xy = (L["ideal"] + L["disp"])[keep]
    ij = L["ij"][keep]
    tri = Delaunay(xy)
    rows = []
    for s in tri.simplices:
        a, b, c = s
        loop = (ij[b] - ij[a]) + (ij[c] - ij[b]) + (ij[a] - ij[c])
        if np.any(loop != 0):                     # closure failure, in lattice units
            cen = xy[s].mean(0)
            bvec = loop @ L["basis"]
            rows.append(dict(x=cen[0], y=cen[1], m=int(loop[0]), n=int(loop[1]),
                             bx=bvec[0], by=bvec[1],
                             b_px=float(np.hypot(*bvec))))
    return pd.DataFrame(rows), tri, xy


def coordination_defects(xy, expected=6):
    tri, nb = delaunay_neighbours(xy)
    coord = np.array([len(nb.get(i, [])) for i in range(len(xy))])
    edge = hull_mask(xy)
    df = pd.DataFrame(dict(x=xy[:, 0], y=xy[:, 1], coord=coord, edge=edge))
    df["defect"] = (~df.edge) & (df.coord != expected)
    # pair 5s with neighbouring 7s
    five = df.index[(df.coord == expected - 1) & ~df.edge]
    seven = df.index[(df.coord == expected + 1) & ~df.edge]
    pairs = []
    if len(five) and len(seven):
        t = cKDTree(xy[seven])
        d, j = t.query(xy[five], k=1)
        for i, (dd, jj) in enumerate(zip(d, j)):
            pairs.append(dict(x=0.5 * (xy[five[i], 0] + xy[seven[jj], 0]),
                              y=0.5 * (xy[five[i], 1] + xy[seven[jj], 1]),
                              sep_px=float(dd)))
    return df, pd.DataFrame(pairs)


def dislocation_cores(L, expected_coord=6):
    B, tri, xy = burgers_field(L)
    C, pairs = coordination_defects(xy, expected_coord)
    a = float(np.linalg.norm(L["basis"], axis=1).mean())
    cores = pd.DataFrame()
    if len(B) and len(pairs):
        t = cKDTree(pairs[["x", "y"]].values)
        d, _ = t.query(B[["x", "y"]].values, k=1)
        cores = B[d < a].copy()
        cores["confirmed_by"] = "burgers+coordination"
    elif len(B):
        cores = B.copy()
        cores["confirmed_by"] = "burgers only"
    return dict(burgers=B, coordination=C, pairs=pairs, cores=cores,
                lattice_px=a, xy=xy)


if len(MASTER):
    D = dislocation_cores(L)
    print(f"Delaunay triangles with a non-zero Burgers circuit: {len(D['burgers'])}")
    print(f"5-7 coordination pairs away from the edge: {len(D['pairs'])}")
    print(f"cores confirmed by both detectors: {len(D['cores'])}")
    if len(D["cores"]):
        bb = D["cores"].b_px * CFG.PIXEL_SIZE_A
        print(f"  |b| {bb.mean():.3f} +- {bb.std():.3f} A "
              f"({(D['cores'].b_px / D['lattice_px']).mean():.2f} lattice units)")
    cnt = Counter(D["coordination"].loc[~D["coordination"].edge, "coord"])
    print(f"  coordination histogram (interior): {dict(sorted(cnt.items()))}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
    ax[0].imshow(R["image"], cmap="gray")
    cdf = D["coordination"]
    sc = ax[0].scatter(cdf.x, cdf.y, c=cdf.coord, cmap="coolwarm", vmin=4, vmax=8, s=12)
    plt.colorbar(sc, ax=ax[0], fraction=0.046, label="Delaunay coordination")
    ax[0].set_title("coordination")
    ax[1].imshow(R["image"], cmap="gray")
    if len(D["burgers"]):
        ax[1].scatter(D["burgers"].x, D["burgers"].y, s=30, facecolors="none",
                      edgecolors="yellow", lw=0.8, label="Burgers closure failure")
    if len(D["cores"]):
        ax[1].scatter(D["cores"].x, D["cores"].y, s=90, marker="o", facecolors="none",
                      edgecolors="red", lw=1.4, label="confirmed core")
        ax[1].quiver(D["cores"].x, D["cores"].y, D["cores"].bx, D["cores"].by,
                     color="red", scale=60)
    ax[1].legend(fontsize=7, loc="upper right")
    ax[1].set_title("dislocation cores")
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    plt.savefig(CFG.OUT_ROOT / "figures" / "defects.png", dpi=150)
    plt.show()

## 13. Radial distribution function and short-range order

g(r) from the detected columns with a Ripley edge correction, since an
uncorrected g(r) on a 256 px field decays to zero from about r = 40 px purely
because of the missing area outside the frame. The first peak position and
width give the spacing and its spread, and the peak area gives the
coordination number.

Bond-orientational order psi_6 per column separates a well-ordered region
from a disordered one without needing any labels. Where the column
amplitudes split into two populations, a Warren-Cowley parameter for the
first shell says whether the two types prefer like or unlike neighbours;
alpha near 0 is random mixing, negative is ordering, positive is clustering.

In [ ]:
# ============================================================
# 13. RDF, bond-orientational order, short-range order
# ============================================================
from sklearn.cluster import KMeans


def rdf(xy, shape=None, r_max=None, dr=0.5, edge_correct=True):
    """g(r) over the region the columns actually occupy.

    The domain is the bounding box of the detected columns, not the frame:
    detection excludes a border, so using the frame area underestimates the
    density and makes g(r) climb with r instead of settling at 1.
    """
    x0, y0 = xy.min(0)
    x1, y1 = xy.max(0)
    W, H = max(x1 - x0, 1.0), max(y1 - y0, 1.0)
    r_max = r_max or min(W, H) / 3.0
    n = len(xy)
    rho = n / (W * H)
    tree = cKDTree(xy)
    edges = np.arange(0, r_max + dr, dr)
    rc = 0.5 * (edges[:-1] + edges[1:])
    th = np.linspace(0, 2 * np.pi, 72, endpoint=False)
    cos_t, sin_t = np.cos(th), np.sin(th)
    counts = np.zeros(len(rc))
    weight = np.zeros(len(rc))
    for p in xy:
        j = tree.query_ball_point(p, r_max)
        d = np.hypot(*(xy[j] - p).T)
        d = d[d > 1e-9]
        hist, _ = np.histogram(d, bins=edges)
        if edge_correct:
            px = p[0] + rc[:, None] * cos_t[None, :]
            py = p[1] + rc[:, None] * sin_t[None, :]
            frac = np.mean((px >= x0) & (px <= x1) & (py >= y0) & (py <= y1), axis=1)
        else:
            frac = np.ones(len(rc))
        counts += hist
        weight += np.maximum(frac, 0.05)
    ideal = 2 * np.pi * rc * dr * rho * weight
    return rc, counts / np.maximum(ideal, 1e-9)


def first_peak(rc, g, min_g=1.5, smooth=1.0, r_min=1.5):
    """Index of the FIRST significant maximum of g(r).

    Not the global maximum: on an ordered lattice a distant shell can hold more
    pairs than the nearest-neighbour shell and the edge correction amplifies it,
    so argmax lands several shells out.
    """
    gs = gaussian_filter(np.asarray(g, float), smooth)
    for i in range(1, len(gs) - 1):
        if rc[i] >= r_min and gs[i] >= min_g and gs[i] >= gs[i - 1] and gs[i] >= gs[i + 1]:
            return i
    return int(np.argmax(gs))


def psi_n(xy, n=6, k=6):
    tree = cKDTree(xy)
    kk = min(k + 1, len(xy))
    _, idx = tree.query(xy, k=kk)
    out = np.zeros(len(xy), complex)
    for i in range(len(xy)):
        v = xy[idx[i, 1:]] - xy[i]
        th = np.arctan2(v[:, 1], v[:, 0])
        out[i] = np.mean(np.exp(1j * n * th))
    return np.abs(out)


def warren_cowley(xy, labels, k=6):
    """alpha_AB for the first shell: 1 - P(B|A) / c_B."""
    tree = cKDTree(xy)
    kk = min(k + 1, len(xy))
    _, idx = tree.query(xy, k=kk)
    types = np.unique(labels)
    c = {t: float(np.mean(labels == t)) for t in types}
    out = {}
    for a in types:
        sel = np.where(labels == a)[0]
        for b in types:
            p = np.mean([np.mean(labels[idx[i, 1:]] == b) for i in sel])
            out[(int(a), int(b))] = float(1 - p / max(c[b], 1e-9))
    return out, c


if len(MASTER):
    pk = R["peaks"]
    xy = pk[["x", "y"]].values
    rc, g = rdf(xy, R["prob"].shape)
    p6 = psi_n(xy, 6)
    p4 = psi_n(xy, 4)

    i1 = first_peak(rc, g)
    r1 = rc[i1]
    half = g[i1] / 2
    lo = rc[:i1][g[:i1] < half][-1] if np.any(g[:i1] < half) else rc[0]
    hi_idx = np.where(g[i1:] < half)[0]
    hi = rc[i1 + hi_idx[0]] if len(hi_idx) else rc[-1]
    # coordination number from the area under the first peak, same density
    # convention as rdf() so the two cannot drift apart
    m = (rc >= lo) & (rc <= hi)
    span = xy.max(0) - xy.min(0)
    rho = len(xy) / max(span[0] * span[1], 1.0)
    cn = float(np.sum(2 * np.pi * rc[m] * g[m] * rho * (rc[1] - rc[0])))
    print(f"g(r) first peak at {r1:.2f} px = {r1 * CFG.PIXEL_SIZE_A:.4f} A, "
          f"FWHM {hi - lo:.2f} px, coordination {cn:.2f}")
    print(f"psi_6 median {np.median(p6):.3f}   psi_4 median {np.median(p4):.3f}  "
          f"-> {'hexagonal' if np.median(p6) > np.median(p4) else 'square-like'} order")

    amps = pk["amp"].values.reshape(-1, 1)
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(amps)
    lab = km.labels_
    sep = abs(np.diff(km.cluster_centers_.ravel())[0]) / (amps.std() + 1e-9)
    if sep > 1.0:
        alpha, conc = warren_cowley(xy, lab)
        print(f"two amplitude populations, separation {sep:.2f} sd, "
              f"fractions {[round(v, 3) for v in conc.values()]}")
        for (a, b), v in alpha.items():
            if a != b:
                print(f"  Warren-Cowley alpha({a},{b}) = {v:+.3f}  "
                      f"({'ordering' if v < -0.05 else 'clustering' if v > 0.05 else 'random'})")
    else:
        print(f"column amplitudes are single-population (separation {sep:.2f} sd), "
              f"no short-range order parameter computed")

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    ax[0].plot(rc * CFG.PIXEL_SIZE_A, g)
    ax[0].axhline(1, color="k", lw=0.6, ls=":")
    ax[0].set_xlabel("r (A)"); ax[0].set_ylabel("g(r)"); ax[0].set_title("radial distribution")
    s = ax[1].scatter(xy[:, 0], xy[:, 1], c=p6, cmap="viridis", vmin=0, vmax=1, s=14)
    ax[1].invert_yaxis(); ax[1].set_title("psi_6"); plt.colorbar(s, ax=ax[1], fraction=0.046)
    ax[2].hist(pk["amp"].values, bins=40, color="0.5")
    ax[2].set_title("column amplitude"); ax[2].set_xlabel("fitted amplitude")
    for a in ax[1:2]:
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    plt.savefig(CFG.OUT_ROOT / "figures" / "rdf_sro.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# 13b. Lattice statistics over every dense frame
# ============================================================
PHYS_MIN_COLS = 40
PHYS_MAX_FRAMES = 60


def physics_one(ckpt, arch, stem):
    R = localize_from_model(ckpt, arch, stem)
    pk = R["peaks"]
    if len(pk) < PHYS_MIN_COLS:
        return None
    out = dict(stem=stem, n_columns=len(pk))
    sp = first_shell_spacing(pk[["x", "y"]].values)
    if sp["ok"]:
        out.update(d_px=sp["d_px"], d_A=sp["d_A"], d_mad=sp["mad_px"])
    try:
        S, L = strain_map(pk)
        out.update(a1_px=S.attrs["a1_px"], a2_px=S.attrs["a2_px"],
                   angle_deg=S.attrs["angle_deg"],
                   indexed_frac=S.attrs["indexed"] / max(S.attrs["total"], 1),
                   resid_px=S.attrs["resid_px"],
                   exx_sd=float(S.exx.std()), eyy_sd=float(S.eyy.std()),
                   exy_sd=float(S.exy.std()))
        if stem in gauss_gt_map:
            t = compare_basis_to_truth(stem, L["basis"])
            if t and "d_exx" in t:
                out.update(err_exx=t["d_exx"], err_eyy=t["d_eyy"],
                           err_exy=t["d_exy"])
    except RuntimeError as e:
        out["lattice_error"] = str(e)[:60]
    return out


dense = [s for s, n in counts.items() if n >= PHYS_MIN_COLS][:PHYS_MAX_FRAMES]
print(f"{len(dense)} frames with at least {PHYS_MIN_COLS} columns "
      f"({len(dense) / len(counts):.1%} of the surveyed fold)")

PHYS = []
for cond in BEST_CONDITIONS:
    for arch in BEST_ARCHS:
        f, ck = best_fold(cond, arch)
        if f is None:
            continue
        t0 = time.time()
        rows = []
        for s in dense:
            try:
                r = physics_one(ck, arch, s)
            except Exception as e:
                continue
            if r:
                r.update(condition=cond, arch=arch, fold=f)
                rows.append(r)
        PHYS += rows
        if rows:
            d = pd.DataFrame(rows)
            print(f"{cond:9s} {arch:11s} {len(rows)} frames  "
                  f"spacing {d.d_A.mean():.4f} +- {d.d_A.std():.4f} A  "
                  f"indexed {d.indexed_frac.mean():.0%}  "
                  f"residual {d.resid_px.mean():.3f} px  "
                  f"({time.time() - t0:.0f} s)")

PHYS = pd.DataFrame(PHYS)
if len(PHYS):
    PHYS.to_csv(CFG.OUT_ROOT / "tables" / "physics_all_frames.csv", index=False)
    print("\nper-model summary over all dense frames")
    print(PHYS.groupby(["condition", "arch"]).agg(
        frames=("stem", "count"), d_A=("d_A", "mean"), d_sd=("d_A", "std"),
        indexed=("indexed_frac", "mean"), resid_px=("resid_px", "mean"),
        strain_noise=("exx_sd", "mean")).round(4).to_string())
    if "err_exx" in PHYS:
        print("\ncell error against ground truth, the systematic floor")
        print(PHYS.groupby(["condition", "arch"])[["err_exx", "err_eyy", "err_exy"]]
              .mean().mul(100).round(3).to_string())

## 14. Centroid metrics, batch driver

Pixel IoU rewards a fat mask. What the downstream physics needs is whether
each column was found and where. This driver runs every checkpoint over a
fixed set of held-out frames, takes ground-truth centres from `gaussianMask`,
matches them to the detected centres with the Hungarian-free greedy KD-tree
rule at `CFG.MATCH_RADIUS_PX`, and reports detection precision, recall, F1,
the RMSE of the matched positions, and the systematic bias.

In [ ]:
# ============================================================
# 14. Centroid metrics over all models, folds and conditions
# ============================================================
CENTROID_CSV = CFG.OUT_ROOT / "tables" / "centroid_metrics.csv"
CENTROID_N_IMAGES = 40
CENTROID_MIN_GT   = 20      # frames below this are scored separately, not mixed in
RADIUS_SWEEP      = (1.0, 1.5, 2.0, 3.0)
FORCE_CENTROID    = True


def gt_peaks_from_gaussian(stem, min_distance=None):
    g = norm01(load_gray(gauss_gt_map[stem]))
    md = min_distance or estimate_min_distance(g)
    c = peak_local_max(gaussian_filter(g, 0.8),
                       min_distance=int(max(2, round(LOC_CFG.PEAK_DIST_FRAC * md))),
                       threshold_abs=0.25, exclude_border=LOC_CFG.BORDER_PX)
    out = np.stack([c[:, 1], c[:, 0]], axis=1).astype(float)
    ref = [refine_peak(g, x, y) for x, y in out]
    return np.array([[r["x"], r["y"]] for r in ref]) if ref else out


def match_points(pred, gt, radius):
    """Greedy nearest-neighbour matching, each GT used at most once."""
    if len(pred) == 0 or len(gt) == 0:
        return np.empty((0, 2), int), np.array([])
    tree = cKDTree(gt)
    d, j = tree.query(pred, k=1, distance_upper_bound=radius)
    order = np.argsort(d)
    used, pairs, dist = set(), [], []
    for i in order:
        if not np.isfinite(d[i]) or j[i] in used:
            continue
        used.add(j[i]); pairs.append((i, j[i])); dist.append(d[i])
    return np.array(pairs, int).reshape(-1, 2), np.array(dist)


def detection_counts(pred, gt, radius):
    """Raw counts and matched errors, so several frames can be pooled."""
    pairs, dist = match_points(pred, gt, radius)
    tp = len(pairs)
    err = (pred[pairs[:, 0]] - gt[pairs[:, 1]]) if tp else np.empty((0, 2))
    return dict(tp=tp, fp=len(pred) - tp, fn=len(gt) - tp,
                n_pred=len(pred), n_gt=len(gt), err=err)


def counts_to_metrics(tp, fp, fn, err=None, prefix=""):
    prec = tp / max(tp + fp, 1e-9)
    rec = tp / max(tp + fn, 1e-9)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    out = {prefix + "precision": prec, prefix + "recall": rec, prefix + "f1": f1}
    if err is not None and len(err):
        rmse = float(np.sqrt((err ** 2).sum(1).mean()))
        out[prefix + "rmse_px"] = rmse
        out[prefix + "rmse_pm"] = rmse * CFG.PIXEL_SIZE_A * 100
        out[prefix + "bias_x"] = float(err[:, 0].mean())
        out[prefix + "bias_y"] = float(err[:, 1].mean())
        out[prefix + "mae_px"] = float(np.abs(err).mean())
    else:
        for k in ("rmse_px", "rmse_pm", "bias_x", "bias_y", "mae_px"):
            out[prefix + k] = np.nan
    return out


def detection_metrics(pred, gt, radius=None):
    """Single-frame convenience wrapper, kept for the other sections."""
    radius = CFG.MATCH_RADIUS_PX if radius is None else radius
    c = detection_counts(pred, gt, radius)
    out = counts_to_metrics(c["tp"], c["fp"], c["fn"], c["err"])
    out.update(n_pred=c["n_pred"], n_gt=c["n_gt"], tp=c["tp"])
    return out


def reference_ceiling(fold, n_images=CENTROID_N_IMAGES, radius=None):
    """circularMask against gaussianMask, no model involved.

    Both folders describe the same atoms, so this is as well as any model
    could possibly score. Whatever it falls short of 1.0 is disagreement
    between the two ground-truth encodings, and it sits inside every model
    score in the table below.
    """
    radius = CFG.MATCH_RADIUS_PX if radius is None else radius
    _, va = fold_indices(fold)
    stems = [common[i] for i in va
             if common[i] in gauss_gt_map and common[i] in mask_map][:n_images]
    tp = fp = fn = 0
    errs = []
    for s in stems:
        m = to_binary(load_gray(mask_map[s])).astype(float)
        pk, md = locate_columns(m, m)
        gt = gt_peaks_from_gaussian(s, md)
        if len(gt) < CENTROID_MIN_GT:
            continue
        c = detection_counts(pk[["x", "y"]].values, gt, radius)
        tp += c["tp"]; fp += c["fp"]; fn += c["fn"]
        if len(c["err"]):
            errs.append(c["err"])
    err = np.vstack(errs) if errs else None
    out = counts_to_metrics(tp, fp, fn, err)
    out.update(fold=int(fold), n_images=len(stems))
    return out


@torch.no_grad()
def centroid_metrics_for(ckpt, arch, fold, n_images=CENTROID_N_IMAGES,
                         radius=None, sweep=RADIUS_SWEEP):
    """Pooled detection metrics over one fold's held-out frames.

    Pooled, not averaged per frame: a frame with four columns would otherwise
    carry the same weight as one with four hundred, and TEM-ImageNet mixes
    magnifications heavily enough for that to dominate the fold mean.
    """
    radius = CFG.MATCH_RADIUS_PX if radius is None else radius
    _, va = fold_indices(fold)
    stems = [common[i] for i in va if common[i] in gauss_gt_map][:n_images]
    if not stems:
        return None
    model = model_from_checkpoint(ckpt, arch)

    pooled = {r: dict(tp=0, fp=0, fn=0, errs=[]) for r in set(sweep) | {radius}}
    per_frame_f1, cov_pred, cov_true = [], [], []
    n_pred_tot = n_gt_tot = 0
    n_used = n_sparse = 0

    for s in stems:
        img = norm01(load_gray(noisy_map[s]))
        den, seg = split_heads(model(torch.from_numpy(img)[None, None].to(CFG.DEVICE)))
        prob = seg_prob(seg)[0, 0].float().cpu().numpy()
        src = norm01(den[0, 0].float().cpu().numpy()) if den is not None else img
        pk, md = locate_columns(prob, src)
        gt = gt_peaks_from_gaussian(s, md)
        if len(gt) < CENTROID_MIN_GT:
            n_sparse += 1
            continue
        n_used += 1
        n_pred_tot += len(pk); n_gt_tot += len(gt)
        cov_pred.append(float((prob > CFG.THRESHOLD).mean()))
        if s in mask_map:
            cov_true.append(float(to_binary(load_gray(mask_map[s])).mean()))
        for r in pooled:
            c = detection_counts(pk[["x", "y"]].values, gt, r)
            pooled[r]["tp"] += c["tp"]; pooled[r]["fp"] += c["fp"]
            pooled[r]["fn"] += c["fn"]
            if len(c["err"]):
                pooled[r]["errs"].append(c["err"])
            if r == radius:
                per_frame_f1.append(counts_to_metrics(c["tp"], c["fp"], c["fn"])["f1"])

    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if n_used == 0:
        return None

    p = pooled[radius]
    out = counts_to_metrics(p["tp"], p["fp"], p["fn"],
                            np.vstack(p["errs"]) if p["errs"] else None)
    out.update(arch=arch, fold=int(fold), radius=radius,
               n_images=n_used, n_sparse=n_sparse,
               n_pred=n_pred_tot, n_gt=n_gt_tot,
               count_ratio=n_pred_tot / max(n_gt_tot, 1),
               f1_macro=float(np.mean(per_frame_f1)),
               f1_frame_sd=float(np.std(per_frame_f1)),
               coverage=float(np.mean(cov_pred)) if cov_pred else np.nan,
               coverage_true=float(np.mean(cov_true)) if cov_true else np.nan)
    out["blob_ratio"] = out["coverage"] / max(out["coverage_true"], 1e-9)
    for r in sorted(sweep):
        q = pooled[r]
        out[f"f1_r{r:g}"] = counts_to_metrics(q["tp"], q["fp"], q["fn"])["f1"]
        if q["errs"]:
            e = np.vstack(q["errs"])
            out[f"rmse_r{r:g}"] = float(np.sqrt((e ** 2).sum(1).mean()))
    return out


# ---- run ----
if CENTROID_CSV.exists() and not FORCE_CENTROID:
    CENTROID = pd.read_csv(CENTROID_CSV)
else:
    CENTROID = pd.DataFrame()

done = set(zip(CENTROID.condition, CENTROID.arch, CENTROID.fold)) if len(CENTROID) else set()
rows = CENTROID.to_dict("records") if len(CENTROID) else []

if not gauss_gt_map:
    print(f"no {CFG.GT_POS} folder, centroid metrics skipped")
else:
    for _, r in todo.iterrows():
        if (r.condition, r.arch, int(r.fold)) in done:
            continue
        t0 = time.time()
        try:
            m = centroid_metrics_for(r.path, r.arch, int(r.fold))
            if m is None:
                print(f"{r.condition:9s} {r.arch:11s} f{int(r.fold)}  "
                      f"no frame with >= {CENTROID_MIN_GT} columns")
                continue
            m["condition"] = r.condition
            rows.append(m)
            print(f"{r.condition:9s} {r.arch:11s} f{int(r.fold)}  "
                  f"P {m['precision']:.3f}  R {m['recall']:.3f}  F1 {m['f1']:.3f}  "
                  f"RMSE {m['rmse_px']:.3f} px = {m['rmse_pm']:.1f} pm  "
                  f"pred/true {m['count_ratio']:.2f}  "
                  f"({m['n_images']} frames, {m['n_sparse']} sparse skipped, "
                  f"{time.time() - t0:.0f} s)")
        except Exception as e:
            print(f"{r.condition:9s} {r.arch:11s} f{int(r.fold)}  FAILED: "
                  f"{type(e).__name__}: {e}")

    CENTROID = pd.DataFrame(rows)
    if len(CENTROID):
        CENTROID.to_csv(CENTROID_CSV, index=False)

        agg = CENTROID.groupby(["condition", "arch"]).agg(
            F1=("f1", "mean"), F1_sd=("f1", "std"),
            P=("precision", "mean"), R=("recall", "mean"),
            rmse_pm=("rmse_pm", "mean"),
            bias_x=("bias_x", "mean"), bias_y=("bias_y", "mean"),
            count_ratio=("count_ratio", "mean"),
            blob=("blob_ratio", "mean"), folds=("fold", "nunique"))
        print("\ncentroid summary, pooled over frames within each fold")
        print(agg.sort_values("F1", ascending=False).round(4).to_string())

        # ---- what any model could possibly score ----
        print("\nreference ceiling: circularMask vs gaussianMask, no model")
        ceils = []
        for f in sorted(CENTROID.fold.unique()):
            try:
                c = reference_ceiling(int(f))
                ceils.append(c)
                print(f"  fold {int(f)}: F1 {c['f1']:.3f}  "
                      f"RMSE {c['rmse_px']:.3f} px = {c['rmse_pm']:.1f} pm  "
                      f"bias ({c['bias_x']:+.3f}, {c['bias_y']:+.3f})  "
                      f"{c['n_images']} frames")
            except Exception as e:
                print(f"  fold {int(f)}: {type(e).__name__}: {e}")
        if ceils:
            cf = pd.DataFrame(ceils)
            print(f"  mean ceiling F1 {cf.f1.mean():.3f}, "
                  f"RMSE {cf.rmse_pm.mean():.1f} pm")
            if cf.f1.mean() < 0.95 or cf.rmse_px.mean() > 0.75:
                print("  the two ground-truth folders disagree with each other. "
                      "Subtract this from every model number above; the gap "
                      "between a model and this ceiling is the real error.")
            else:
                print("  the reference is consistent, so the model scores above "
                      "are the models' own error.")

        # ---- is RMSE pinned by the match radius? ----
        sweep_cols = [c for c in CENTROID.columns if c.startswith("f1_r")]
        if sweep_cols:
            print("\nF1 against match radius (px)")
            print(CENTROID.groupby(["condition", "arch"])[sweep_cols]
                  .mean().round(3).to_string())
            rr = [c for c in CENTROID.columns if c.startswith("rmse_r")]
            print("\nRMSE against match radius; if RMSE tracks the radius, it is "
                  "the matcher speaking, not the model")
            print(CENTROID.groupby(["condition", "arch"])[rr]
                  .mean().round(3).to_string())

        # ---- does pixel IoU predict finding atoms? ----
        j = MASTER.merge(CENTROID, on=["condition", "arch", "fold"], suffixes=("", "_c"))
        if len(j) > 3:
            for c in ("iou", "dice", "best_f1", "psnr"):
                if c in j:
                    print(f"corr({c}, detection F1) = {np.corrcoef(j[c], j.f1)[0, 1]:+.3f}")
            print(f"corr(blob ratio, detection F1) = "
                  f"{np.corrcoef(j.blob_ratio, j.f1)[0, 1]:+.3f}")
            print(f"corr(count ratio, detection F1) = "
                  f"{np.corrcoef(j.count_ratio, j.f1)[0, 1]:+.3f}")

## 15. Real-frame transfer across every fold, architecture and condition

The experimental frames carry no labels, so nothing here is an accuracy. What
can be measured without labels is whether a model behaves like a detector on
these frames at all: how much of the frame it calls atom, how many columns it
finds, whether the spacing it reports agrees with the lattice the raw FFT
already shows, and whether the denoising head moved the lattice.

Frames are processed at full size with overlapping tiles and a Hann window,
because a model trained on 256 px crops produces seam artefacts on a 2048 px
micrograph when it is run in one pass.

In [ ]:
# ============================================================
# 15a. Real frames: loading, calibration-free FFT lattice, tiled inference
# ============================================================
REAL_EXT = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".dm3", ".dm4")

real_files = sorted([p for p in CFG.REAL_ROOT.rglob("*")
                     if p.is_file() and p.suffix.lower() in REAL_EXT
                     and p.suffix.lower() not in (".dm3", ".dm4")])
print(f"{len(real_files)} real frames under {CFG.REAL_ROOT}")
by_dir = Counter(p.parent.name for p in real_files)
print(f"by folder: {dict(by_dir)}")


def strip_scalebar(img, frac=0.12):
    """Burned-in scale bars sit in a corner strip and wreck the FFT."""
    h, w = img.shape
    cut = int(h * frac)
    band = img[-cut:, :]
    if band.std() > 2.5 * img[: h - cut].std() or band.max() >= img.max():
        return img[: h - cut, :]
    return img


def radial_power(img, hann=True):
    a = img - img.mean()
    if hann:
        wy = np.hanning(a.shape[0])[:, None]
        wx = np.hanning(a.shape[1])[None, :]
        a = a * wy * wx
    P = np.abs(np.fft.fftshift(np.fft.fft2(a))) ** 2
    cy, cx = np.array(P.shape) // 2
    yy, xx = np.indices(P.shape)
    r = np.hypot(yy - cy, xx - cx).astype(int)
    n = np.bincount(r.ravel())
    prof = np.bincount(r.ravel(), P.ravel()) / np.maximum(n, 1)
    rmax = min(cy, cx)
    return np.arange(rmax), prof[:rmax]


def find_rings(img, r_min_frac=0.02, r_max_frac=0.45, prominence=2.0, max_rings=4):
    """Peaks in the background-flattened radial power profile."""
    r, p = radial_power(img)
    N = min(img.shape)
    lo, hi = int(r_min_frac * N), int(r_max_frac * N)
    lo, hi = max(lo, 4), min(hi, len(r) - 2)
    if hi - lo < 10:
        return []
    seg = np.log10(p[lo:hi] + 1e-9)
    base = gaussian_filter(seg, 12)
    flat = seg - base
    from scipy.signal import find_peaks
    idx, props = find_peaks(flat, prominence=prominence * flat.std(), distance=6)
    rings = sorted(zip(r[lo:hi][idx], flat[idx]), key=lambda z: -z[1])[:max_rings]
    return sorted([(int(a), float(b)) for a, b in rings])


def lattice_snr(img):
    """Height of the strongest ring over the local background, in the log profile."""
    rings = find_rings(img)
    if not rings:
        return 0.0
    return float(max(h for _, h in rings))


def hann2d(n):
    w = np.hanning(n)
    return np.outer(w, w)


@torch.no_grad()
def tiled_infer(model, img, tile=256, overlap=64):
    """Overlap-tiled inference with Hann blending. Returns (prob, denoised)."""
    h, w = img.shape
    tile = min(tile, h, w)
    step = max(8, tile - overlap)
    win = hann2d(tile) + 1e-6
    P = np.zeros((h, w), np.float32)
    D = np.zeros((h, w), np.float32)
    W = np.zeros((h, w), np.float32)
    has_den = True
    ys = list(range(0, max(h - tile, 0) + 1, step))
    xs = list(range(0, max(w - tile, 0) + 1, step))
    if ys[-1] != h - tile:
        ys.append(h - tile)
    if xs[-1] != w - tile:
        xs.append(w - tile)
    for y in ys:
        for x in xs:
            patch = img[y:y + tile, x:x + tile]
            t = torch.from_numpy(norm01(patch))[None, None].to(CFG.DEVICE)
            den, seg = split_heads(model(t))
            pr = seg_prob(seg)[0, 0].float().cpu().numpy()
            P[y:y + tile, x:x + tile] += pr * win
            if den is not None:
                D[y:y + tile, x:x + tile] += norm01(den[0, 0].float().cpu().numpy()) * win
            else:
                has_den = False
            W[y:y + tile, x:x + tile] += win
    P /= np.maximum(W, 1e-6)
    D /= np.maximum(W, 1e-6)
    return P, (D if has_den else None)


REAL = {}
for p in real_files:
    try:
        img = strip_scalebar(load_gray(p))
        REAL[p] = norm01(img)
    except Exception as e:
        print(f"  skip {p.name}: {e}")
print(f"loaded {len(REAL)} frames, sizes {Counter(v.shape for v in REAL.values())}")

In [ ]:
# ============================================================
# 15b. Transfer run over every checkpoint
# ============================================================
TRANSFER_CSV = CFG.OUT_ROOT / "tables" / "real_transfer.csv"
TRANSFER_MAX_FRAMES = 12       # None uses all of them
FORCE_TRANSFER = False

frames = list(REAL)[:TRANSFER_MAX_FRAMES] if TRANSFER_MAX_FRAMES else list(REAL)


def frame_gates(img):
    """Label-free quality gates that belong to the image, not to any model."""
    from scipy.ndimage import laplace
    rings = find_rings(img)
    snr = lattice_snr(img)
    focus = float(np.var(laplace(gaussian_filter(img, 1.0))))
    return dict(lat_snr=snr, n_rings=len(rings),
                ring_r_px=rings[0][0] if rings else np.nan,
                focus=focus, contrast=float(img.std()),
                usable_image=bool(snr > 1.0 and len(rings) >= 1))


GATES = pd.DataFrame([{**frame_gates(REAL[f]), "file": f.name, "path": str(f)}
                      for f in frames])
if len(GATES):
    print("image gates")
    print(GATES[["file", "lat_snr", "n_rings", "ring_r_px", "focus", "usable_image"]]
          .to_string(index=False))
else:
    print(f"no real frames loaded from {CFG.REAL_ROOT}; sections 15 to 18 will "
          f"produce nothing. Check the path and the file extensions.")


@torch.no_grad()
def transfer_one(model, img, expected_ring_px=None):
    P, D = tiled_infer(model, img)
    cov = float((P > 0.5).mean())
    pk, md = locate_columns(P, D if D is not None else img)
    sp = first_shell_spacing(pk[["x", "y"]].values) if len(pk) > 20 else {}
    # did the denoiser move the lattice? compare the strongest FFT ring
    r_in = find_rings(img)
    r_dn = find_rings(D) if D is not None else []
    ring_shift = np.nan
    if r_in and r_dn:
        ring_shift = float(abs(r_dn[0][0] - r_in[0][0]) / max(r_in[0][0], 1))
    # a probability map that is all one value carries no information
    ent = float(-np.mean(P * np.log(P + 1e-9) + (1 - P) * np.log(1 - P + 1e-9)))
    return dict(coverage=cov, n_columns=len(pk),
                fitted_frac=float(pk.ok.mean()) if len(pk) else np.nan,
                d_px=sp.get("d_px", np.nan), d_mad=sp.get("mad_px", np.nan),
                min_distance=md, ring_shift=ring_shift, prob_entropy=ent,
                prob_mean=float(P.mean()),
                psnr_proxy=float(psnr(norm01(D), img)) if D is not None else np.nan), P, D


if TRANSFER_CSV.exists() and not FORCE_TRANSFER:
    TRANSFER = pd.read_csv(TRANSFER_CSV)
else:
    TRANSFER = pd.DataFrame()

seen = set(zip(TRANSFER.condition, TRANSFER.arch, TRANSFER.fold, TRANSFER.file)) \
    if len(TRANSFER) else set()
rows = TRANSFER.to_dict("records") if len(TRANSFER) else []

for _, r in todo.iterrows():
    if all((r.condition, r.arch, int(r.fold), f.name) in seen for f in frames):
        continue
    try:
        model = model_from_checkpoint(r.path, r.arch)
    except Exception as e:
        print(f"{r.condition} {r.arch} f{int(r.fold)}: cannot load, {e}")
        continue
    t0 = time.time()
    for f in frames:
        if (r.condition, r.arch, int(r.fold), f.name) in seen:
            continue
        try:
            m, P, D = transfer_one(model, REAL[f])
        except Exception as e:
            print(f"  {f.name}: {type(e).__name__}: {e}")
            continue
        m.update(condition=r.condition, arch=r.arch, fold=int(r.fold),
                 file=f.name, path=str(f))
        rows.append(m)
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"{r.condition:9s} {r.arch:11s} f{int(r.fold)}  "
          f"{len(frames)} frames in {time.time() - t0:.0f} s")

TRANSFER = pd.DataFrame(rows)
if len(TRANSFER):
    TRANSFER.to_csv(TRANSFER_CSV, index=False)
    agg = TRANSFER.groupby(["condition", "arch"]).agg(
        coverage=("coverage", "mean"), columns=("n_columns", "mean"),
        d_px=("d_px", "mean"), d_spread=("d_px", "std"),
        ring_shift=("ring_shift", "mean"), folds=("fold", "nunique"))
    print("\nreal-frame behaviour, averaged over frames and folds")
    print(agg.to_string())

## 16. Automated transfer verdict

Two independent sets of gates. The image gates ask whether the frame could
support a measurement by any method, using the FFT rings and the focus. The
model gates ask whether this model produced something usable on that frame.

The verdict for each model-frame pair is one of: usable, mask uninformative
(the probability map is saturated or empty, so no columns exist to measure),
structure altered (the denoising head moved the strongest FFT ring by more
than 3 percent, so the reported positions describe the model rather than the
specimen), or spacing implausible (the detected first-shell spacing disagrees
with the spacing the raw FFT already implies).

In [ ]:
# ============================================================
# 16. Verdict
# ============================================================
class GATE:
    COVERAGE = (0.01, 0.40)      # fraction of pixels called atom
    MIN_COLUMNS = 50
    RING_SHIFT_MAX = 0.03        # 3 percent
    SPACING_TOL = 0.20           # vs the FFT-implied spacing
    MIN_FITTED_FRAC = 0.50
    MIN_PROB_ENTROPY = 0.05


def fft_spacing_px(img):
    """Real-space spacing implied by the strongest FFT ring, in pixels."""
    rings = find_rings(img)
    if not rings:
        return np.nan
    N = min(img.shape)
    return float(N / rings[0][0])


FFT_SPACING = {f.name: fft_spacing_px(REAL[f]) for f in frames}


def verdict_row(r):
    reasons = []
    g = GATES[GATES.file == r.file]
    if len(g) and not bool(g.usable_image.iloc[0]):
        return "image unusable", "no FFT ring or flat contrast"
    if not (GATE.COVERAGE[0] <= r.coverage <= GATE.COVERAGE[1]):
        reasons.append(f"coverage {r.coverage:.3f}")
    if r.prob_entropy < GATE.MIN_PROB_ENTROPY:
        reasons.append(f"prob entropy {r.prob_entropy:.3f}")
    if r.n_columns < GATE.MIN_COLUMNS:
        reasons.append(f"{int(r.n_columns)} columns")
    if reasons:
        return "mask uninformative", "; ".join(reasons)
    if np.isfinite(r.ring_shift) and r.ring_shift > GATE.RING_SHIFT_MAX:
        return "structure altered", f"FFT ring moved {r.ring_shift:.1%}"
    exp = FFT_SPACING.get(r.file, np.nan)
    if np.isfinite(exp) and np.isfinite(r.d_px):
        rel = abs(r.d_px - exp) / exp
        if rel > GATE.SPACING_TOL and abs(r.d_px * 2 - exp) / exp > GATE.SPACING_TOL:
            return "spacing implausible", f"detected {r.d_px:.1f} px vs FFT {exp:.1f} px"
    if np.isfinite(r.fitted_frac) and r.fitted_frac < GATE.MIN_FITTED_FRAC:
        return "usable, weak fits", f"only {r.fitted_frac:.0%} clean Gaussian fits"
    return "usable", ""


if len(TRANSFER):
    V = TRANSFER.copy()
    V[["verdict", "why"]] = V.apply(lambda r: pd.Series(verdict_row(r)), axis=1)
    V.to_csv(CFG.OUT_ROOT / "tables" / "real_transfer_verdict.csv", index=False)

    print("verdicts by model")
    tab = V.pivot_table(index=["condition", "arch"], columns="verdict",
                        values="file", aggfunc="count", fill_value=0)
    print(tab.to_string())
    print("\nverdicts by frame")
    tab2 = V.pivot_table(index="file", columns="verdict", values="arch",
                         aggfunc="count", fill_value=0)
    print(tab2.to_string())

    usable_rate = (V.verdict.str.startswith("usable")
                   .groupby([V.condition, V.arch]).mean().sort_values(ascending=False))
    print("\nusable rate, the transfer headline")
    for (c, a), v in usable_rate.items():
        print(f"  {c:9s} {a:11s} {v:.0%}")

    bad = V[V.verdict != "usable"].groupby("why").size().sort_values(ascending=False)
    if len(bad):
        print("\nmost common failure reasons")
        print(bad.head(10).to_string())

## 17. Equivariance on real frames

No labels are needed for this one. Segmenting a rotated frame should give the
rotated segmentation of the original. The gap between the two is an error the
model makes on real data, measured in the same probability units as the
training metric, and it is the only number in this notebook that tests the
experimental frames directly rather than by proxy.

The D4 group (four rotations, two flips) is exact on a square grid, so any
disagreement is the model and not interpolation. A sub-pixel translation test
is included separately because translation equivariance is what column
positions actually depend on.

In [ ]:
# ============================================================
# 17. Equivariance
# ============================================================
D4 = [("id", lambda a: a, lambda a: a),
      ("rot90", lambda a: np.rot90(a, 1), lambda a: np.rot90(a, -1)),
      ("rot180", lambda a: np.rot90(a, 2), lambda a: np.rot90(a, -2)),
      ("rot270", lambda a: np.rot90(a, 3), lambda a: np.rot90(a, -3)),
      ("flipud", np.flipud, np.flipud),
      ("fliplr", np.fliplr, np.fliplr)]

EQUI_FRAMES = 4
EQUI_TILE = 512


@torch.no_grad()
def equivariance(model, img, tile=EQUI_TILE):
    h, w = img.shape
    t = min(tile, h, w)
    y0, x0 = (h - t) // 2, (w - t) // 2
    crop = img[y0:y0 + t, x0:x0 + t]
    P0, _ = tiled_infer(model, crop)
    rows = []
    for name, fwd, inv in D4[1:]:
        Pg, _ = tiled_infer(model, np.ascontiguousarray(fwd(crop)))
        back = np.ascontiguousarray(inv(Pg))
        rows.append(dict(op=name, mae=float(np.mean(np.abs(back - P0))),
                         iou=float(np.logical_and(back > 0.5, P0 > 0.5).sum() /
                                   max(np.logical_or(back > 0.5, P0 > 0.5).sum(), 1))))
    # sub-pixel translation: shift in, shift the output back, compare
    for s in (0.25, 0.5):
        sh = fshift(crop, s, s)
        Ps, _ = tiled_infer(model, norm01(sh))
        back = fshift(Ps, -s, -s)
        m = slice(8, -8)
        rows.append(dict(op=f"shift{s}", mae=float(np.mean(np.abs(back[m, m] - P0[m, m]))),
                         iou=float(np.logical_and(back[m, m] > 0.5, P0[m, m] > 0.5).sum() /
                                   max(np.logical_or(back[m, m] > 0.5, P0[m, m] > 0.5).sum(), 1))))
    return pd.DataFrame(rows)


EQUI = []
equi_frames = [f for f in frames][:EQUI_FRAMES]
for cond in BEST_CONDITIONS:
    for arch in BEST_ARCHS:
        f, ck = best_fold(cond, arch)
        if f is None:
            continue
        try:
            model = model_from_checkpoint(ck, arch)
        except Exception as e:
            print(f"{cond} {arch}: {e}")
            continue
        for fr in equi_frames:
            try:
                d = equivariance(model, REAL[fr])
            except Exception as e:
                print(f"  {fr.name}: {e}")
                continue
            d["condition"], d["arch"], d["fold"], d["file"] = cond, arch, f, fr.name
            EQUI.append(d)
        del model
        if CFG.DEVICE.type == "cuda":
            torch.cuda.empty_cache()

if EQUI:
    EQUI = pd.concat(EQUI, ignore_index=True)
    EQUI.to_csv(CFG.OUT_ROOT / "tables" / "equivariance.csv", index=False)
    piv = EQUI.pivot_table(index=["condition", "arch"], columns="op", values="mae")
    print("equivariance error, mean absolute probability difference")
    print(piv.round(4).to_string())
    print("\nself-consistency IoU (1.00 would be a perfectly equivariant model)")
    print(EQUI.pivot_table(index=["condition", "arch"], columns="op", values="iou")
          .round(3).to_string())
    worst = EQUI.groupby(["condition", "arch"]).mae.mean().sort_values()
    print("\nmost self-consistent on real frames:")
    for (c, a), v in worst.items():
        print(f"  {c:9s} {a:11s} mean MAE {v:.4f}")

## 18. Per-image ring indexing and angstrom per pixel

The simulated set has a known pixel size; the experimental frames do not,
unless the DigitalMicrograph metadata survived the export. Without a scale,
every spacing in section 15 is in pixels and cannot be compared with a
published lattice parameter.

The rings in the FFT give the scale directly. For a known material the ratios
of the ring radii identify which reflections they are, and then the scale
follows from `A/px = d_hkl * r_px / N`. The ratio test is what makes this
safe: a single ring assigned by assumption would silently absorb any error,
whereas two rings whose radius ratio matches a known reflection pair to
better than a few percent is a real identification.

In [ ]:
# ============================================================
# 18. Ring indexing and scale calibration
# ============================================================
MATERIALS = {
    "diamond": {"a_A": 3.567, "reflections": {(1, 1, 1): 2.0593, (2, 2, 0): 1.2611,
                                              (3, 1, 1): 1.0754, (4, 0, 0): 0.8917}},
    "graphite": {"a_A": 2.464, "reflections": {(1, 0, 0): 2.134, (1, 1, 0): 1.232,
                                               (2, 0, 0): 1.067}},
    "silicon": {"a_A": 5.431, "reflections": {(1, 1, 1): 3.1355, (2, 2, 0): 1.9201,
                                              (3, 1, 1): 1.6375}},
}


def index_rings(img, material="diamond", tol=0.04):
    """Match observed ring-radius ratios to a reflection pair, return A/px."""
    rings = find_rings(img, max_rings=5)
    N = min(img.shape)
    if len(rings) < 1:
        return dict(file=None, n_rings=0, A_per_px=np.nan, note="no rings")
    radii = sorted(r for r, _ in rings)
    ref = MATERIALS[material]["reflections"]
    keys = sorted(ref, key=lambda k: -ref[k])
    best = None
    for (i, r1), (j, r2) in itertools.combinations(list(enumerate(radii)), 2):
        obs = r2 / r1
        for a, b in itertools.combinations(keys, 2):
            exp = ref[a] / ref[b]                 # radius ratio is the inverse d ratio
            if abs(obs - exp) / exp < tol:
                scale1 = ref[a] * r1 / N
                scale2 = ref[b] * r2 / N
                err = abs(scale1 - scale2) / scale1
                cand = dict(r1=r1, r2=r2, hkl1=a, hkl2=b, ratio_obs=obs,
                            ratio_exp=exp, A_per_px=0.5 * (scale1 + scale2),
                            scale_disagreement=err)
                if best is None or err < best["scale_disagreement"]:
                    best = cand
    if best is None:
        return dict(n_rings=len(radii), radii=radii, A_per_px=np.nan,
                    note="no reflection pair matched the observed ratios")
    best.update(n_rings=len(radii), radii=radii, material=material, note="ok")
    return best


MATERIAL_BY_FOLDER = {}          # e.g. {"Carbon": "diamond", "MWCNT": "graphite"}
DEFAULT_MATERIAL = "diamond"

cal_rows = []
for f in frames:
    mat = MATERIAL_BY_FOLDER.get(Path(f).parent.name, DEFAULT_MATERIAL)
    r = index_rings(REAL[f], mat)
    r["file"] = f.name
    r["folder"] = Path(f).parent.name
    cal_rows.append(r)

CAL = pd.DataFrame(cal_rows)
if not len(CAL):
    print("no frames to calibrate")
cols = [c for c in ["file", "folder", "material", "n_rings", "r1", "r2", "hkl1",
                    "hkl2", "ratio_obs", "ratio_exp", "A_per_px",
                    "scale_disagreement", "note"] if c in CAL]
print(CAL[cols].to_string(index=False))
CAL.to_csv(CFG.OUT_ROOT / "tables" / "scale_calibration.csv", index=False)

ok = CAL[CAL.A_per_px.notna()]
if len(ok):
    print(f"\ncalibrated {len(ok)}/{len(CAL)} frames")
    print(f"A/px median {ok.A_per_px.median():.5f}, "
          f"spread {ok.A_per_px.std():.5f} "
          f"({100 * ok.A_per_px.std() / ok.A_per_px.median():.1f} %)")
    if len(TRANSFER):
        m = TRANSFER.merge(ok[["file", "A_per_px"]], on="file", how="inner")
        m["d_A"] = m.d_px * m.A_per_px
        s = m.groupby(["condition", "arch"]).d_A.agg(["mean", "std", "count"])
        print("\nfirst-shell spacing on real frames, in angstrom")
        print(s.round(4).to_string())
        print("compare against the material's expected nearest-column spacing; "
              "a model whose mean sits several sd away is measuring its own prior.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    f0 = frames[0]
    r, p = radial_power(REAL[f0])
    ax[0].semilogy(r, p + 1e-9)
    for rr, _ in find_rings(REAL[f0]):
        ax[0].axvline(rr, color="r", lw=0.7, ls="--")
    ax[0].set_xlabel("FFT radius (px)"); ax[0].set_ylabel("power")
    ax[0].set_title(f"{f0.name}: radial power and detected rings")
    ax[1].hist(ok.A_per_px, bins=12, color="0.5")
    ax[1].set_xlabel("A / px"); ax[1].set_title("per-frame calibration")
    plt.tight_layout()
    plt.savefig(CFG.OUT_ROOT / "figures" / "ring_calibration.png", dpi=150)
    plt.show()

## 19. The rest of TEM-ImageNet-v1.3

The folder listing has sixteen entries and a segmentation run uses two of
them. What the other fourteen support, in rough order of how much they add
for the effort:

An input ablation. `image`, `noNoise`, and `noBackgroundnoNoise` are the same
specimen with the noise and the amorphous background switched off
independently. Running a trained model across all three separates the two
things that cost it IoU, which is a two-line experiment with a result that
goes straight into a paper: if IoU recovers on `noNoise` the model is
noise-limited, and if it only recovers on `noBackgroundnoNoise` the
background is the problem and more denoising will not help. Section 19b runs
it.

A target ablation. `circularMask` and `smallcircularMask` are the same
columns at two disc radii. Training on the small disc raises the localization
weight per pixel and usually lowers IoU while improving centroid F1, which is
the cleanest demonstration available that pixel IoU and the physics disagree.
`gaussianMask` as a regression target instead of a binary mask is the third
point on that axis.

A resolution axis. `noNoiseUpinterpolation2x`,
`noNoiseNoBackgroundUpinterpolation2x` and `noNoiseNoBackgroundSuperresolution`
give a 2x target for the same field. A model trained to output the 2x frame is
a super-resolution head rather than a denoiser, and sub-pixel precision
measured on its output against section 9 is a direct test of whether
super-resolution buys real precision or only a smoother-looking image.

Radius regression. `radius`, `smallradius` and `positionRadius` carry a
per-column radius, which is a proxy for column height and species. A second
regression head predicting radius turns the segmentation model into something
that estimates thickness, and radius error against these maps is a supervised
metric nobody in this comparison has reported yet.

Stratification by simulation parameters. `params` holds the per-frame
simulation settings. Every table in this notebook is a mean over a mixture of
dose, defocus and thickness; grouping the same rows by those parameters shows
where each architecture fails, which is a stronger result than any single
mean. Section 19c reads the folder and reports what fields are available.

`position` holds unit-cell vectors rather than atom coordinates, which is the
bug that was already found here; it is still useful as the ground-truth
lattice basis for section 10, since it says what the basis should have been.

In [ ]:
# ============================================================
# 19b. Input ablation: noise and background, separated
# ============================================================
ABLATION_VARIANTS = [v for v in ["image", "noNoise", "noBackgroundnoNoise",
                                 "noNoiseUpinterpolation2x"] if v in VARIANTS]
ABLATION_N = 60


@torch.no_grad()
def ablate_inputs(ckpt, arch, fold, variants=ABLATION_VARIANTS, n=ABLATION_N):
    _, va = fold_indices(fold)
    stems = [common[i] for i in va][:n]
    model = model_from_checkpoint(ckpt, arch)
    rows = []
    for v in variants:
        vm = VARIANTS[v]
        acc = defaultdict(list)
        for s in stems:
            if s not in vm or s not in mask_map:
                continue
            img = norm01(load_gray(vm[s]))
            gt = to_binary(load_gray(mask_map[s]))
            if img.shape != gt.shape:               # the 2x variants
                img = np.asarray(Image.fromarray((img * 255).astype(np.uint8))
                                 .resize(gt.shape[::-1], Image.BICUBIC), np.float32) / 255
            _, seg = split_heads(model(torch.from_numpy(img)[None, None].to(CFG.DEVICE)))
            prob = seg_prob(seg)[0, 0].float().cpu().numpy()
            i_, d_, p_, r_ = pixel_stats(prob, gt, CFG.THRESHOLD)
            acc["iou"].append(i_); acc["dice"].append(d_)
            acc["precision"].append(p_); acc["recall"].append(r_)
        if acc["iou"]:
            rows.append(dict(variant=v, n=len(acc["iou"]),
                             **{k: float(np.mean(x)) for k, x in acc.items()}))
    del model
    if CFG.DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows).assign(arch=arch, fold=fold)


ABL = []
for cond in BEST_CONDITIONS:
    for arch in BEST_ARCHS:
        f, ck = best_fold(cond, arch)
        if f is None:
            continue
        try:
            d = ablate_inputs(ck, arch, f)
        except Exception as e:
            print(f"{cond} {arch}: {e}")
            continue
        d["condition"] = cond
        ABL.append(d)

if ABL:
    ABL = pd.concat(ABL, ignore_index=True)
    ABL.to_csv(CFG.OUT_ROOT / "tables" / "input_ablation.csv", index=False)
    piv = ABL.pivot_table(index=["condition", "arch"], columns="variant", values="iou")
    print("IoU by input variant (the model saw only `image` during training)")
    print(piv.round(4).to_string())
    if {"image", "noNoise"} <= set(piv.columns):
        piv2 = piv.copy()
        piv2["noise cost"] = piv["noNoise"] - piv["image"]
        if "noBackgroundnoNoise" in piv:
            piv2["background cost"] = piv["noBackgroundnoNoise"] - piv["noNoise"]
        print("\nwhat each degradation costs, in IoU")
        print(piv2[[c for c in ["noise cost", "background cost"] if c in piv2]]
              .round(4).to_string())
        print("a large noise cost with a small background cost means denoising "
              "is the lever; the reverse means it is not.")

    fig, ax = plt.subplots(figsize=(7, 3.4))
    for (c, a), g in ABL.groupby(["condition", "arch"]):
        ax.plot(g.variant, g.iou, marker="o", ms=4, label=f"{a} {c}")
    ax.set_ylabel("IoU"); ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=7); ax.set_title("input ablation")
    plt.tight_layout()
    plt.savefig(CFG.OUT_ROOT / "figures" / "input_ablation.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# 19c. What is inside params, radius and position
# ============================================================
def peek_params(n=3):
    d = CFG.DATA_ROOT / "params"
    if not d.exists():
        print("no params folder")
        return None
    files = sorted(d.iterdir())[:n]
    print(f"params/: {len(list(d.iterdir()))} files, example {files[0].name}")
    for p in files[:1]:
        if p.suffix.lower() == ".npy":
            a = np.load(p, allow_pickle=True)
            print(f"  npy shape {getattr(a, 'shape', None)} dtype {getattr(a, 'dtype', None)}")
            print(f"  {str(a)[:400]}")
        else:
            print("  " + p.read_text(errors="ignore")[:400])
    return files


def peek_variant(name):
    if name not in VARIANTS:
        print(f"no {name} folder")
        return
    s = common[0] if common[0] in VARIANTS[name] else list(VARIANTS[name])[0]
    a = load_gray(VARIANTS[name][s])
    print(f"{name}/: {len(VARIANTS[name])} files, example {s} shape {a.shape} "
          f"range [{a.min():.4g}, {a.max():.4g}], {len(np.unique(a))} unique values")
    return a


peek_params()
for v in ["radius", "smallradius", "positionRadius", "position", "gaussianMask",
          "smallcircularMask", "noNoiseNoBackgroundSuperresolution"]:
    peek_variant(v)

# the ground-truth basis, for comparison with what section 10 recovered
if "position" in VARIANTS and len(MASTER):
    try:
        pos = load_gray(VARIANTS["position"][_stem]) if _stem in VARIANTS["position"] else None
        if pos is not None:
            print(f"\nposition/{_stem}: shape {pos.shape}; these are unit-cell "
                  f"vectors, so compare them with the basis section 10 recovered "
                  f"({np.round(L['basis'], 3).tolist()}), not with atom coordinates.")
    except Exception as e:
        print(f"position read failed: {e}")

## 20. Final report

One cell that collects every table into a single markdown file, so the
results survive a kernel restart.

In [ ]:
# ============================================================
# 20. Report
# ============================================================
def md_table(df, cols=None, floatfmt=4):
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    df = df.round(floatfmt)
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"
    body = ["| " + " | ".join(map(str, r)) + " |" for r in df.values]
    return "\n".join([head, sep] + body)


lines = [f"# TEM benchmark report",
         f"", f"Generated {time.strftime('%Y-%m-%d %H:%M')}.",
         f"Dataset {CFG.DATA_ROOT} with {len(common)} paired frames, "
         f"{CFG.N_FOLDS}-fold splits, seed {CFG.SEED}.",
         f"Threshold {CFG.THRESHOLD}, pixel size {CFG.PIXEL_SIZE_A} A/px.", ""]

if len(MASTER):
    lines += ["## Segmentation and denoising", "",
              md_table(SUMMARY, ["condition", "arch", "folds", "params_M", "iou",
                                 "iou_sd", "dice", "best_f1", "psnr", "ssim",
                                 "psnr_gain_vs_gauss"]), ""]
if len(CENTROID):
    lines += ["## Column detection", "",
              md_table(CENTROID.groupby(["condition", "arch"])
                       .agg(f1=("f1", "mean"), precision=("precision", "mean"),
                            recall=("recall", "mean"), rmse_pm=("rmse_pm", "mean"),
                            folds=("fold", "nunique")).reset_index()), ""]
if PRECISION:
    pr = pd.concat([d.assign(condition=k[0], arch=k[1]) for k, d in PRECISION.items()])
    lines += ["## Sub-pixel precision", "",
              md_table(pr.groupby(["condition", "arch", "dose"])
                       .agg(jitter_px=("jitter_x", "mean"), bias_px=("bias_x", "mean"),
                            crb_px=("crb", "mean")).reset_index()), ""]
if len(TRANSFER):
    lines += ["## Real-frame transfer", "",
              md_table(V.pivot_table(index=["condition", "arch"], columns="verdict",
                                     values="file", aggfunc="count",
                                     fill_value=0).reset_index(), floatfmt=2), ""]
if isinstance(EQUI, pd.DataFrame) and len(EQUI):
    lines += ["## Equivariance on real frames", "",
              md_table(EQUI.pivot_table(index=["condition", "arch"], columns="op",
                                        values="mae").reset_index()), ""]
if isinstance(ABL, pd.DataFrame) and len(ABL):
    lines += ["## Input ablation", "",
              md_table(ABL.pivot_table(index=["condition", "arch"], columns="variant",
                                       values="iou").reset_index()), ""]

report = CFG.OUT_ROOT / "REPORT.md"
report.write_text("\n".join(lines), encoding="utf-8")
print(f"report written to {report}")
print("\n".join(lines[:40]))

In [ ]:
_stem = "07311"
for name in ("position", "positionRadius", "radius", "smallradius"):
    d = CFG.DATA_ROOT / name
    if not d.exists():
        print(f"{name}: missing")
        continue
    f = next((p for p in d.iterdir() if p.stem == _stem), None) or next(d.iterdir())
    print(f"\n{name}/{f.name}  ({f.suffix})")
    if f.suffix.lower() == ".npy":
        a = np.load(f, allow_pickle=True)
        print(f"  shape {getattr(a, 'shape', None)}  dtype {getattr(a, 'dtype', None)}")
        print(f"  {str(a)[:400]}")
    elif f.suffix.lower() in (".txt", ".csv", ".json", ".md"):
        print("  " + f.read_text(errors="ignore")[:400])
    else:
        a = load_gray(f)
        print(f"  image {a.shape}, range [{a.min():.4g}, {a.max():.4g}], "
              f"{len(np.unique(a))} unique values, nonzero {int((a > 0).sum())}")